# Can a small Combine movement block survive player-held-out testing?

**NFL Big Data Bowl2027: research continuation, not a submitted or validated scouting model.**

We test whether a fixed, interpretable movement block adds information to conventional Combine measurements for recorded rookie pressure frequency. This is a smaller follow-up to the baseline-preserving residual experiment. That experiment's selected strategy **worsened repeated nested MSE2.878%**; its temporal fits chose zero correction. Earlier favorable movement comparisons and all2025 comparisons remain exploratory. Both engines reuse players and are not independent replications.

The official challenge requires Combine sensor tracking linked to regular-season NFL performance. This notebook rebuilds the historical eligible rush cohort from official files and extracts source-only movement features. The roster remains conditional on observed NFL participation, charted get-off and at least30 eligible records; it does not represent all prospects. The response is a charted pressure-event fraction, not pressures per pass-rush opportunity for every player or a causal athletic effect.

**No competition data are bundled, downloaded or fabricated.** Attach the official competition dataset. The default input path is `/kaggle/input/nfl-big-data-bowl-2027`, overridable through `BDB_DATA_ROOT`. The supplied input hashes pin the original dataset version; a newer version must be reconciled explicitly before continuing.

The separate two-season2025 linebacker confirmation project is not opened or scored here.


In [ ]:
import os, sys, json, hashlib, tempfile, contextlib, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display=print

WORK=Path(os.environ.get('BDB_WORK_ROOT', tempfile.mkdtemp(prefix='bdb_compact_'))).resolve()
WORK.mkdir(parents=True,exist_ok=True)
REPO=WORK/'public_code';REPO.mkdir(exist_ok=True)
MODULES={'upstream/analysis.py': '"""Combine-to-game research. No competition data are bundled or fabricated."""\nfrom pathlib import Path\nimport json, hashlib, argparse, platform\nimport numpy as np\nimport pandas as pd\nfrom scipy.signal import savgol_filter\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import Ridge\nfrom sklearn.ensemble import RandomForestRegressor\n\nSEED=20271006\nBASE=[\'combine_height\',\'combine_weight\',\'ten_yd_split\',\'forty\',\'vertical\',\'broad_jump\',\'three_cone\',\'short_shuttle\']\nCONTEXT=[\'cushion\',\'down\',\'yards_to_go\',\'yardline_number\',\'pre_snap_home_score\',\'pre_snap_visitor_score\']\nCATS=[\'route_ran\',\'team_coverage_man_zone\',\'offense_formation\',\'in_motion_at_ball_snap\']\nMOTION=[\'entry_speed\',\'speed_retention\',\'exit_gain\',\'braking_impulse\',\'turn_angle\',\'curvature_energy\',\'jerk_rms\',\'signature_area\']\nCUT_ROUTES={\'OUT\',\'SLANT\',\'POST\',\'CORNER\',\'HITCH\'}\n\ndef movement(g,window=7):\n    """Largest interior heading change; fixed +/-0.5s event window, yards/seconds."""\n    g=g.sort_values(\'time\').copy()\n    t=pd.to_datetime(g.time,utc=True,format=\'ISO8601\').astype(\'int64\').to_numpy()/1e9\n    if len(t)<21 or np.any(np.diff(t)<=0): raise ValueError(\'short or duplicate/nonmonotone trajectory\')\n    dt=np.diff(t)\n    if not np.allclose(dt,.1,atol=.015): raise ValueError(\'irregular sampling; do not interpolate missing cuts\')\n    p=g[[\'x\',\'y\']].to_numpy(float)\n    if not np.isfinite(p).all(): raise ValueError(\'nonfinite coordinates\')\n    p=savgol_filter(p,window,3,axis=0)\n    v=np.gradient(p,.1,axis=0);a=np.gradient(v,.1,axis=0)\n    speed=np.linalg.norm(v,axis=1)\n    theta=np.unwrap(np.arctan2(v[:,1],v[:,0]))\n    k=np.arange(5,len(p)-5)\n    score=np.abs(theta[k+3]-theta[k-3]);score[(speed[k-3]<1)|(speed[k+3]<1)]=0\n    if score.max()<np.deg2rad(25): raise ValueError(\'no qualifying turn\')\n    c=int(k[np.flatnonzero(np.isclose(score,score.max(),rtol=0,atol=1e-10))[0]]);sl=slice(c-5,c+6)\n    turn=theta[c+3]-theta[c-3]\n    tangent=v[c-3]/speed[c-3];normal=np.array([-tangent[1],tangent[0]])\n    q=(p[sl]-p[c-5])@np.stack([tangent,normal],axis=1)\n    q[:,1]*=1 if turn>=0 else -1\n    # Antisymmetric level-two path signature, equal to signed area with closing chord.\n    area=.5*np.sum(q[:-1,0]*q[1:,1]-q[:-1,1]*q[1:,0])\n    sp=speed[sl];at=np.gradient(sp,.1)\n    omega=np.gradient(theta,.1)[sl];curv=omega/np.maximum(sp,1)\n    out=dict(entry_speed=float(np.mean(sp[:3])),speed_retention=float(sp.min()/np.mean(sp[:3])),\n      exit_gain=float(np.mean(sp[-3:])-sp.min()),braking_impulse=float(np.trapezoid(np.maximum(-at,0),dx=.1)),\n      turn_angle=float(abs(turn)),curvature_energy=float(np.trapezoid(curv**2*sp,dx=.1)),\n      jerk_rms=float(np.sqrt(np.mean(np.gradient(a[sl],.1,axis=0)**2))),signature_area=float(area))\n    diag={\'t\':np.arange(-5,6)*.1,\'q\':q,\'speed\':sp,\'tangential_acceleration\':at,\'cut_index\':c}\n    return out,diag\n\ndef inventory(root):\n    """Stream all CSVs; retain observed counts/schema/hash, never treat descriptions as measured counts."""\n    rows=[]\n    for p in sorted(Path(root).glob(\'*.csv\')):\n        count=0;ids=set();years=set();cols=None\n        for c in pd.read_csv(p,chunksize=200000):\n            count+=len(c);cols=list(c.columns)\n            if \'nfl_id\' in c: ids.update(c.nfl_id.dropna().astype(int))\n            for key in (\'season\',\'draft_year\'):\n                if key in c:years.update(c[key].dropna().astype(int))\n        h=hashlib.sha256()\n        with p.open(\'rb\') as f:\n            for b in iter(lambda:f.read(8*1024*1024),b\'\'): h.update(b)\n        rows.append(dict(file=p.name,rows=count,unique_players=len(ids),years=sorted(years),columns=cols,sha256=h.hexdigest()))\n    return rows\n\ndef prepare(root,out):\n    root,out=Path(root),Path(out);out.mkdir(parents=True,exist_ok=True)\n    players=pd.read_csv(root/\'players.csv\');combine=pd.read_csv(root/\'combine_results.csv\');games=pd.read_csv(root/\'games.csv\')\n    for d,key in [(players,[\'nfl_id\']),(combine,[\'nfl_id\',\'draft_year\']),(games,[\'game_id\'])]:\n        if d.duplicated(key).any():raise ValueError(\'duplicate metadata join key\')\n    ids=set(players.loc[players.nfl_position.eq(\'WR\'),\'nfl_id\']);pieces=[]\n    drill_counts={}\n    for c in pd.read_csv(root/\'combine_tracking.csv\',chunksize=200000):\n        c=c[c.nfl_id.isin(ids)&c.entity_type.eq(\'PLAYER\')&c.drill_type.eq(\'SKILL_DRILLS_WR\')]\n        pieces.append(c)\n        for name,n in c.drill_name.value_counts().items():drill_counts[name]=drill_counts.get(name,0)+int(n)\n    track=pd.concat(pieces,ignore_index=True);features=[];rejects=[]\n    for keys,g in track.groupby([\'nfl_id\',\'draft_year\',\'event_id\'],sort=True):\n        try:\n            f,_=movement(g);features.append(dict(nfl_id=keys[0],draft_year=keys[1],event_id=keys[2],drill_name=g.drill_name.iloc[0],**f))\n        except ValueError as e:rejects.append(dict(event_id=keys[2],reason=str(e)))\n    f=pd.DataFrame(features)\n    if f.empty:raise ValueError(\'no qualifying WR turns; inspect drill availability before changing protocol\')\n    f.to_csv(out/\'attempt_features.csv\',index=False)\n    # Choose one homologous drill using Combine availability only, before NFL outcomes.\n    availability=f.groupby([\'drill_name\',\'draft_year\']).nfl_id.nunique().unstack(fill_value=0).reindex(columns=[2023,2024,2025],fill_value=0)\n    minimum=availability.min(axis=1);eligible=minimum[minimum>=5]\n    if eligible.empty:raise ValueError(\'no qualifying drill shared by >=5 WRs in every class; inspect availability\')\n    selected=sorted(eligible.index,key=lambda k:(-eligible[k],k))[0]\n    f=f[f.drill_name.eq(selected)]\n    agg=f.groupby([\'nfl_id\',\'draft_year\'])[MOTION].median().reset_index()\n    pp=pd.read_csv(root/\'player_play.csv\')\n    if pp.duplicated([\'game_id\',\'play_id\',\'nfl_id\']).any():raise ValueError(\'duplicate player-play key\')\n    pp=pp.merge(games[[\'game_id\',\'season\',\'season_type\']],on=\'game_id\',validate=\'many_to_one\').merge(players[[\'nfl_id\',\'draft_year\',\'nfl_position\']],on=\'nfl_id\',validate=\'many_to_one\')\n    d=pp[pp.nfl_position.eq(\'WR\')&pp.season.eq(pp.draft_year)&pp.season_type.eq(\'REG\')&pp.route_ran.isin(CUT_ROUTES)&pp.play_nullified_by_penalty.eq(\'N\')].copy()\n    d=d[np.isfinite(d.separation_at_pass_forward)&(d.separation_at_pass_forward>=0)]\n    d=d.merge(combine,on=[\'nfl_id\',\'draft_year\'],validate=\'many_to_one\').merge(agg,on=[\'nfl_id\',\'draft_year\'],validate=\'many_to_one\')\n    # Each player must have >=20 measured routes. Threshold is frozen before labels are inspected.\n    n=d.groupby(\'nfl_id\').size();d=d[d.nfl_id.isin(n[n>=20].index)]\n    audit={\'status\':\'actual_data_prepared\',\'draft_class_players\':d.groupby(\'draft_year\').nfl_id.nunique().to_dict(),\n      \'draft_class_routes\':d.groupby(\'draft_year\').size().to_dict(),\'wr_roster_players\':len(ids),\'qualifying_attempts\':len(f),\n      \'selected_drill\':selected,\'drill_player_availability\':availability.to_dict(),\'excluded_attempts\':rejects,\'drill_frame_counts\':drill_counts,\'target_only_fraction\':float(d.target.mean()) if len(d) else None,\n      \'game_tracking_use\':\'Primary outcome uses official separation; game trajectory replication remains secondary and unrun.\'}\n    temp=out/\'private_analysis_rows.csv.tmp\';d.to_csv(temp,index=False);temp.replace(out/\'private_analysis_rows.csv\');\n    if len(pd.read_csv(out/\'private_analysis_rows.csv\'))!=len(d):raise ValueError(\'analysis CSV roundtrip row mismatch\')\n    d.to_pickle(out/\'private_analysis_rows.pkl\');(out/\'population_audit.json\').write_text(json.dumps(audit,indent=2))\n    return d,f,audit\n\ndef model(d,nums,kind,alpha=10):\n    numeric=Pipeline([(\'impute\',SimpleImputer(strategy=\'median\',add_indicator=True,keep_empty_features=True)),(\'scale\',StandardScaler())])\n    cats=Pipeline([(\'impute\',SimpleImputer(strategy=\'most_frequent\')),(\'onehot\',OneHotEncoder(handle_unknown=\'ignore\',sparse_output=False))])\n    pre=ColumnTransformer([(\'numeric\',numeric,nums),(\'category\',cats,CATS)])\n    reg=Ridge(alpha=alpha) if kind==\'ridge\' else RandomForestRegressor(n_estimators=200,min_samples_leaf=20,max_depth=6,max_features=.8,random_state=SEED,n_jobs=2)\n    return Pipeline([(\'pre\',pre),(\'reg\',reg)])\n\ndef weights(d):return 1/d.groupby(\'nfl_id\').nfl_id.transform(\'size\').to_numpy()\ndef score(d,p):return float(np.average((d.separation_at_pass_forward.to_numpy()-p)**2,weights=weights(d)))\n\ndef compare(d,out):\n    """Player-disjoint, rookie-season-disjoint validation and locked 2025 test."""\n    out=Path(out);out.mkdir(parents=True,exist_ok=True);d=d.copy()\n    if not (d.season==d.draft_year).all():raise ValueError(\'rookie seasons required\')\n    for col in MOTION:\n        d[col+\'_mass\']=d[col]*d.combine_weight/200\n        d[col+\'_man\']=d[col]*d.team_coverage_man_zone.eq(\'MAN_COVERAGE\').astype(int)\n    families={\'conventional\':BASE+CONTEXT,\'tracking\':BASE+CONTEXT+MOTION,\n      \'physical_context_interactions\':BASE+CONTEXT+MOTION+[c+s for c in MOTION for s in (\'_mass\',\'_man\')],\n      \'scouting_reference\':BASE+CONTEXT+[\'ngs_college_production_score\']}\n    a,b,t=[d[d.draft_year.eq(y)] for y in (2023,2024,2025)]\n    if min(x.nfl_id.nunique() for x in (a,b,t))<10:raise ValueError(\'fewer than ten WRs in a class; insufficient locked-cohort study\')\n    assert not(set(a.nfl_id)&set(b.nfl_id) or set(a.nfl_id)&set(t.nfl_id) or set(b.nfl_id)&set(t.nfl_id))\n    dev=pd.concat([a,b]);results=[];pred={};selections={}\n    for kind in (\'ridge\',\'forest\'):\n      for family,nums in families.items():\n        options=(1,10,100) if kind==\'ridge\' else (10,)\n        vals=[]\n        for alpha in options:\n            m=model(a,nums,kind,alpha);m.fit(a,a.separation_at_pass_forward,reg__sample_weight=weights(a));vals.append((score(b,m.predict(b)),alpha))\n        _,alpha=min(vals);selections[kind+\'_\'+family]={\'alpha\':alpha,\'validation_mse\':min(vals)[0]}\n        m=model(dev,nums,kind,alpha);m.fit(dev,dev.separation_at_pass_forward,reg__sample_weight=weights(dev));p=m.predict(t)\n        name=kind+\'_\'+family;pred[name]=p\n        results.append({\'model\':name,\'test_player_balanced_mse\':score(t,p),\'test_rmse_yards\':score(t,p)**.5,\'validation_mse\':min(vals)[0]})\n    # Primary contrast is ridge tracking vs ridge conventional. Others exploratory.\n    losses=pd.DataFrame({\'nfl_id\':t.nfl_id,\'baseline\':(t.separation_at_pass_forward-pred[\'ridge_conventional\'])**2,\'tracking\':(t.separation_at_pass_forward-pred[\'ridge_tracking\'])**2}).groupby(\'nfl_id\').mean()\n    delta=(losses.baseline-losses.tracking).to_numpy();rng=np.random.default_rng(SEED)\n    boots=np.mean(rng.choice(delta,(5000,len(delta)),replace=True),axis=1)\n    report={\'status\':\'actual_data_completed\',\'primary_delta_mse_yards_squared\':float(delta.mean()),\'player_bootstrap_ci95\':np.quantile(boots,[.025,.975]).tolist(),\n      \'test_players\':len(delta),\'test_routes\':len(t),\'results\':results,\'selection\':selections,\n      \'interpretation\':\'positive delta favors motion; CI crossing zero is inconclusive; cohort-generalization requires more years\',\n      \'selection_warning\':\'Exploratory models cannot replace the preregistered primary contrast after observing test labels.\'}\n    pd.DataFrame(results).to_csv(out/\'model_comparisons.csv\',index=False)\n    (out/\'experiment_results.json\').write_text(json.dumps(report,indent=2))\n    pp=t[[\'nfl_id\',\'game_id\',\'play_id\']].copy()\n    for k,v in pred.items():pp[k]=v\n    pp.to_csv(out/\'private_test_predictions.csv\',index=False)\n    return report\n\ndef plot_motion(diag,path,label):\n    import matplotlib;matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    plt.rcParams.update({\'font.family\':\'DejaVu Sans\',\'text.color\':\'#e6edf6\',\'axes.labelcolor\':\'#c2cfde\',\'xtick.color\':\'#8aa0b6\',\'ytick.color\':\'#8aa0b6\',\'axes.edgecolor\':\'#304359\',\'axes.facecolor\':\'#102236\',\'figure.facecolor\':\'#071421\'})\n    fig,ax=plt.subplots(1,3,figsize=(16,5),layout=\'constrained\');q=diag[\'q\'];t=diag[\'t\'];s=diag[\'speed\'];a=diag[\'tangential_acceleration\']\n    sc=ax[0].scatter(q[:,0],q[:,1],c=s,cmap=\'plasma\',s=80,zorder=3);ax[0].plot(q[:,0],q[:,1],color=\'#39dbbd\',alpha=.8);ax[0].set_aspect(\'equal\');ax[0].set(xlabel=\'Along entry direction · yards\',ylabel=\'Across entry direction · yards\',title=\'THE SHAPE OF THE CUT\');fig.colorbar(sc,ax=ax[0],label=\'Speed · yd/s\',shrink=.7)\n    ax[1].plot(t,s,lw=3,color=\'#39dbbd\');ax[1].axvline(0,color=\'#efb455\',ls=\':\');ax[1].set(xlabel=\'Seconds from largest turn\',ylabel=\'Speed · yd/s\',title=\'BRAKE → TURN → RECOVER\')\n    ax[2].plot(s,a,lw=2,color=\'#efb455\');ax[2].scatter(s,a,c=t,cmap=\'cool\',s=40);ax[2].axhline(0,color=\'#304359\');ax[2].set(xlabel=\'Speed · yd/s\',ylabel=\'Tangential acceleration · yd/s²\',title=\'MOVEMENT PHASE PORTRAIT\')\n    for z in ax:z.grid(alpha=.12)\n    fig.suptitle(label,fontsize=17,fontweight=\'bold\');fig.savefig(path,dpi=150);plt.close(fig)\n\ndef run(root,out):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    (out/\'inventory.json\').write_text(json.dumps(inventory(root),indent=2))\n    d,f,audit=prepare(root,out);report=compare(d,out)\n    # Selected example by sorted ID, independent of outcomes. Read only WR tracking rows.\n    chosen=f.sort_values([\'nfl_id\',\'event_id\']).iloc[0];parts=[]\n    for c in pd.read_csv(Path(root)/\'combine_tracking.csv\',chunksize=200000):parts.append(c[c.event_id.eq(chosen.event_id)&c.entity_type.eq(\'PLAYER\')])\n    g=pd.concat(parts);_,diag=movement(g);plot_motion(diag,out/\'actual_combine_geometry.png\',\'Observed Combine trajectory · selected without NFL outcomes\')\n    (out/\'actual_combine_geometry.html\').write_text(motion_html(diag,\'Observed Combine attempt · selected independently of NFL outcomes\'))\n    return report\n\n\ndef game_transfer(root,d,out):\n    """Secondary descriptive mechanics replication; game motion never a prospect predictor."""\n    out=Path(out);pieces=[];ids=set(d.nfl_id)\n    keys=d[[\'game_id\',\'play_id\',\'nfl_id\']].drop_duplicates()\n    for year in (2023,2024,2025):\n        p=Path(root)/f\'game_tracking_{year}.csv\';parts=[]\n        for c in pd.read_csv(p,chunksize=200000):\n            c=c[c.nfl_id.isin(ids)].merge(keys,on=[\'game_id\',\'play_id\',\'nfl_id\'],how=\'inner\',validate=\'many_to_one\');parts.append(c)\n        frames=pd.concat(parts,ignore_index=True)\n        for key,g in frames.groupby([\'game_id\',\'play_id\',\'nfl_id\']):\n            g=g.sort_values(\'time\');snap=g[g.event.eq(\'ball_snap\')];release=g[g.event.eq(\'pass_forward\')]\n            if snap.empty or release.empty:continue\n            g=g[g.time.ge(snap.time.iloc[0])&g.time.le(release.time.iloc[0])]\n            try:\n                f,_=movement(g);pieces.append(dict(game_id=key[0],play_id=key[1],nfl_id=key[2],**f))\n            except ValueError:continue\n    pd.DataFrame(pieces).to_csv(out/\'private_game_motion.csv\',index=False)\n    return {\'status\':\'descriptive_only\',\'qualifying_game_turns\':len(pieces),\'note\':\'No 22-player coverage reconstruction. Game movement is outcome-side diagnostic, never a pre-draft predictor.\'}\n\ndef motion_html(diag,label):\n    """Offline SVG animation of measured samples, not a generated football replay."""\n    import html\n    data=json.dumps({k:diag[k].tolist() for k in (\'t\',\'q\',\'speed\',\'tangential_acceleration\')})\n    template=\'\'\'<!doctype html><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Inside the cut</title><style>\nbody{background:#071421;color:#e6edf6;font:17px system-ui;margin:0;padding:4vw}main{max-width:1000px;margin:auto}.tag{color:#39dbbd;font-size:12px;letter-spacing:2px}h1{font-size:clamp(28px,5vw,52px)}.layout{display:grid;grid-template-columns:2fr 1fr;gap:24px}svg{background:#102236;width:100%;border-radius:20px}button{background:#39dbbd;color:#071421;border:0;border-radius:9px;padding:12px 22px;font-weight:700}input{width:100%;accent-color:#39dbbd}small{color:#9cafc3}strong{font-size:40px;display:block}.metric{margin:22px 0}@media(max-width:650px){.layout{grid-template-columns:1fr}} </style>\n<main><div class="tag">MOVEMENT LAB · FRAME BY FRAME</div><h1>Inside the cut.</h1><p>LABEL</p><div class="layout"><svg id="field" viewBox="0 0 600 400" role="img" aria-label="Canonical trajectory in yards"><g id="grid"></g><path id="path" fill="none" stroke="#39dbbd" stroke-width="3"/><g id="samples"></g><circle id="player" r="12" fill="#efb455" stroke="#071421" stroke-width="4"/><text x="22" y="380" fill="#9cafc3" font-size="14">Canonical entry frame · yards</text></svg><aside><div class="metric">Time from detected turn<strong id="time"></strong><small>seconds</small></div><div class="metric">Speed<strong id="speed"></strong><small>yards / second</small></div><div class="metric">Braking / reacceleration<strong id="acc"></strong><small>yards / second²</small></div></aside></div><p><button id="play">Play movement</button> <small>Eleven observed samples; interpolation animates the marker only.</small></p><input id="frame" type="range" min="0" max="10" step=".01" value="0" aria-label="Inspect tracking moment"><p><small>The line is the sampled path. Color indicates speed. A positive acceleration value means the athlete is regaining speed. The dot does not represent a defender or a missing NFL player.</small></p></main>\n<script>const D=DATA;const ns=\'http://www.w3.org/2000/svg\';const xs=D.q.map(p=>p[0]),ys=D.q.map(p=>p[1]);let xmin=Math.min(...xs),xmax=Math.max(...xs),ymin=Math.min(...ys),ymax=Math.max(...ys);const scale=Math.min(500/Math.max(xmax-xmin,.2),280/Math.max(ymax-ymin,.2));const P=D.q.map(p=>[50+(p[0]-xmin)*scale,330-(p[1]-ymin)*scale]);function node(tag,attrs,parent){let e=document.createElementNS(ns,tag);for(let k in attrs)e.setAttribute(k,attrs[k]);parent.appendChild(e);return e}for(let x=Math.floor(xmin);x<=xmax;x++){const z=50+(x-xmin)*scale;node(\'line\',{x1:z,y1:20,x2:z,y2:350,stroke:\'#233c51\'},document.querySelector(\'#grid\'));const t=node(\'text\',{x:z,y:365,fill:\'#9cafc3\',\'font-size\':12},document.querySelector(\'#grid\'));t.textContent=x;}document.querySelector(\'#path\').setAttribute(\'d\',P.map((p,i)=>(i?\'L\':\'M\')+p.join(\',\')).join(\' \'));let low=Math.min(...D.speed),high=Math.max(...D.speed);P.forEach((p,i)=>node(\'circle\',{cx:p[0],cy:p[1],r:5,fill:`hsl(${180-140*(D.speed[i]-low)/Math.max(high-low,.01)} 75% 60%)`},document.querySelector(\'#samples\')));let f=document.querySelector(\'#frame\'),playing=false,last=0;function show(){let z=+f.value,i=Math.min(9,Math.floor(z)),a=z-i;let p=P[i].map((v,k)=>v+(P[i+1][k]-v)*a);document.querySelector(\'#player\').setAttribute(\'cx\',p[0]);document.querySelector(\'#player\').setAttribute(\'cy\',p[1]);for(let [id,key] of [[\'time\',\'t\'],[\'speed\',\'speed\'],[\'acc\',\'tangential_acceleration\']])document.querySelector(\'#\'+id).textContent=(D[key][i]+(D[key][i+1]-D[key][i])*a).toFixed(2)}f.oninput=()=>{playing=false;document.querySelector(\'#play\').textContent=\'Play movement\';show()};document.querySelector(\'#play\').onclick=()=>{playing=!playing;if(+f.value>=10)f.value=0;document.querySelector(\'#play\').textContent=playing?\'Pause\':\'Play movement\';last=performance.now()};function tick(now){if(playing){f.value=Math.min(10,+f.value+(now-last)/300);show();if(+f.value>=10){playing=false;document.querySelector(\'#play\').textContent=\'Replay movement\'}}last=now;requestAnimationFrame(tick)}show();requestAnimationFrame(tick);</script>\'\'\'\n    return template.replace(\'LABEL\',html.escape(label)).replace(\'DATA\',data)\n\nif __name__==\'__main__\':\n    ap=argparse.ArgumentParser();ap.add_argument(\'--data\',required=True);ap.add_argument(\'--out\',default=\'private_results\');args=ap.parse_args();print(json.dumps(run(args.data,args.out),indent=2))\n', 'upstream/repo_predict.py': '"""Prospect-balanced compression and development-only model selection.\n\n2025 was already inspected before this push; its results are exploratory.\n"""\nfrom pathlib import Path\nimport json,time\nimport numpy as np,pandas as pd\nfrom sklearn.base import BaseEstimator,TransformerMixin\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import StandardScaler,OneHotEncoder\nfrom sklearn.decomposition import PCA\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.linear_model import Ridge\nfrom sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor\nfrom sklearn.model_selection import StratifiedGroupKFold\nimport analysis as A\n\nPHYS=A.BASE+[\'hand_size\',\'arm_length\',\'wing_span\',\'bench_reps\',\'ngs_athleticism_score\',\'ngs_college_production_score\',\'ngs_final_score\']\nNUM=A.CONTEXT+[\'quarter\']\nCAT=A.CATS+[\'team_coverage_type\',\'receiver_alignment\',\'team_abbr\',\'defense_team\']\nCONFIGS=[{\'reg\':\'ridge\',\'alpha\':1,\'k\':4},{\'reg\':\'ridge\',\'alpha\':10,\'k\':4},{\'reg\':\'ridge\',\'alpha\':100,\'k\':8},{\'reg\':\'forest\',\'alpha\':0,\'k\':4},{\'reg\':\'extra\',\'alpha\':0,\'k\':8}]\n\nclass ProspectEncoder(BaseEstimator,TransformerMixin):\n    def __init__(self,motion=(),k=4):self.motion=motion;self.k=k\n    def fit(self,x,y=None):\n        players=x.drop_duplicates(\'nfl_id\');self.fit_player_ids_=set(players.nfl_id)\n        self.phys=Pipeline([(\'impute\',SimpleImputer(strategy=\'median\',add_indicator=True,keep_empty_features=True)),(\'scale\',StandardScaler())]);self.phys.fit(players[PHYS])\n        self.num=Pipeline([(\'impute\',SimpleImputer(strategy=\'median\',add_indicator=True,keep_empty_features=True)),(\'scale\',StandardScaler())]);self.num.fit(x[NUM])\n        self.cat=OneHotEncoder(handle_unknown=\'ignore\',sparse_output=False);self.cat.fit(x[CAT].astype(str).where(x[CAT].notna(),\'MISSING\'))\n        if self.motion:\n            m=players[list(self.motion)].to_numpy(float);self.lo=np.nanquantile(m,.01,axis=0);self.hi=np.nanquantile(m,.99,axis=0)\n            self.lo=np.nan_to_num(self.lo);self.hi=np.nan_to_num(self.hi)\n            self.mi=SimpleImputer(strategy=\'median\',add_indicator=True,keep_empty_features=True);m=self.mi.fit_transform(np.clip(m,self.lo,self.hi));self.ms=StandardScaler();m=self.ms.fit_transform(m)\n            self.pca=PCA(n_components=min(self.k,len(players)-1,m.shape[1]),svd_solver=\'full\');self.pca.fit(m)\n        return self\n    def transform(self,x):\n        blocks=[self.phys.transform(x[PHYS]),self.num.transform(x[NUM]),self.cat.transform(x[CAT].astype(str).where(x[CAT].notna(),\'MISSING\'))]\n        if self.motion:\n            m=np.clip(x[list(self.motion)].to_numpy(float),self.lo,self.hi);blocks.append(self.pca.transform(self.ms.transform(self.mi.transform(m))))\n        return np.column_stack(blocks)\n\ndef columns(panel,family):\n    cols=[c for c in panel if c.startswith(\'motion__\')]\n    if family==\'baseline\':return ()\n    if family==\'summary\':return tuple(c for c in cols if c.split(\'__\')[-1] in A.MOTION)\n    if family==\'signature\':return tuple(c for c in cols if c.split(\'__\')[-1].startswith(\'sig_\'))\n    if family==\'dynamics\':return tuple(c for c in cols if c.split(\'__\')[-1].startswith((\'memory_\',\'hankel_\',\'profile_\')))\n    if family==\'all\':return tuple(cols)\n    raise ValueError(\'family\')\n\ndef make(motion,config):\n    enc=ProspectEncoder(motion,config[\'k\'])\n    if config[\'reg\']==\'ridge\':reg=Ridge(alpha=config[\'alpha\'])\n    elif config[\'reg\']==\'forest\':reg=RandomForestRegressor(n_estimators=150,max_depth=6,min_samples_leaf=20,max_features=.8,n_jobs=2,random_state=A.SEED)\n    else:reg=ExtraTreesRegressor(n_estimators=150,max_depth=8,min_samples_leaf=20,max_features=.8,n_jobs=2,random_state=A.SEED)\n    return Pipeline([(\'enc\',enc),(\'reg\',reg)])\n\ndef fit_predict(train,test,motion,config):\n    if set(train.nfl_id)&set(test.nfl_id):raise ValueError(\'player overlap\')\n    m=make(motion,config);m.fit(train,train.separation_at_pass_forward,reg__sample_weight=A.weights(train));return m,m.predict(test)\n\ndef prepare(root,out,data_root=None,analysis_out=None):\n    root,out=Path(root),Path(out);data_root=Path(data_root) if data_root else root/\'data\';analysis_out=Path(analysis_out) if analysis_out else root/\'private_results\';d=pd.read_pickle(analysis_out/\'private_analysis_rows.pkl\') if (analysis_out/\'private_analysis_rows.pkl\').exists() else pd.read_csv(analysis_out/\'private_analysis_rows.csv\');panel=pd.read_csv(out/\'private_motion_panel.csv\');d=d.drop(columns=A.MOTION).merge(panel,on=\'nfl_id\',validate=\'many_to_one\')\n    g=pd.read_csv(data_root/\'games.csv\').set_index(\'game_id\');home=d.game_id.map(g.home_team_abbr);away=d.game_id.map(g.visitor_team_abbr);d[\'defense_team\']=np.where(home==d.team_abbr,away,home)\n    if d.nfl_id.nunique()!=64 or len(d)!=7348:raise ValueError(\'frozen population changed\')\n    return d,panel\n\ndef bootstrap(test,p0,p1):\n    x=pd.DataFrame({\'id\':test.nfl_id,\'delta\':(test.separation_at_pass_forward-p0)**2-(test.separation_at_pass_forward-p1)**2}).groupby(\'id\').delta.mean().to_numpy();rng=np.random.default_rng(A.SEED);boots=rng.choice(x,(5000,len(x)),replace=True).mean(axis=1)\n    return {\'delta_mse\':float(x.mean()),\'ci95\':np.quantile(boots,[.025,.975]).tolist(),\'players\':len(x)}\n\ndef cohort_trial(root,out,data_root=None,analysis_out=None):\n    root,out=Path(root),Path(out);d,panel=prepare(root,out,data_root,analysis_out);train=d[d.draft_year.eq(2023)];val=d[d.draft_year.eq(2024)];dev=d[d.draft_year.isin([2023,2024])];test=d[d.draft_year.eq(2025)]\n    bank=[];selected={};preds={};start=time.perf_counter()\n    for family in [\'baseline\',\'summary\',\'signature\',\'dynamics\',\'all\']:\n        cc=columns(panel,family);candidates=[]\n        for i,config in enumerate(CONFIGS):\n            m,p=fit_predict(train,val,cc,config);v=A.score(val,p);candidates.append((v,i));bank.append({\'family\':family,\'config\':i,\'validation_mse\':v});print(\'Validation\',family,i,round(v,6),flush=True)\n        v,i=min(candidates);m,p=fit_predict(dev,test,cc,CONFIGS[i]);preds[family]=p;selected[family]={\'config\':CONFIGS[i],\'config_index\':i,\'validation_mse\':v,\'test_mse\':A.score(test,p),\'test_rmse_yards\':A.score(test,p)**.5,\'motion_input_columns\':len(cc),\'motion_latent_dimension\':m.named_steps[\'enc\'].pca.n_components_ if cc else 0}\n    for family in [\'summary\',\'signature\',\'dynamics\',\'all\']:selected[family][\'paired_vs_strong_conventional\']=bootstrap(test,preds[\'baseline\'],preds[family])\n    original=pd.read_csv((Path(analysis_out) if analysis_out else root/\'private_results\')/\'private_test_predictions.csv\');q=test[[\'nfl_id\',\'game_id\',\'play_id\']].merge(original,on=[\'nfl_id\',\'game_id\',\'play_id\'],validate=\'one_to_one\')\n    old=q.forest_conventional.to_numpy();selected[\'all\'][\'paired_vs_original_forest\']=bootstrap(test,old,preds[\'all\'])\n    result={\'status\':\'exploratory_actual_data\',\'test_previously_inspected\':True,\'selection\':\'2023 train; 2024 tune; refit 2023+2024; 2025 exploratory transport\',\'primary_new_family\':\'all\',\'selected\':selected,\'validation_bank\':bank,\'same_candidates_per_family\':len(CONFIGS),\'test_players\':test.nfl_id.nunique(),\'test_routes\':len(test),\'elapsed_seconds\':time.perf_counter()-start,\'interpretation\':\'Ablations exploratory; superiority requires stronger development-only corroboration and fresh future validation.\'}\n    (out/\'cohort_trial.json\').write_text(json.dumps(result,indent=2));p=test[[\'nfl_id\',\'game_id\',\'play_id\']].copy()\n    for k,v in preds.items():p[k]=v\n    p.to_csv(out/\'private_cohort_predictions.csv\',index=False);print(json.dumps(result,indent=2),flush=True);return result\n\ndef nested_development(root,out,data_root=None,analysis_out=None):\n    root,out=Path(root),Path(out);d,panel=prepare(root,out,data_root,analysis_out);dev=d[d.draft_year.isin([2023,2024])].reset_index(drop=True);outer=StratifiedGroupKFold(n_splits=4,shuffle=True,random_state=A.SEED);preds={f:np.zeros(len(dev)) for f in [\'baseline\',\'all\']};receipts=[]\n    for fold,(ii,jj) in enumerate(outer.split(dev,dev.draft_year,dev.nfl_id)):\n        tr,te=dev.iloc[ii],dev.iloc[jj];inner=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=A.SEED+fold+1)\n        for family in [\'baseline\',\'all\']:\n            cc=columns(panel,family);rank=[]\n            for ci,config in enumerate(CONFIGS):\n                losses=[]\n                for a,b in inner.split(tr,tr.draft_year,tr.nfl_id):\n                    ti,vi=tr.iloc[a],tr.iloc[b];_,p=fit_predict(ti,vi,cc,config);losses.append(A.score(vi,p))\n                rank.append((float(np.mean(losses)),ci))\n            _,ci=min(rank);_,p=fit_predict(tr,te,cc,CONFIGS[ci]);preds[family][jj]=p;receipts.append({\'outer_fold\':fold,\'family\':family,\'chosen_config\':ci,\'inner_scores\':[v for v,i in rank],\'outer_mse\':A.score(te,p),\'outer_players\':te.nfl_id.nunique()});print(\'Nested\',fold,family,\'MSE\',round(A.score(te,p),6),flush=True)\n    result={\'status\':\'development_only_nested_player_validation\',\'years\':[2023,2024],\'test_2025_outcomes_used\':False,\'drill_dictionary_scope\':\'fixed using full supplied Combine drill availability, including 2025; no outcome-based selection\',\'players\':dev.nfl_id.nunique(),\'routes\':len(dev),\'baseline_mse\':A.score(dev,preds[\'baseline\']),\'motion_mse\':A.score(dev,preds[\'all\']),\'paired_improvement\':bootstrap(dev,preds[\'baseline\'],preds[\'all\']),\'fold_receipts\':receipts,\'scope\':\'unseen players in pooled earlier seasons; complements but does not replace a future-season holdout\'}\n    (out/\'nested_development.json\').write_text(json.dumps(result,indent=2));p=dev[[\'nfl_id\',\'game_id\',\'play_id\']].copy()\n    for k,v in preds.items():p[k]=v\n    p.to_csv(out/\'private_nested_predictions.csv\',index=False);print(json.dumps(result,indent=2),flush=True);return result\n\nif __name__==\'__main__\':\n    import argparse\n    ap=argparse.ArgumentParser();ap.add_argument(\'--nested\',action=\'store_true\');args=ap.parse_args();root=Path(__file__).parent;out=root/\'private_repo_results\'\n    if args.nested:nested_development(root,out)\n    else:cohort_trial(root,out)\n', 'upstream/mechanism.py': '"""Exploratory, player-cross-fitted context residuals and three-second mechanics.\n\nNo 2025 labels select features, regularization, or nuisance specifications.\nEarlier experiments already inspected 2025, so transport remains exploratory.\n"""\nfrom pathlib import Path\nimport json\nimport numpy as np\nimport pandas as pd\nfrom scipy.signal import savgol_filter\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import StandardScaler, OneHotEncoder\nfrom sklearn.ensemble import ExtraTreesRegressor\nfrom sklearn.linear_model import Ridge\nfrom sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold\nimport analysis as A\nimport repo_predict as R\n\nMAP={\'IN\':[\'DAGGER_ROUTE_LEFT\'],\'OUT\':[\'SPEED_OUT_ROUTE_LEFT\'],\n     \'SLANT\':[\'SLANT_ROUTE_LEFT\'],\'HITCH\':[\'CURL_ROUTE_RIGHT\',\'COMEBACK_ROUTE_RIGHT\']}\nFEATURES=[\'entry\',\'retention\',\'recovery\',\'braking\',\'propulsion\',\'turn\',\'bend\',\n          \'area\',\'speed_mode_1\',\'speed_mode_2\',\'speed_mode_3\',\'speed_mode_4\']\nALPHAS=[10.,100.,1000.]\n\ndef trajectory(g):\n    _,diag=A.movement(g)\n    g=g.sort_values(\'time\');c=diag[\'cut_index\']\n    if c<15 or c+15>=len(g):raise ValueError(\'insufficient three-second turn window\')\n    p=savgol_filter(g[[\'x\',\'y\']].to_numpy(float),7,3,axis=0)\n    v=np.gradient(p,.1,axis=0);s=np.linalg.norm(v,axis=1)\n    th=np.unwrap(np.arctan2(v[:,1],v[:,0]));sl=slice(c-15,c+16)\n    tangent=v[c-5]/max(s[c-5],1e-9);normal=np.array([-tangent[1],tangent[0]])\n    q=(p[sl]-p[c-15])@np.stack([tangent,normal],axis=1)\n    turn=th[c+5]-th[c-5];q[:,1]*=1 if turn>=0 else -1\n    speed=s[sl];acc=np.gradient(speed,.1);entry=float(speed[5:11].mean())\n    if entry<1:raise ValueError(\'insufficient approach speed\')\n    arc=np.linalg.norm(np.diff(q,axis=0),axis=1).sum()\n    mode=np.polynomial.legendre.legfit(np.linspace(-1,1,31),speed/entry,4)\n    area=.5*np.sum(q[:-1,0]*q[1:,1]-q[:-1,1]*q[1:,0])/max(arc**2,1e-9)\n    vals=[entry,float(speed[10:21].min()/entry),float(speed[21:27].mean()/entry),\n          float(np.trapezoid(np.maximum(-acc[5:21],0),dx=.1)/entry),\n          float(np.trapezoid(np.maximum(acc[15:27],0),dx=.1)/entry),abs(float(turn)),\n          float(np.trapezoid(np.gradient(th,.1)[sl]**2/np.maximum(speed,1),dx=.1)),\n          area,*mode[1:]]\n    return dict(zip(FEATURES,map(float,vals))),{\'q\':q,\'speed\':speed,\'t\':np.arange(-15,16)*.1}\n\ndef prepare(data,out):\n    data,out=Path(data),Path(out);out.mkdir(parents=True,exist_ok=True)\n    players=pd.read_csv(data/\'players.csv\');tr=pd.read_csv(data/\'combine_tracking.csv\')\n    tr=tr[tr.nfl_id.isin(players.loc[players.nfl_position.eq(\'WR\'),\'nfl_id\'])&tr.entity_type.eq(\'PLAYER\')&tr.drill_type.eq(\'SKILL_DRILLS_WR\')].copy()\n    tr[\'drill_name\']=tr.drill_name.replace({\'SPEED_OUT_LEFT\':\'SPEED_OUT_ROUTE_LEFT\'})\n    attempts=[];rejections={};example=None\n    for key,g in tr.groupby([\'nfl_id\',\'draft_year\',\'event_id\'],sort=True):\n        if g.drill_name.iloc[0] not in {x for v in MAP.values() for x in v}:continue\n        try:f,diag=trajectory(g)\n        except ValueError as e:rejections[str(e)]=rejections.get(str(e),0)+1;continue\n        if example is None:example=diag\n        attempts.append(dict(nfl_id=key[0],drill_name=g.drill_name.iloc[0],**f))\n    feats=pd.DataFrame(attempts).groupby([\'nfl_id\',\'drill_name\'])[FEATURES].median()\n    blocks=[]\n    for route,drills in MAP.items():\n        b=feats.loc[feats.index.get_level_values(\'drill_name\').isin(drills)].groupby(\'nfl_id\').mean().reset_index();b[\'route_ran\']=route;blocks.append(b)\n    panel=pd.concat(blocks,ignore_index=True)\n    d=pd.read_csv(data/\'player_play.csv\').merge(players[[\'nfl_id\',\'draft_year\',\'nfl_position\']],on=\'nfl_id\',validate=\'many_to_one\')\n    games=pd.read_csv(data/\'games.csv\');d=d.merge(games[[\'game_id\',\'season\',\'season_type\',\'home_team_abbr\',\'visitor_team_abbr\']],on=\'game_id\',validate=\'many_to_one\')\n    eligible=d[d.nfl_position.eq(\'WR\')&d.season.eq(d.draft_year)&d.season_type.eq(\'REG\')&d.route_ran.isin(MAP)&d.play_nullified_by_penalty.eq(\'N\')].copy()\n    missing=eligible.groupby([\'draft_year\',\'team_coverage_man_zone\']).separation_at_pass_forward.apply(lambda x:float(x.isna().mean()))\n    d=eligible[np.isfinite(eligible.separation_at_pass_forward)&eligible.separation_at_pass_forward.ge(0)].copy()\n    counts=d.groupby(\'nfl_id\').size();d=d[d.nfl_id.isin(counts[counts>=20].index)]\n    d=d.merge(pd.read_csv(data/\'combine_results.csv\'),on=[\'nfl_id\',\'draft_year\'],validate=\'many_to_one\').merge(panel,on=[\'nfl_id\',\'route_ran\'],how=\'left\',validate=\'many_to_one\')\n    d[\'defense_team\']=np.where(d.team_abbr.eq(d.home_team_abbr),d.visitor_team_abbr,d.home_team_abbr)\n    d=d.reset_index(drop=True);d.to_pickle(out/\'private_rows.pkl\')\n    audit={\'players_by_class\':{str(k):int(v) for k,v in d.groupby(\'draft_year\').nfl_id.nunique().items()},\'routes_by_class\':{str(k):int(v) for k,v in d.groupby(\'draft_year\').size().items()},\'accepted_attempts\':len(attempts),\'rejected_attempts\':rejections,\'missing_movement_fraction\':float(d.entry.isna().mean()),\'missing_separation_by_class_coverage\':{str(k):v for k,v in missing.items()},\'movement_window_seconds\':3,\'features\':FEATURES,\'mapping\':MAP,\'feature_selection\':\'fixed mechanics and drill mapping; no outcome-directed selection\'}\n    (out/\'mechanism_population.json\').write_text(json.dumps(audit,indent=2))\n    if example:\n        import matplotlib;matplotlib.use(\'Agg\')\n        import matplotlib.pyplot as plt\n        fig,ax=plt.subplots(1,2,figsize=(12,4),layout=\'constrained\')\n        q=example[\'q\'];sc=ax[0].scatter(q[:,0],q[:,1],c=example[\'speed\'],cmap=\'viridis\');ax[0].plot(q[:,0],q[:,1],alpha=.4);ax[0].set_aspect(\'equal\');ax[0].set(xlabel=\'Entry direction · yards\',ylabel=\'Across entry · yards\',title=\'Three-second measured turn\');fig.colorbar(sc,ax=ax[0],label=\'Speed · yd/s\')\n        ax[1].plot(example[\'t\'],example[\'speed\'],lw=3);ax[1].axvline(0,ls=\':\',color=\'gray\');ax[1].set(xlabel=\'Seconds from detected cut\',ylabel=\'Speed · yd/s\',title=\'Approach, brake and recover\')\n        fig.savefig(out/\'mechanism_geometry.png\',dpi=160);plt.close(fig)\n    return d,audit\n\ndef nuisance():\n    num=make_pipeline(SimpleImputer(strategy=\'median\',add_indicator=True),StandardScaler())\n    cat=make_pipeline(SimpleImputer(strategy=\'most_frequent\'),OneHotEncoder(handle_unknown=\'ignore\',sparse_output=False))\n    pre=ColumnTransformer([(\'num\',num,R.NUM),(\'cat\',cat,R.CAT)])\n    return make_pipeline(pre,ExtraTreesRegressor(n_estimators=100,max_depth=6,min_samples_leaf=20,max_features=.8,random_state=A.SEED,n_jobs=2))\n\ndef crossfit(tr,te,seed):\n    assert not(set(tr.nfl_id)&set(te.nfl_id))\n    oof=np.zeros(len(tr));folds=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=seed)\n    for a,b in folds.split(tr,tr.draft_year,tr.nfl_id):\n        m=nuisance();m.fit(tr.iloc[a],tr.iloc[a].separation_at_pass_forward,extratreesregressor__sample_weight=A.weights(tr.iloc[a]));oof[b]=m.predict(tr.iloc[b])\n    m=nuisance();m.fit(tr,tr.separation_at_pass_forward,extratreesregressor__sample_weight=A.weights(tr))\n    z=tr.copy();z[\'residual\']=z.separation_at_pass_forward-oof\n    # Equal total influence per prospect; multiple route/coverage means are partial pooling units.\n    cols=R.PHYS+FEATURES\n    groups=z.groupby([\'nfl_id\',\'route_ran\',\'team_coverage_man_zone\'],dropna=False)\n    agg=groups[cols+[\'residual\']].mean().reset_index()\n    return agg,m.predict(te)\n\ndef correction(agg,te,alpha,motion):\n    cols=R.PHYS+(FEATURES if motion else [])\n    # Fit scaling only on one record per player/route/coverage; no held-out features.\n    p=make_pipeline(SimpleImputer(strategy=\'median\',add_indicator=True,keep_empty_features=True),StandardScaler())\n    x=p.fit_transform(agg[cols]);xt=p.transform(te[cols])\n    cats=OneHotEncoder(handle_unknown=\'ignore\',sparse_output=False);c=cats.fit_transform(agg[[\'route_ran\',\'team_coverage_man_zone\']].fillna(\'MISSING\'));ct=cats.transform(te[[\'route_ran\',\'team_coverage_man_zone\']].fillna(\'MISSING\'))\n    if motion:\n        # Predefined physical and coverage interactions, after train-only scaling.\n        j=cols.index(\'combine_weight\');k=len(R.PHYS);man=agg.team_coverage_man_zone.eq(\'MAN_COVERAGE\').to_numpy()[:,None];mant=te.team_coverage_man_zone.eq(\'MAN_COVERAGE\').to_numpy()[:,None]\n        x=np.c_[x,x[:,k:k+len(FEATURES)]*x[:,j,None],x[:,k:k+len(FEATURES)]*man]\n        xt=np.c_[xt,xt[:,k:k+len(FEATURES)]*xt[:,j,None],xt[:,k:k+len(FEATURES)]*mant]\n    m=Ridge(alpha=alpha);m.fit(np.c_[x,c],agg.residual,sample_weight=A.weights(agg));return m.predict(np.c_[xt,ct])\n\ndef run(data,out):\n    out=Path(out);d,audit=prepare(data,out);dev=d[d.draft_year.isin([2023,2024])].reset_index(drop=True);test=d[d.draft_year.eq(2025)].reset_index(drop=True)\n    preds={k:np.zeros(len(dev)) for k in [\'context\',\'physical\',\'motion\']};records=[]\n    outer=StratifiedGroupKFold(n_splits=4,shuffle=True,random_state=A.SEED+80)\n    for fold,(a,b) in enumerate(outer.split(dev,dev.draft_year,dev.nfl_id)):\n        tr,te=dev.iloc[a].reset_index(drop=True),dev.iloc[b];losses={k:np.zeros(len(ALPHAS)) for k in [\'physical\',\'motion\']}\n        inner=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=A.SEED+81+fold)\n        for i,j in inner.split(tr,tr.draft_year,tr.nfl_id):\n            ti,vi=tr.iloc[i].reset_index(drop=True),tr.iloc[j];agg,base=crossfit(ti,vi,A.SEED+90+fold)\n            for family in losses:\n                for ci,alpha in enumerate(ALPHAS):losses[family][ci]+=A.score(vi,base+correction(agg,vi,alpha,family==\'motion\'))/3\n        agg,base=crossfit(tr,te,A.SEED+100+fold);preds[\'context\'][b]=base\n        for family in losses:\n            ci=int(np.argmin(losses[family]));p=base+correction(agg,te,ALPHAS[ci],family==\'motion\');preds[family][b]=p;records.append({\'fold\':fold,\'family\':family,\'alpha\':ALPHAS[ci],\'inner_mse\':losses[family].tolist(),\'outer_mse\':A.score(te,p)})\n        print(\'Mechanism outer\',fold,{k:round(A.score(te,preds[k][b]),6) for k in preds},flush=True)\n    nested={\'mse\':{k:A.score(dev,p) for k,p in preds.items()},\'paired_motion_increment\':R.bootstrap(dev,preds[\'physical\'],preds[\'motion\']),\'folds\':records,\'players\':int(dev.nfl_id.nunique()),\'routes\':len(dev)}\n    # Forward-cohort selection, then refit earlier years; no 2025 tuning.\n    tr=dev[dev.draft_year.eq(2023)].reset_index(drop=True);va=dev[dev.draft_year.eq(2024)];agg,base=crossfit(tr,va,A.SEED+200);chosen={}\n    for family in [\'physical\',\'motion\']:\n        losses=[A.score(va,base+correction(agg,va,alpha,family==\'motion\')) for alpha in ALPHAS];ci=int(np.argmin(losses));chosen[family]={\'alpha\':ALPHAS[ci],\'validation_mse\':losses}\n    agg,base=crossfit(dev,test,A.SEED+201);tp={\'context\':base}\n    for family in chosen:tp[family]=base+correction(agg,test,chosen[family][\'alpha\'],family==\'motion\')\n    transport={\'mse\':{k:A.score(test,p) for k,p in tp.items()},\'paired_motion_increment\':R.bootstrap(test,tp[\'physical\'],tp[\'motion\']),\'selection\':chosen,\'players\':int(test.nfl_id.nunique()),\'routes\':len(test)}\n    subsets={}\n    for label,rows,pp in [(\'nested\',dev,preds),(\'exploratory_2025\',test,tp)]:\n        mask=rows.team_coverage_man_zone.eq(\'MAN_COVERAGE\');n=rows[mask].groupby(\'nfl_id\').size();mask&=rows.nfl_id.isin(n[n>=5].index)\n        subsets[label]={\'man_coverage_players\':int(rows[mask].nfl_id.nunique()),\'man_coverage_routes\':int(mask.sum()),\'paired_motion_increment\':R.bootstrap(rows[mask],pp[\'physical\'][mask],pp[\'motion\'][mask])}\n    result={\'status\':\'completed_actual_data_exploratory\',\'population\':audit,\'nested_development\':nested,\'exploratory_2025\':transport,\'predefined_man_subset\':subsets,\'limitations\':[\'2025 previously inspected; no fresh holdout\',\'Residual targets are cross-fitted by player, not causal receiver effects\',\'Context-only nuisance fixed before this run; identical three-alpha selection budgets\',\'No correction for unmeasured quarterback, defender ability, or missing-not-at-random outcomes\',\'Earlier nested folds overlap prior research; this is corroboration, not independent confirmatory evidence\']}\n    (out/\'mechanism_results.json\').write_text(json.dumps(result,indent=2));print(json.dumps(result,indent=2),flush=True)\n    plot(result,out/\'mechanism_results.png\')\n    return result\n\ndef plot(result,path):\n    import matplotlib;matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    fig,ax=plt.subplots(1,2,figsize=(14,5),layout=\'constrained\')\n    for i,(name,key) in enumerate([(\'Earlier nested players\',\'nested_development\'),(\'Exploratory 2025\',\'exploratory_2025\')]):\n        vals=result[key][\'mse\'];ax[0].bar(np.arange(3)+i*.36,[np.sqrt(vals[k]) for k in [\'context\',\'physical\',\'motion\']],width=.34,label=name)\n    ax[0].set_xticks(np.arange(3)+.18,[\'Context only\',\'+ Conventional\',\'+ Mechanics\']);ax[0].set(ylabel=\'Player-balanced RMSE · yards\',title=\'Cross-fitted receiver contribution\');ax[0].legend()\n    vals=[result[\'nested_development\'][\'paired_motion_increment\'],result[\'exploratory_2025\'][\'paired_motion_increment\'],result[\'predefined_man_subset\'][\'nested\'][\'paired_motion_increment\'],result[\'predefined_man_subset\'][\'exploratory_2025\'][\'paired_motion_increment\']]\n    for i,v in enumerate(vals):ax[1].plot(v[\'ci95\'],[i,i],lw=4);ax[1].scatter(v[\'delta_mse\'],i,s=70)\n    ax[1].set_yticks(range(4),[\'Nested · all coverage\',\'2025 · all coverage\',\'Nested · man coverage\',\'2025 · man coverage\']);ax[1].axvline(0,color=\'gray\',ls=\':\');ax[1].set(xlabel=\'Conventional − mechanics MSE · yd²\',title=\'Positive favors mechanics; intervals are exploratory\')\n    fig.savefig(path,dpi=160);plt.close(fig)\n\ndef player_splits(rows,n_splits,seed):\n    """Randomize unique players within class, rather than near-fixed route counts."""\n    players=rows[[\'nfl_id\',\'draft_year\']].drop_duplicates().reset_index(drop=True)\n    if players.nfl_id.duplicated().any():raise ValueError(\'player spans multiple classes\')\n    split=StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=seed)\n    for a,b in split.split(players,players.draft_year):\n        yield np.flatnonzero(rows.nfl_id.isin(players.iloc[a].nfl_id)),np.flatnonzero(rows.nfl_id.isin(players.iloc[b].nfl_id))\n\ndef split_stability(data,out):\n    """Repeat full nested selection; report conditional intervals and split variation."""\n    out=Path(out);d=pd.read_pickle(out/\'private_rows.pkl\');dev=d[d.draft_year.isin([2023,2024])].reset_index(drop=True)\n    results=[];deltas=[];partition_hashes=[]\n    for repeat in range(3):\n        seed=A.SEED+80+200*repeat;pred={k:np.zeros(len(dev)) for k in [\'context\',\'physical\',\'motion\']}\n        folds=list(player_splits(dev,4,seed))\n        import hashlib\n        partition_hashes.append(hashlib.sha256(json.dumps([sorted(map(int,dev.iloc[b].nfl_id.unique())) for a,b in folds]).encode()).hexdigest())\n        for fold,(a,b) in enumerate(folds):\n            tr,te=dev.iloc[a].reset_index(drop=True),dev.iloc[b];loss={k:np.zeros(3) for k in [\'physical\',\'motion\']}\n            for i,j in player_splits(tr,3,seed+1+fold):\n                agg,base=crossfit(tr.iloc[i].reset_index(drop=True),tr.iloc[j],seed+10+fold)\n                for family in loss:\n                    for ci,alpha in enumerate(ALPHAS):loss[family][ci]+=A.score(tr.iloc[j],base+correction(agg,tr.iloc[j],alpha,family==\'motion\'))/3\n            agg,base=crossfit(tr,te,seed+20+fold);pred[\'context\'][b]=base\n            for family in loss:pred[family][b]=base+correction(agg,te,ALPHAS[int(np.argmin(loss[family]))],family==\'motion\')\n        delta=pd.DataFrame({\'id\':dev.nfl_id,\'delta\':(dev.separation_at_pass_forward-pred[\'physical\'])**2-(dev.separation_at_pass_forward-pred[\'motion\'])**2}).groupby(\'id\').delta\n        deltas.append(delta.mean().to_numpy());results.append({\'outer_seed\':seed,\'mse\':{k:A.score(dev,p) for k,p in pred.items()},\'paired_motion_increment\':R.bootstrap(dev,pred[\'physical\'],pred[\'motion\'])})\n        print(\'Split stability\',repeat,results[-1],flush=True)\n    x=np.mean(deltas,axis=0);rng=np.random.default_rng(A.SEED);ci=np.quantile(rng.choice(x,(5000,len(x)),replace=True).mean(axis=1),[.025,.975])\n    if len(set(partition_hashes))!=3:raise ValueError(\'repeat partitions did not change\')\n    result={\'repeated_nested_results\':results,\'partition_hashes\':partition_hashes,\'split_method\':\'Stratify unique players by draft year, then map all their routes to each fold\',\'mean_player_delta_mse\':float(x.mean()),\'conditional_player_ci95\':ci.tolist(),\'positive_split_count\':int(sum(r[\'paired_motion_increment\'][\'delta_mse\']>0 for r in results)),\'scope\':\'Repeated folds share players; these are not three independent studies. Bootstrap averages each player across splits and conditions on fitted folds.\'}\n    (out/\'mechanism_split_stability.json\').write_text(json.dumps(result,indent=2));return result\n\nif __name__==\'__main__\':\n    root=Path(__file__).parent;run(root/\'data\',root/\'private_mechanism\')\n', 'upstream/observability.py': '"""Reconstructed movement-observability protocol. Prior WR 2025 results exploratory.\n\nAll transformations and model selection fit on training players only. Intermediate\nplayer rows remain private. Reconstructed implementation is not byte-identical to\nthe lost implementation; population and prediction differences are reported.\n"""\nfrom pathlib import Path\nimport json, hashlib, time\nimport numpy as np\nimport pandas as pd\nfrom scipy.optimize import least_squares\nfrom scipy.signal import savgol_filter\nfrom sklearn.base import BaseEstimator, TransformerMixin\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import StandardScaler, OneHotEncoder\nfrom sklearn.decomposition import PCA\nfrom sklearn.cross_decomposition import PLSRegression\nfrom sklearn.linear_model import Ridge\nfrom sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor\nimport analysis as A\nimport repo_predict as R\nfrom mechanism import player_splits\n\nPREFIX=[\'snap_speed\',\'early_speed\',\'early_acceleration\',\'early_distance\']\nCONFIGS=R.CONFIGS\n\ndef response(t,V,tau,delay,order):\n    z=np.maximum((np.asarray(t)-delay)/tau,0)\n    return V*(1-np.exp(-z)*(1+z if order==2 else 1))\n\ndef sprint_features(g):\n    g=g.sort_values(\'time\');t=pd.to_datetime(g.time,utc=True,format=\'ISO8601\').astype(\'int64\').to_numpy()/1e9;t-=t[0]\n    if len(t)<31 or not np.allclose(np.diff(t),.1,atol=.015):raise ValueError(\'short or irregular sprint\')\n    s=g.s.to_numpy(float)\n    if not np.isfinite(s).all() or t[-1]<3:raise ValueError(\'invalid speed\')\n    s=savgol_filter(s,7,3);f={}\n    for order in (1,2):\n        fit=least_squares(lambda z:response(t,*z,order)-s,[12,1,-.1],bounds=([6,.1,-1],[20,4,1]),max_nfev=400)\n        for name,value in zip([\'V\',\'tau\',\'delay\'],fit.x):f[f\'motion__order{order}_{name}\']=float(value)\n        f[f\'motion__order{order}_rmse\']=float(np.sqrt(np.mean(fit.fun**2)))\n        f[f\'motion__order{order}_condition\']=float(np.linalg.cond(fit.jac))\n    for tt in [.5,1,1.5,2,2.5,3]:f[f\'motion__speed_{tt}\']=float(np.interp(tt,t,s))\n    acceleration=np.gradient(s,.1)\n    for tt in [.5,1,1.5,2]:f[f\'motion__acceleration_{tt}\']=float(np.interp(tt,t,acceleration))\n    profile=np.interp(np.linspace(0,3,31),t,s)\n    for i,value in enumerate(np.polynomial.legendre.legfit(np.linspace(-1,1,31),profile,4)):f[f\'motion__legendre_{i}\']=float(value)\n    f[\'motion__speed_ratio\']=float(np.interp(1,t,s)/max(np.interp(3,t,s),1e-9))\n    return f,{\'t\':t.tolist(),\'speed\':s.tolist(),\'xy\':g[[\'x\',\'y\']].to_numpy(float).tolist()}\n\ndef prepare(data,out):\n    data,out=Path(data),Path(out);out.mkdir(parents=True,exist_ok=True)\n    players=pd.read_csv(data/\'players.csv\');ids=set(players.loc[players.nfl_position.eq(\'WR\'),\'nfl_id\'])\n    tr=pd.read_csv(data/\'combine_tracking.csv\');tr=tr[tr.nfl_id.isin(ids)&tr.entity_type.eq(\'PLAYER\')&tr.drill_name.eq(\'FORTY_YARD_DASH\')]\n    attempts=[];rejected={};example=None\n    for key,g in tr.groupby([\'nfl_id\',\'draft_year\',\'event_id\'],sort=True):\n        try:f,diag=sprint_features(g)\n        except ValueError as e:rejected[str(e)]=rejected.get(str(e),0)+1;continue\n        attempts.append(dict(nfl_id=key[0],draft_year=key[1],event_id=key[2],**f))\n        if example is None:example=diag\n    if not attempts:raise ValueError(\'No forty-yard-dash tracking; inspect official drill identifiers\')\n    attempts=pd.DataFrame(attempts);motion=[c for c in attempts if c.startswith(\'motion__\')]\n    panel=attempts.groupby([\'nfl_id\',\'draft_year\'])[motion].median().reset_index()\n    pp=pd.read_csv(data/\'player_play.csv\').merge(players[[\'nfl_id\',\'draft_year\',\'nfl_position\']],on=\'nfl_id\',validate=\'many_to_one\')\n    games=pd.read_csv(data/\'games.csv\');pp=pp.merge(games,on=\'game_id\',validate=\'many_to_one\')\n    pp=pp[pp.nfl_position.eq(\'WR\')&pp.season.eq(pp.draft_year)&pp.season_type.eq(\'REG\')&pp.play_nullified_by_penalty.eq(\'N\')&pp.route_ran.notna()&pp.route_ran.ne(\'NONE\')&pp.nfl_id.isin(panel.nfl_id)].copy()\n    pp[\'defense_team\']=np.where(pp.team_abbr.eq(pp.home_team_abbr),pp.visitor_team_abbr,pp.home_team_abbr)\n    keys=pp[[\'game_id\',\'play_id\',\'nfl_id\']];rows=[];reject={};eligible=len(pp)\n    for year in (2023,2024,2025):\n        selected=keys[keys.game_id.isin(pp.loc[pp.season.eq(year),\'game_id\'])];parts=[]\n        for chunk in pd.read_csv(data/f\'game_tracking_{year}.csv\',chunksize=200000):\n            q=chunk[chunk.nfl_id.isin(panel.nfl_id)].merge(selected,on=[\'game_id\',\'play_id\',\'nfl_id\'],validate=\'many_to_one\')\n            if len(q):parts.append(q)\n        frames=pd.concat(parts,ignore_index=True)\n        for key,g in frames.groupby([\'game_id\',\'play_id\',\'nfl_id\'],sort=True):\n            g=g.sort_values(\'time\');snap=g[g.event.eq(\'ball_snap\')];release=g[g.event.eq(\'pass_forward\')]\n            reason=None\n            if len(snap)!=1 or len(release)!=1:reason=\'missing_or_nonunique_event\'\n            else:\n                t=pd.to_datetime(g.time,utc=True,format=\'ISO8601\');start=pd.to_datetime(snap.time.iloc[0],utc=True);end=pd.to_datetime(release.time.iloc[0],utc=True)\n                if (end-start).total_seconds()<1.5-1e-5:reason=\'early_release\'\n                else:\n                    q=g[(t-start).dt.total_seconds().between(-1e-5,1.5+1e-5)].copy();qt=pd.to_datetime(q.time,utc=True,format=\'ISO8601\').astype(\'int64\').to_numpy()/1e9\n                    if len(q)!=16 or not np.allclose(np.diff(qt),.1,atol=.015):reason=\'noncontinuous_window\'\n            if reason:reject[reason]=reject.get(reason,0)+1;continue\n            s=q.s.to_numpy(float);xy=q[[\'x\',\'y\']].to_numpy(float)\n            if not np.isfinite(s).all() or not np.isfinite(xy).all():reject[\'nonfinite\']=reject.get(\'nonfinite\',0)+1;continue\n            early=float(s[3:6].mean());late=float(s[13:16].mean())\n            rows.append(dict(game_id=key[0],play_id=key[1],nfl_id=key[2],snap_speed=float(s[0]),early_speed=early,early_acceleration=float((early-s[:3].mean())/.3),early_distance=float(np.linalg.norm(np.diff(xy[:6],axis=0),axis=1).sum()),future_speed=late,speed_gain=late-early))\n        print(\'Extracted game windows\',year,len(rows),flush=True)\n    d=pp.merge(pd.DataFrame(rows),on=[\'game_id\',\'play_id\',\'nfl_id\'],validate=\'one_to_one\').merge(pd.read_csv(data/\'combine_results.csv\'),on=[\'nfl_id\',\'draft_year\'],validate=\'many_to_one\').merge(panel,on=[\'nfl_id\',\'draft_year\'],validate=\'many_to_one\')\n    counts=d.groupby(\'nfl_id\').size();d=d[d.nfl_id.isin(counts[counts>=20].index)].reset_index(drop=True)\n    d.to_pickle(out/\'private_rows.pkl\');attempts.to_pickle(out/\'private_attempts.pkl\');(out/\'private_source_example.json\').write_text(json.dumps(example))\n    audit={\'status\':\'reconstructed_actual_data\',\'attempts\':len(attempts),\'source_players\':int(panel.nfl_id.nunique()),\'source_features\':motion,\'rejected_source\':rejected,\'eligible_routes\':eligible,\'rejected_game\':reject,\'players_by_year\':{str(k):int(v) for k,v in d.groupby(\'draft_year\').nfl_id.nunique().items()},\'routes_by_year\':{str(k):int(v) for k,v in d.groupby(\'draft_year\').size().items()},\'time_scope\':\'input <=0.5 s; outcome mean speed 1.3–1.5 s; release >=1.5 s; rookie REG routes; minimum 20 windows/player\',\'limitation\':\'Charted context and vendor speed may not be causally available online; eligibility conditions on future release.\'}\n    (out/\'population.json\').write_text(json.dumps(audit,indent=2));return d,audit\n\nclass Encoder(BaseEstimator,TransformerMixin):\n    def __init__(self,family=\'conventional\',prefix=True,k=4,seed=A.SEED):self.family=family;self.prefix=prefix;self.k=k;self.seed=seed\n    def fit(self,x,y):\n        p=x.drop_duplicates(\'nfl_id\');self.fit_ids_=set(p.nfl_id);self.numcols=R.NUM+(PREFIX if self.prefix else [])\n        def numeric():return Pipeline([(\'impute\',SimpleImputer(strategy=\'median\',add_indicator=True,keep_empty_features=True)),(\'scale\',StandardScaler())])\n        self.phys=numeric().fit(p[R.PHYS]);self.num=numeric().fit(x[self.numcols]);self.cat=OneHotEncoder(handle_unknown=\'ignore\',sparse_output=False).fit(x[R.CAT].astype(str).where(x[R.CAT].notna(),\'MISSING\'))\n        if self.family!=\'conventional\':\n            self.cols=[c for c in x if c.startswith(\'motion__\')];v=p[self.cols].to_numpy(float);self.lo=np.nanquantile(v,.01,axis=0);self.hi=np.nanquantile(v,.99,axis=0)\n            self.motion=numeric().fit(pd.DataFrame(np.clip(v,self.lo,self.hi)));z=self.motion.transform(pd.DataFrame(np.clip(v,self.lo,self.hi)))\n            self.innovation=Ridge(alpha=10).fit(self.phys.transform(p[R.PHYS]),z);z-=self.innovation.predict(self.phys.transform(p[R.PHYS]));self.lift=StandardScaler().fit(np.c_[z,z*z]);z=self.lift.transform(np.c_[z,z*z])\n            if self.family==\'pca_motion\':self.reducer=PCA(n_components=min(self.k,len(p)-1,z.shape[1]),svd_solver=\'full\').fit(z)\n            else:\n                oof=np.zeros(len(x))\n                for a,b in player_splits(x,3,self.seed):\n                    # Nested train-only encoders prevent nuisance preprocessing leakage.\n                    enc=Encoder(\'conventional\',self.prefix,self.k,self.seed).fit(x.iloc[a],np.asarray(y)[a]);reg=Ridge(alpha=10).fit(enc.transform(x.iloc[a]),np.asarray(y)[a],sample_weight=A.weights(x.iloc[a]));oof[b]=reg.predict(enc.transform(x.iloc[b]))\n                residual=pd.DataFrame({\'id\':x.nfl_id.to_numpy(),\'r\':np.asarray(y)-oof}).groupby(\'id\').r.mean().reindex(p.nfl_id).to_numpy()\n                self.reducer=PLSRegression(n_components=min(1 if self.k==4 else 2,len(p)-1,z.shape[1]),scale=False).fit(z,residual)\n        return self\n    def base(self,x):return np.c_[self.phys.transform(x[R.PHYS]),self.num.transform(x[self.numcols]),self.cat.transform(x[R.CAT].astype(str).where(x[R.CAT].notna(),\'MISSING\'))]\n    def transform(self,x):\n        b=self.base(x)\n        if self.family==\'conventional\':return b\n        z=self.motion.transform(pd.DataFrame(np.clip(x[self.cols].to_numpy(float),self.lo,self.hi)));z-=self.innovation.predict(self.phys.transform(x[R.PHYS]));z=self.lift.transform(np.c_[z,z*z]);return np.c_[b,self.reducer.transform(z)]\n\ndef fit_predict(tr,te,target,family,prefix,config,seed):\n    if set(tr.nfl_id)&set(te.nfl_id):raise ValueError(\'Player leakage\')\n    if config[\'reg\']==\'ridge\':reg=Ridge(alpha=config[\'alpha\'])\n    elif config[\'reg\']==\'forest\':reg=RandomForestRegressor(n_estimators=150,max_depth=6,min_samples_leaf=20,max_features=.8,n_jobs=2,random_state=A.SEED)\n    else:reg=ExtraTreesRegressor(n_estimators=150,max_depth=8,min_samples_leaf=20,max_features=.8,n_jobs=2,random_state=A.SEED)\n    model=Pipeline([(\'enc\',Encoder(family,prefix,config[\'k\'],seed)),(\'reg\',reg)]);model.fit(tr,tr[target],reg__sample_weight=A.weights(tr));return model.predict(te)\n\ndef score(d,p,target):return float(np.average((d[target].to_numpy()-p)**2,weights=A.weights(d)))\ndef interval(d,p0,p1,target):\n    v=pd.DataFrame({\'id\':d.nfl_id,\'delta\':(d[target].to_numpy()-p0)**2-(d[target].to_numpy()-p1)**2}).groupby(\'id\').delta.mean().to_numpy();rng=np.random.default_rng(A.SEED)\n    return {\'delta_mse\':float(v.mean()),\'ci95\':np.quantile(rng.choice(v,(5000,len(v)),replace=True).mean(axis=1),[.025,.975]).tolist(),\'players\':len(v),\'scope\':\'Conditional player bootstrap; not refitting uncertainty or a causal effect.\'}\n\ndef experiment(d,out,target=\'future_speed\',seed=A.SEED+510):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True);dev=d[d.draft_year.isin([2023,2024])].reset_index(drop=True);test=d[d.draft_year.eq(2025)].reset_index(drop=True)\n    specs=[(\'conventional_without_state\',\'conventional\',False),(\'conventional_with_state\',\'conventional\',True),(\'pca_with_state\',\'pca_motion\',True),(\'target_motion_with_state\',\'target_motion\',True),(\'pca_without_state\',\'pca_motion\',False),(\'target_motion_without_state\',\'target_motion\',False)]\n    nested={};forward={};records=[];validation=[]\n    for name,family,prefix in specs:\n        oof=np.zeros(len(dev))\n        for fold,(a,b) in enumerate(player_splits(dev,4,seed)):\n            tr,te=dev.iloc[a].reset_index(drop=True),dev.iloc[b];loss=[]\n            for ci,c in enumerate(CONFIGS):\n                scores=[]\n                for i,j in player_splits(tr,3,seed+1+fold):scores.append(score(tr.iloc[j],fit_predict(tr.iloc[i],tr.iloc[j],target,family,prefix,c,seed+1+fold),target))\n                loss.append(float(np.mean(scores)))\n            ci=int(np.argmin(loss));oof[b]=fit_predict(tr,te,target,family,prefix,CONFIGS[ci],seed+1+fold);records.append({\'family\':name,\'fold\':fold,\'inner_mse\':loss,\'config\':ci,\'train_player_hash\':hashlib.sha256(json.dumps(sorted(map(int,tr.nfl_id.unique()))).encode()).hexdigest(),\'test_player_hash\':hashlib.sha256(json.dumps(sorted(map(int,te.nfl_id.unique()))).encode()).hexdigest()})\n            print(target,name,\'outer\',fold,\'MSE\',score(te,oof[b],target),flush=True)\n        nested[name]=oof\n        tr=dev[dev.draft_year.eq(2023)];va=dev[dev.draft_year.eq(2024)];loss=[score(va,fit_predict(tr,va,target,family,prefix,c,seed),target) for c in CONFIGS];ci=int(np.argmin(loss));forward[name]=fit_predict(dev,test,target,family,prefix,CONFIGS[ci],seed);validation.append({\'family\':name,\'config\':ci,\'validation_mse\':loss})\n        # Persist complete per-family predictions before moving to another family.\n        pd.DataFrame({\'nfl_id\':dev.nfl_id,\'game_id\':dev.game_id,\'play_id\':dev.play_id,\'truth\':dev[target],\'prediction\':oof}).to_pickle(out/f\'private_nested_{name}.pkl\')\n        pd.DataFrame({\'nfl_id\':test.nfl_id,\'game_id\':test.game_id,\'play_id\':test.play_id,\'truth\':test[target],\'prediction\':forward[name]}).to_pickle(out/f\'private_forward_{name}.pkl\')\n    comparisons={}\n    for label,rows,preds in [(\'nested\',dev,nested),(\'exploratory_2025\',test,forward)]:\n        pairs=[(\'state_channel\',\'conventional_without_state\',\'conventional_with_state\'),(\'target_motion_increment\',\'conventional_with_state\',\'target_motion_with_state\'),(\'pca_increment\',\'conventional_with_state\',\'pca_with_state\'),(\'target_motion_scouting\',\'conventional_without_state\',\'target_motion_without_state\')]\n        comparisons[label]={\'mse\':{k:score(rows,p,target) for k,p in preds.items()},\'comparisons\':{k:interval(rows,preds[a],preds[b],target) for k,a,b in pairs},\'players\':int(rows.nfl_id.nunique()),\'rows\':len(rows)}\n    result={\'status\':\'completed_reconstructed_actual_data\',\'target\':target,\'outer_seed\':seed,\'nested\':comparisons[\'nested\'],\'exploratory_2025\':comparisons[\'exploratory_2025\'],\'folds\':records,\'forward_selection\':validation,\'selection\':\'Unique-player class-stratified 4 outer / 3 inner folds; matched 5 candidate models. 2023 training/2024 tuning then 2025 exploratory transport. All transformations train-only.\',\'prior_2025_inspection\':True,\'not_identical_to_lost_implementation\':True}\n    (out/\'results.json\').write_text(json.dumps(result,indent=2));return result\n\ndef run(data,out):\n    out=Path(out);d,audit=prepare(data,out)\n    return experiment(d,out/\'absolute_speed\')\n\nif __name__==\'__main__\':\n    import argparse\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',default=\'data\');p.add_argument(\'--out\',default=\'private_observability\');a=p.parse_args();run(a.data,a.out)\n', 'upstream/turn_transfer.py': '"""Demand-adjusted Combine-to-NFL WR turn translation.\n\nSource nuisance maps and physical innovations use training players only. NFL turn\ndemands are descriptive outcome-side conditioning, not real-time covariates.\n"""\nfrom pathlib import Path\nimport json,hashlib,time\nimport numpy as np,pandas as pd\nfrom scipy.signal import savgol_filter,find_peaks\nfrom sklearn.base import BaseEstimator,TransformerMixin\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import StandardScaler,OneHotEncoder\nfrom sklearn.linear_model import Ridge\nfrom sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor\nimport analysis as A,repo_predict as R\nfrom mechanism import player_splits\nfrom observability import score,interval\n\nMAP={\'IN\':[\'DAGGER_ROUTE_LEFT\'],\'OUT\':[\'SPEED_OUT_ROUTE_LEFT\'],\'SLANT\':[\'SLANT_ROUTE_LEFT\'],\'HITCH\':[\'CURL_ROUTE_RIGHT\',\'COMEBACK_ROUTE_RIGHT\']}\nSTYLE=[\'burden\',\'retention\',\'exit_ratio\',\'area\']\nDEMAND=[\'entry\',\'angle\',\'impulse\',\'source_count\']\nGAME=[\'game_entry\',\'game_angle\',\'game_impulse\',\'cut_time\']\nSEED=A.SEED+1200\n\ndef first_turn(g,game=False):\n    g=g.sort_values(\'time\').reset_index(drop=True);t=pd.to_datetime(g.time,utc=True,format=\'ISO8601\').astype(\'int64\').to_numpy()/1e9;t-=t[0]\n    if len(g)<12 or not np.allclose(np.diff(t),.1,atol=.015):raise ValueError(\'short_or_irregular\')\n    p=g[[\'x\',\'y\']].to_numpy(float);s=g.s.to_numpy(float)\n    if not np.isfinite(p).all() or not np.isfinite(s).all():raise ValueError(\'nonfinite\')\n    smooth=savgol_filter(p,7,3,axis=0);v=np.gradient(smooth,.1,axis=0);ps=np.linalg.norm(v,axis=1);th=np.unwrap(np.arctan2(v[:,1],v[:,0]))\n    kk=np.arange(5,len(g)-3);turn=np.abs(th[kk+3]-th[kk-3]);peaks=find_peaks(np.r_[0.,turn,0.],height=np.deg2rad(25),distance=10)[0]-1;peaks=peaks[(peaks>=0)&(peaks<len(kk))]\n    eligible=[]\n    for j in peaks:\n        c=int(kk[j]);entry=float(s[c-5:c-2].mean());angle=float(turn[j])\n        if entry<2.5 or angle>np.deg2rad(150) or min(ps[c-3],ps[c+3])<1:continue\n        if game and not .5-1e-5<=t[c]<=3.+1e-5:continue\n        eligible.append((c,entry,angle))\n    if not eligible:raise ValueError(\'no_qualifying_first_turn\')\n    c,entry,angle=eligible[0];stop=min(len(g)-1,c+15);local=s[c-3:c+4];imin=c-3+int(np.argmin(local));event=False;recovery=float(t[stop]-t[c])\n    if local.min()>=.9*entry:event=True;recovery=0.\n    else:\n        for i in range(max(c,imin),stop):\n            if s[i]>=.9*entry and s[i+1]>=.9*entry:event=True;recovery=float(t[i]-t[c]);break\n    record={\'entry\':entry,\'angle\':angle,\'impulse\':float(2*entry*np.sin(angle/2)),\'cut_time\':float(t[c]),\'recovery_time\':recovery,\'recovery_event\':event,\'complete_half_second\':c+5<len(g)}\n    if c+5<len(g):\n        deficit=np.maximum(0,1-s[c:c+6]/entry);q=p[c-3:c+6]-p[c-3];u=v[c-3]/max(ps[c-3],1e-9);q=q@np.array([[u[0],-u[1]],[u[1],u[0]]]);q[:,1]*=1 if th[c+3]>=th[c-3] else -1;arc=np.linalg.norm(np.diff(q,axis=0),axis=1).sum()\n        record.update(burden=float(np.trapezoid(deficit,dx=.1)),retention=float(local.min()/entry),exit_ratio=float(s[c+3:c+6].mean()/entry),area=float(.5*np.sum(q[:-1,0]*q[1:,1]-q[:-1,1]*q[1:,0])/max(arc**2,1e-9)))\n    else:record.update({k:np.nan for k in STYLE})\n    diagnostic={\'q\':(p[max(0,c-5):stop+1]-p[c]).tolist(),\'speed\':s[max(0,c-5):stop+1].tolist(),\'t\':(t[max(0,c-5):stop+1]-t[c]).tolist(),\'entry\':entry,\'cut\':c,\'angle\':angle}\n    return record,diagnostic\n\ndef recovery_bounds(d,horizon=.5):\n    recovered=d.recovery_event&d.recovery_time.le(horizon);unresolved=~d.recovery_event&d.recovery_time.lt(horizon)\n    return {\'rows\':len(d),\'observed_recovered_fraction\':float(recovered.mean()),\'recovered_fraction_lower\':float(recovered.mean()),\'recovered_fraction_upper\':float((recovered|unresolved).mean()),\'unresolved_before_horizon\':int(unresolved.sum()),\'right_censored_total\':int((~d.recovery_event).sum()),\'horizon_seconds\':horizon,\'scope\':\'Distribution-free bounds over observed records; no independent-censoring assumption. Not player balanced.\'}\n\ndef prepare(data,out):\n    data,out=Path(data),Path(out);out.mkdir(parents=True,exist_ok=True);p=pd.read_csv(data/\'players.csv\');phys=pd.read_csv(data/\'combine_results.csv\');ids=set(p.loc[p.nfl_position.eq(\'WR\'),\'nfl_id\'])\n    tr=pd.read_csv(data/\'combine_tracking.csv\');tr=tr[tr.nfl_id.isin(ids)&tr.entity_type.eq(\'PLAYER\')&tr.drill_type.eq(\'SKILL_DRILLS_WR\')].copy();tr[\'drill_name\']=tr.drill_name.replace({\'SPEED_OUT_LEFT\':\'SPEED_OUT_ROUTE_LEFT\'});drill_route={v:k for k,values in MAP.items() for v in values};source=[];source_reject={};source_example=None\n    for key,g in tr.groupby([\'nfl_id\',\'draft_year\',\'event_id\'],sort=True):\n        drill=g.drill_name.iloc[0]\n        if drill not in drill_route:continue\n        try:f,diag=first_turn(g)\n        except ValueError as e:source_reject[str(e)]=source_reject.get(str(e),0)+1;continue\n        if not f[\'complete_half_second\']:source_reject[\'short_response_window\']=source_reject.get(\'short_response_window\',0)+1;continue\n        source.append(dict(nfl_id=key[0],draft_year=key[1],event_id=key[2],drill_name=drill,route_ran=drill_route[drill],**f))\n        if source_example is None:source_example=diag\n    source=pd.DataFrame(source).merge(phys[[\'nfl_id\',\'draft_year\']+R.PHYS],on=[\'nfl_id\',\'draft_year\'],validate=\'many_to_one\');source.to_pickle(out/\'private_source.pkl\')\n    pp=pd.read_csv(data/\'player_play.csv\').merge(p[[\'nfl_id\',\'draft_year\',\'nfl_position\']],on=\'nfl_id\',validate=\'many_to_one\');games=pd.read_csv(data/\'games.csv\');pp=pp.merge(games,on=\'game_id\',validate=\'many_to_one\')\n    pp=pp[pp.nfl_position.eq(\'WR\')&pp.season.eq(pp.draft_year)&pp.season_type.eq(\'REG\')&pp.route_ran.isin(MAP)&pp.play_nullified_by_penalty.eq(\'N\')].copy();pp[\'defense_team\']=np.where(pp.team_abbr.eq(pp.home_team_abbr),pp.visitor_team_abbr,pp.home_team_abbr)\n    keys=pp[[\'game_id\',\'play_id\',\'nfl_id\']];rows=[];reject={};game_example=None\n    for year in [2023,2024,2025]:\n        selected=keys[keys.game_id.isin(pp.loc[pp.season.eq(year),\'game_id\'])];parts=[]\n        for chunk in pd.read_csv(data/f\'game_tracking_{year}.csv\',chunksize=200000):\n            q=chunk[chunk.nfl_id.isin(ids)].merge(selected,on=[\'game_id\',\'play_id\',\'nfl_id\'],validate=\'many_to_one\')\n            if len(q):parts.append(q)\n        frames=pd.concat(parts,ignore_index=True)\n        for key,g in frames.groupby([\'game_id\',\'play_id\',\'nfl_id\'],sort=True):\n            g=g.sort_values(\'time\');snap=g[g.event.eq(\'ball_snap\')];release=g[g.event.eq(\'pass_forward\')]\n            if len(snap)!=1 or len(release)!=1:reject[\'missing_or_nonunique_events\']=reject.get(\'missing_or_nonunique_events\',0)+1;continue\n            q=g[g.time.ge(snap.time.iloc[0])&g.time.le(release.time.iloc[0])]\n            try:f,diag=first_turn(q,game=True)\n            except ValueError as e:reject[str(e)]=reject.get(str(e),0)+1;continue\n            rows.append(dict(game_id=key[0],play_id=key[1],nfl_id=key[2],**f))\n            if game_example is None and f[\'complete_half_second\']:game_example=diag\n        print(\'Turn extraction\',year,len(rows),flush=True)\n    allrows=pp.merge(pd.DataFrame(rows),on=[\'game_id\',\'play_id\',\'nfl_id\'],validate=\'one_to_one\').merge(phys,on=[\'nfl_id\',\'draft_year\'],validate=\'many_to_one\');allrows.to_pickle(out/\'private_all_turns.pkl\')\n    d=allrows[allrows.complete_half_second].copy();counts=d.groupby(\'nfl_id\').size();d=d[d.nfl_id.isin(counts[counts>=10].index)].rename(columns={\'entry\':\'game_entry\',\'angle\':\'game_angle\',\'impulse\':\'game_impulse\'}).reset_index(drop=True);d.to_pickle(out/\'private_rows.pkl\')\n    audit={\'source_attempts\':len(source),\'source_players\':int(source.nfl_id.nunique()),\'source_rejections\':source_reject,\'eligible_route_records\':len(pp),\'game_rejections\':reject,\'all_qualifying_game_turns\':len(allrows),\'incomplete_half_second_turns\':int((~allrows.complete_half_second).sum()),\'primary_players_by_year\':{str(k):int(v) for k,v in d.groupby(\'draft_year\').nfl_id.nunique().items()},\'primary_turns_by_year\':{str(k):int(v) for k,v in d.groupby(\'draft_year\').size().items()},\'source_missing_player_route_fraction\':float(pd.MultiIndex.from_frame(d[[\'nfl_id\',\'route_ran\']]).isin(pd.MultiIndex.from_frame(source[[\'nfl_id\',\'route_ran\']])).mean()),\'recovery_bounds_all_turns\':recovery_bounds(allrows),\'measured_separation_fraction\':float(np.isfinite(d.separation_at_pass_forward).mean()),\'primary_target\':\'burden: observed half-second normalized speed deficit in seconds\',\'prior_2025_inspection\':True}\n    audit[\'source_available_player_route_fraction\']=audit.pop(\'source_missing_player_route_fraction\');(out/\'population.json\').write_text(json.dumps(audit,indent=2));(out/\'private_examples.json\').write_text(json.dumps({\'source\':source_example,\'game\':game_example}));return d,source,audit\n\ndef numeric():return Pipeline([(\'impute\',SimpleImputer(strategy=\'median\',add_indicator=True,keep_empty_features=True)),(\'scale\',StandardScaler())])\ndef source_demand(x):\n    a=x.entry.to_numpy();b=x.angle.to_numpy();return np.c_[a,b,a*b,a*a,b*b]\n\nclass TurnEncoder(BaseEstimator,TransformerMixin):\n    def __init__(self,source,family=\'conventional\'):self.source=source;self.family=family\n    def fit(self,x,y=None):\n        players=x.drop_duplicates(\'nfl_id\');self.fit_ids_=set(players.nfl_id);self.phys=numeric().fit(players[R.PHYS]);self.num=numeric().fit(x[R.NUM+GAME]);self.cat=OneHotEncoder(handle_unknown=\'ignore\',sparse_output=False).fit(x[R.CAT].astype(str).where(x[R.CAT].notna(),\'MISSING\'))\n        if self.family!=\'conventional\':\n            train_source=self.source[self.source.nfl_id.isin(self.fit_ids_)];self.source_fit_ids_=set(train_source.nfl_id)\n            self.demand_scale=StandardScaler().fit(source_demand(train_source));self.source_model=Ridge(alpha=10).fit(self.demand_scale.transform(source_demand(train_source)),train_source[STYLE],sample_weight=A.weights(train_source))\n            allsource=self.source.copy();prediction=self.source_model.predict(self.demand_scale.transform(source_demand(allsource)))\n            for j,k in enumerate(STYLE):allsource[\'innovation_\'+k]=allsource[k]-prediction[:,j]\n            table=allsource.groupby([\'nfl_id\',\'route_ran\'])[[\'entry\',\'angle\',\'impulse\']+[\'innovation_\'+k for k in STYLE]].median();table[\'source_count\']=allsource.groupby([\'nfl_id\',\'route_ran\']).size();self.table=table\n            units=x.drop_duplicates([\'nfl_id\',\'route_ran\']);dm=self.lookup(units,DEMAND);self.source_demands=numeric().fit(dm)\n            if self.family==\'execution\':\n                z=self.lookup(units,[\'innovation_\'+k for k in STYLE]);self.style=numeric().fit(z);z=self.style.transform(z);self.physical_innovation=Ridge(alpha=10).fit(self.phys.transform(units[R.PHYS]),z,sample_weight=A.weights(units));z-=self.physical_innovation.predict(self.phys.transform(units[R.PHYS]));self.style_scale=StandardScaler().fit(z)\n        return self\n    def lookup(self,x,cols):return self.table.reindex(pd.MultiIndex.from_frame(x[[\'nfl_id\',\'route_ran\']]))[cols].to_numpy(float)\n    def transform(self,x):\n        physical=self.phys.transform(x[R.PHYS]);event=self.num.transform(x[R.NUM+GAME]);base=np.c_[physical,event,self.cat.transform(x[R.CAT].astype(str).where(x[R.CAT].notna(),\'MISSING\'))]\n        if self.family==\'conventional\':return base\n        base=np.c_[base,self.source_demands.transform(self.lookup(x,DEMAND))]\n        if self.family==\'source_demands\':return base\n        z=self.style.transform(self.lookup(x,[\'innovation_\'+k for k in STYLE]));z-=self.physical_innovation.predict(physical);z=self.style_scale.transform(z);entry=event[:,len(R.NUM)];angle=event[:,len(R.NUM)+1];mass=physical[:,R.PHYS.index(\'combine_weight\')]\n        return np.c_[base,z,z*z,z*entry[:,None],z*angle[:,None],z*mass[:,None]]\n\ndef fit_predict(tr,te,source,family,config,target):\n    if set(tr.nfl_id)&set(te.nfl_id):raise ValueError(\'player leakage\')\n    if config[\'reg\']==\'ridge\':reg=Ridge(alpha=config[\'alpha\'])\n    elif config[\'reg\']==\'forest\':reg=RandomForestRegressor(n_estimators=150,max_depth=6,min_samples_leaf=20,max_features=.8,n_jobs=2,random_state=A.SEED)\n    else:reg=ExtraTreesRegressor(n_estimators=150,max_depth=8,min_samples_leaf=20,max_features=.8,n_jobs=2,random_state=A.SEED)\n    model=Pipeline([(\'enc\',TurnEncoder(source,family)),(\'reg\',reg)]);model.fit(tr,tr[target],reg__sample_weight=A.weights(tr));return model.predict(te)\n\ndef experiment(d,source,out,target=\'burden\',seed=SEED,forward=True):\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    if target==\'separation_at_pass_forward\':d=d[np.isfinite(d[target])&d[target].ge(0)].copy()\n    counts=d.groupby(\'draft_year\').nfl_id.nunique()\n    if min(counts.reindex([2023,2024,2025],fill_value=0))<10:raise ValueError(\'Frozen minimum player count failed; do not relax after test.\')\n    dev=d[d.draft_year.isin([2023,2024])].reset_index(drop=True);test=d[d.draft_year.eq(2025)].reset_index(drop=True);families=[\'conventional\',\'source_demands\',\'execution\'];pred={k:np.zeros(len(dev)) for k in families};tp={};records=[];bank=[]\n    for fold,(a,b) in enumerate(player_splits(dev,4,seed)):\n        tr=dev.iloc[a].reset_index(drop=True);te=dev.iloc[b]\n        for family in families:\n            loss=[]\n            for config in R.CONFIGS:\n                values=[]\n                for i,j in player_splits(tr,3,seed+1+fold):values.append(score(tr.iloc[j],fit_predict(tr.iloc[i],tr.iloc[j],source,family,config,target),target))\n                loss.append(float(np.mean(values)))\n            ci=int(np.argmin(loss));pred[family][b]=fit_predict(tr,te,source,family,R.CONFIGS[ci],target);records.append({\'fold\':fold,\'family\':family,\'inner_mse\':loss,\'config\':ci,\'train_player_hash\':hashlib.sha256(json.dumps(sorted(map(int,tr.nfl_id.unique()))).encode()).hexdigest(),\'test_player_hash\':hashlib.sha256(json.dumps(sorted(map(int,te.nfl_id.unique()))).encode()).hexdigest()});print(target,seed,fold,family,score(te,pred[family][b],target),flush=True)\n    nested={\'mse\':{k:score(dev,p,target) for k,p in pred.items()},\'execution_increment\':interval(dev,pred[\'source_demands\'],pred[\'execution\'],target),\'tracking_increment\':interval(dev,pred[\'conventional\'],pred[\'execution\'],target),\'players\':int(dev.nfl_id.nunique()),\'turns\':len(dev)}\n    for family in families:pd.DataFrame({\'nfl_id\':dev.nfl_id,\'truth\':dev[target],\'prediction\':pred[family]}).to_pickle(out/f\'private_nested_{family}.pkl\')\n    result={\'status\':\'completed_actual_data_exploratory\',\'target\':target,\'seed\':seed,\'nested\':nested,\'folds\':records,\'prior_2025_inspection\':True,\'primary_contrast\':\'execution versus source_demands; conventional is secondary\',\'scope\':\'Conditional mechanical transfer using measured NFL turn demands; not live deployment or causal training benefit.\'}\n    if forward:\n        tr=dev[dev.draft_year.eq(2023)];va=dev[dev.draft_year.eq(2024)]\n        for family in families:\n            loss=[score(va,fit_predict(tr,va,source,family,c,target),target) for c in R.CONFIGS];ci=int(np.argmin(loss));tp[family]=fit_predict(dev,test,source,family,R.CONFIGS[ci],target);bank.append({\'family\':family,\'validation_mse\':loss,\'config\':ci});pd.DataFrame({\'nfl_id\':test.nfl_id,\'game_id\':test.game_id,\'play_id\':test.play_id,\'truth\':test[target],\'prediction\':tp[family]}).to_pickle(out/f\'private_forward_{family}.pkl\')\n        result[\'exploratory_2025\']={\'mse\':{k:score(test,p,target) for k,p in tp.items()},\'execution_increment\':interval(test,tp[\'source_demands\'],tp[\'execution\'],target),\'tracking_increment\':interval(test,tp[\'conventional\'],tp[\'execution\'],target),\'players\':int(test.nfl_id.nunique()),\'turns\':len(test)};result[\'forward_selection\']=bank\n    (out/\'results.json\').write_text(json.dumps(result,indent=2));return result\n\ndef run(data,out):\n    out=Path(out);d,source,audit=prepare(data,out)\n    try:\n        primary=experiment(d,source,out/\'burden\');secondary=experiment(d,source,out/\'separation\',target=\'separation_at_pass_forward\')\n    except ValueError as e:\n        (out/\'feasibility_failure.json\').write_text(json.dumps({\'status\':\'frozen_protocol_feasibility_failure\',\'reason\':str(e),\'population\':audit},indent=2));raise\n    return primary,secondary\nif __name__==\'__main__\':\n    import argparse\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',default=\'data\');p.add_argument(\'--out\',default=\'private_turn_transfer\');a=p.parse_args();run(a.data,a.out)\n', 'upstream/front_transfer.py': '"""Defensive-front scouting transfer: drill geometry beyond timed/body baselines."""\nfrom pathlib import Path\nimport json,hashlib\nimport numpy as np,pandas as pd\nfrom scipy.signal import savgol_filter\nfrom sklearn.linear_model import Ridge\nfrom sklearn.decomposition import PCA\nfrom sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor\nimport analysis as A,repo_predict as R,observability as O,turn_transfer as T\nfrom mechanism import player_splits\n\nPOSITIONS=[\'DE\',\'DT\',\'NT\',\'OLB\']\nDRILLS=[\'PASS_RUSH_DRILL\',\'RUN_THE_HOOP_DRILL\',\'FOUR_BAG_AGILITY_DRILL\',\'FRONT_START_WAVE_DRILL_AND_LATERAL_REACTION\']\nNUM=[\'down\',\'yards_to_go\',\'yardline_number\',\'pre_snap_home_score\',\'pre_snap_visitor_score\',\'quarter\',\'snap_depth\',\'snap_width\',\'snap_speed\']\nCAT=[\'nfl_position\',\'lined_up_position\',\'offense_formation\',\'blitzing\',\'team_abbr\',\'possession_team\']\nSEED=A.SEED+3300\n\ndef skill_features(g,onset_speed=.5):\n    g=g.sort_values(\'time\');t=pd.to_datetime(g.time,utc=True,format=\'ISO8601\').astype(\'int64\').to_numpy()/1e9;t-=t[0];s=g.s.to_numpy(float)\n    if len(g)<16 or not np.allclose(np.diff(t),.1,atol=.015) or not np.isfinite(s).all():raise ValueError(\'short_irregular_nonfinite\')\n    # Fixed sustained motion onset, not a presumed official start cue.\n    on=next((i for i in range(len(s)-2) if np.all(s[i:i+3]>onset_speed)),None)\n    if on is None or len(s)-on<16:raise ValueError(\'no_complete_active_window\')\n    stop=min(len(s),on+51);p=g[[\'x\',\'y\']].to_numpy(float)[on:stop]\n    if not np.isfinite(p).all():raise ValueError(\'nonfinite_xy\')\n    p=savgol_filter(p,7,3,axis=0);v=np.gradient(p,.1,axis=0);sp=s[on:stop];a=np.gradient(sp,.1);length=np.linalg.norm(np.diff(p,axis=0),axis=1);arc=np.r_[0,np.cumsum(length)];theta=np.unwrap(np.arctan2(v[:,1],v[:,0]));w=np.gradient(theta,.1);valid=np.linalg.norm(v,axis=1)>1\n    first=min(16,len(sp));z=sp[:first];at=a[:first];mid=(z[:-1]+z[1:])/2;da=np.diff(at);ma=(at[:-1]+at[1:])/2;ds=np.diff(z)\n    f={\'speed_05\':float(sp[5]),\'speed_10\':float(sp[10]),\'speed_15\':float(sp[15]),\'accel_early\':float(np.mean(a[1:6])),\'peak_speed\':float(np.quantile(sp,.95)),\'braking\':float(np.trapezoid(np.maximum(-a,0),dx=.1)),\'turn_work\':float(np.sum(np.abs(w[valid])*sp[valid])*.1),\'bend_energy\':float(np.sum(w[valid]**2/np.maximum(sp[valid],1))*.1),\'straightness\':float(np.linalg.norm(p[-1]-p[0])/max(arc[-1],1e-9)),\'speed_accel_area\':float(np.sum(mid*da-ma*ds)/2),\'jerk_rms\':float(np.sqrt(np.mean(np.gradient(a,.1)**2)))}\n    for dist in [1,3,5]:f[\'time_\'+str(dist)+\'yd\']=float(np.interp(dist,arc,np.arange(len(arc))*.1)) if arc[-1]>=dist else np.nan\n    return f\n\ndef prepare(data,out):\n    data,out=Path(data),Path(out);out.mkdir(exist_ok=True);p=pd.read_csv(data/\'players.csv\');phys=pd.read_csv(data/\'combine_results.csv\');ids=set(p.loc[p.nfl_position.isin(POSITIONS),\'nfl_id\']);tr=pd.read_csv(data/\'combine_tracking.csv\');tr=tr[tr.nfl_id.isin(ids)&tr.entity_type.eq(\'PLAYER\')];sprints=[];skills=[];reject={}\n    for key,g in tr.groupby([\'nfl_id\',\'draft_year\',\'event_id\'],sort=True):\n        drill=g.drill_name.iloc[0]\n        try:\n            if drill==\'FORTY_YARD_DASH\':f,_=O.sprint_features(g);sprints.append(dict(nfl_id=key[0],draft_year=key[1],event_id=key[2],**f))\n            elif drill in DRILLS:f=skill_features(g);skills.append(dict(nfl_id=key[0],draft_year=key[1],event_id=key[2],drill_name=drill,**f))\n        except ValueError as e:reject[str(e)]=reject.get(str(e),0)+1\n    sp=pd.DataFrame(sprints);sk=pd.DataFrame(skills);sprintcols=[c for c in sp if c.startswith(\'motion__\')];panel=sp.groupby([\'nfl_id\',\'draft_year\'])[sprintcols].median()\n    values=[c for c in sk if c not in [\'nfl_id\',\'draft_year\',\'event_id\',\'drill_name\']];wide=sk.groupby([\'nfl_id\',\'draft_year\',\'drill_name\'])[values].median().unstack(\'drill_name\');wide.columns=[\'skill__\'+drill+\'__\'+name for name,drill in wide.columns];panel=wide.join(panel,how=\'outer\').reset_index();motioncols=sprintcols+list(wide.columns)\n    d=pd.read_csv(data/\'player_play.csv\').merge(p[[\'nfl_id\',\'draft_year\',\'nfl_position\']],on=\'nfl_id\',validate=\'many_to_one\').merge(pd.read_csv(data/\'games.csv\'),on=\'game_id\',validate=\'many_to_one\').merge(phys,on=[\'nfl_id\',\'draft_year\'],validate=\'many_to_one\').merge(panel,on=[\'nfl_id\',\'draft_year\'],validate=\'many_to_one\')\n    d=d[d.nfl_position.isin(POSITIONS)&d.season.eq(d.draft_year)&d.season_type.eq(\'REG\')&d.play_nullified_by_penalty.eq(\'N\')&np.isfinite(d.player_get_off)&d.nfl_id.isin(panel.nfl_id)].copy();counts=d.groupby(\'nfl_id\').size();d=d[d.nfl_id.isin(counts[counts>=30].index)].reset_index(drop=True)\n    snaps=[]\n    for year in [2023,2024,2025]:\n        keys=d.loc[d.draft_year.eq(year),[\'game_id\',\'play_id\',\'nfl_id\']]\n        for chunk in pd.read_csv(data/f\'game_tracking_{year}.csv\',chunksize=250000):\n            q=chunk[chunk.event.eq(\'ball_snap\')].merge(keys,on=[\'game_id\',\'play_id\',\'nfl_id\'],validate=\'many_to_one\')\n            if len(q):snaps.append(q[[\'game_id\',\'play_id\',\'nfl_id\',\'x\',\'y\',\'s\']])\n        print(\'Front snap extraction\',year,flush=True)\n    snap=pd.concat(snaps);counts=snap.groupby([\'game_id\',\'play_id\',\'nfl_id\']).size();unique=counts[counts.eq(1)].index;snap=snap.set_index([\'game_id\',\'play_id\',\'nfl_id\']).loc[unique].reset_index();d=d.merge(snap,on=[\'game_id\',\'play_id\',\'nfl_id\'],validate=\'one_to_one\');yards=np.where(d.yardline_side.eq(d.possession_team),d.yardline_number,100-d.yardline_number);right=d.play_direction.eq(\'right\');los=np.where(right,10+yards,110-yards);d[\'snap_depth\']=(d.x-los)*np.where(right,1,-1);d[\'snap_width\']=np.abs(d.y-80/3);d[\'snap_speed\']=d.s;d[\'pressure_event\']=d.time_to_pressure.notna().astype(float)\n    # Reapply the same threshold after missing/nonunique snap exclusion.\n    counts=d.groupby(\'nfl_id\').size();d=d[d.nfl_id.isin(counts[counts>=30].index)].reset_index(drop=True);d.to_pickle(out/\'private_rows.pkl\');sp.to_pickle(out/\'private_sprints.pkl\');sk.to_pickle(out/\'private_skills.pkl\')\n    audit={\'positions\':POSITIONS,\'sprint_attempts\':len(sp),\'skill_attempts\':len(sk),\'source_rejections\':reject,\'source_features\':motioncols,\'players_by_year\':{str(k):int(v) for k,v in d.groupby(\'draft_year\').nfl_id.nunique().items()},\'rush_records_by_year\':{str(k):int(v) for k,v in d.groupby(\'draft_year\').size().items()},\'pressure_observed_fraction\':float(d.pressure_event.mean()),\'snap_depth_range\':d.snap_depth.quantile([0,.01,.5,.99,1]).to_dict(),\'eligibility\':\'Rookie REG, non-nullified, finite charted player_get_off, source profile, unique ball_snap and >=30 records/player\',\'target_scope\':\'Charted get-off and observed time_to_pressure indicator among charted get-off records. Not an unconditional pass-rush survival model.\'};(out/\'population.json\').write_text(json.dumps(audit,indent=2));return d,motioncols,audit\n\ndef fit_predict(tr,te,cols,config,target,availability=True,return_predictor=False):\n    if set(tr.nfl_id)&set(te.nfl_id):raise ValueError(\'player leakage\')\n    units=tr.drop_duplicates(\'nfl_id\');phys=T.numeric().fit(units[R.PHYS]);num=T.numeric().fit(tr[NUM]);cat=T.OneHotEncoder(handle_unknown=\'ignore\',sparse_output=False).fit(tr[CAT].fillna(\'MISSING\').astype(str))\n    sourcegroups=[[\'motion__\'],*[[f\'skill__{name}__\'] for name in DRILLS]]\n    def presence(x):return np.c_[[x[[c for c in x if any(c.startswith(prefix) for prefix in group)]].notna().any(axis=1).to_numpy(float) for group in sourcegroups]].T\n    def base(x):\n        b=np.c_[phys.transform(x[R.PHYS]),num.transform(x[NUM]),cat.transform(x[CAT].fillna(\'MISSING\').astype(str))]\n        return np.c_[b,presence(x)] if availability else b\n    x,y=base(tr),base(te)\n    if cols:\n        scaler=T.numeric().fit(units[cols]);z=scaler.transform(units[cols]);residual=Ridge(alpha=10).fit(phys.transform(units[R.PHYS]),z);z-=residual.predict(phys.transform(units[R.PHYS]));pca=PCA(n_components=min(config[\'k\'],len(units)-1,z.shape[1]),svd_solver=\'full\').fit(z)\n        def motion(d):\n            z=pca.transform(scaler.transform(d[cols])-residual.predict(phys.transform(d[R.PHYS])));mass=phys.transform(d[R.PHYS])[:,R.PHYS.index(\'combine_weight\')];return np.c_[z,z*z,z*mass[:,None]]\n        x=np.c_[x,motion(tr)];y=np.c_[y,motion(te)]\n    if config[\'reg\']==\'ridge\':reg=Ridge(alpha=config[\'alpha\'])\n    elif config[\'reg\']==\'forest\':reg=RandomForestRegressor(n_estimators=120,max_depth=6,min_samples_leaf=30,max_features=.8,n_jobs=2,random_state=SEED)\n    else:reg=ExtraTreesRegressor(n_estimators=120,max_depth=8,min_samples_leaf=30,max_features=.8,n_jobs=2,random_state=SEED)\n    reg.fit(x,tr[target],sample_weight=A.weights(tr))\n    def fitted_predict(d):\n        z=base(d)\n        if cols:z=np.c_[z,motion(d)]\n        p=reg.predict(z);return np.clip(p,0,1) if target==\'pressure_event\' else p\n    return fitted_predict if return_predictor else fitted_predict(te)\n\nCONFIGS=[{\'reg\':\'ridge\',\'alpha\':a,\'k\':k} for a,k in [(1,4),(10,4),(100,6)]]+[{\'reg\':\'forest\',\'alpha\':0,\'k\':4},{\'reg\':\'extra\',\'alpha\':0,\'k\':6}]\n\ndef selection_score(rows,p,target,objective):\n    if objective==\'play\':return O.score(rows,p,target)\n    q=pd.DataFrame({\'nfl_id\':rows.nfl_id,\'truth\':rows[target],\'prediction\':p}).groupby(\'nfl_id\').mean();return float(np.mean((q.truth-q.prediction)**2))\n\ndef experiment(d,cols,out,target,seed=SEED,forward=True,availability=True,objective=\'play\'):\n    out=Path(out)/target;out.mkdir(parents=True,exist_ok=True);dev=d[d.draft_year.isin([2023,2024])].reset_index(drop=True);test=d[d.draft_year.eq(2025)].reset_index(drop=True);families={\'conventional\':[],\'sprint\':[c for c in cols if c.startswith(\'motion__\')],\'skills\':[c for c in cols if c.startswith(\'skill__\')],\'combined\':cols};pred={k:np.zeros(len(dev)) for k in families};tp={};folds=[];selection=[]\n    for fold,(i,j) in enumerate(player_splits(dev,4,seed)):\n        tr=dev.iloc[i].reset_index(drop=True);te=dev.iloc[j]\n        for family,cc in families.items():\n            loss=[np.mean([selection_score(tr.iloc[b],fit_predict(tr.iloc[a],tr.iloc[b],cc,c,target,availability),target,objective) for a,b in player_splits(tr,3,seed+1+fold)]) for c in CONFIGS];ci=int(np.argmin(loss));pred[family][j]=fit_predict(tr,te,cc,CONFIGS[ci],target,availability);folds.append({\'fold\':fold,\'family\':family,\'config\':ci,\'inner_loss\':list(map(float,loss)),\'train_player_hash\':hashlib.sha256(json.dumps(sorted(map(int,tr.nfl_id.unique()))).encode()).hexdigest()});print(target,objective,fold,family,ci,flush=True)\n    r={\'target\':target,\'seed\':seed,\'selection_objective\':objective,\'folds\':folds,\'source_availability_in_every_baseline\':availability,\'conditional_player_weighting\':True,\'prior_global_2025_WR_DB_inspection\':True,\'DL_model_selection_2025_used\':False}\n    if forward:\n        tr=dev[dev.draft_year.eq(2023)];va=dev[dev.draft_year.eq(2024)]\n        for family,cc in families.items():\n            loss=[selection_score(va,fit_predict(tr,va,cc,c,target,availability),target,objective) for c in CONFIGS];ci=int(np.argmin(loss));tp[family]=fit_predict(dev,test,cc,CONFIGS[ci],target,availability);selection.append({\'family\':family,\'config\':ci,\'validation_loss\':loss})\n        r[\'selection\']=selection\n    for label,rows,pp in [(\'nested\',dev,pred)]+([(\'exploratory_2025\',test,tp)] if forward else []):\n        r[label]={\'mse\':{k:O.score(rows,p,target) for k,p in pp.items()},\'comparisons\':{k:O.interval(rows,pp[\'conventional\'],p,target) for k,p in pp.items() if k!=\'conventional\'},\'players\':int(rows.nfl_id.nunique()),\'rows\':len(rows)}\n        q=rows[[\'nfl_id\',\'game_id\',\'play_id\',target]].copy()\n        for k,v in pp.items():q[k]=v\n        q.to_pickle(out/f\'private_{label}.pkl\')\n    (out/\'results.json\').write_text(json.dumps(r,indent=2));return r\n\nif __name__==\'__main__\':\n    import argparse\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',default=\'data\');p.add_argument(\'--out\',default=\'private_front\');args=p.parse_args();d,cols,audit=prepare(args.data,args.out)\n    for target in [\'player_get_off\',\'pressure_event\']:experiment(d,cols,args.out,target)\n', 'python/perfectpower/__init__.py': 'from .core import (RigidCertificate, count, hit_indices, integer_power_root,\n                   rigid_certificate, verify_certificate, windows)\n', 'python/perfectpower/core.py': '"""Exact arithmetic for polynomial perfect-power hits. Coefficients are low-to-high."""\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom fractions import Fraction\nfrom math import gcd, isqrt, lcm\nfrom typing import Iterable\n\n\ndef normalize(coefficients: Iterable[int | Fraction]) -> tuple[Fraction, ...]:\n    a = [Fraction(c) for c in coefficients]\n    if not a:\n        raise ValueError(\'polynomial requires at least one coefficient\')\n    while len(a) > 1 and a[-1] == 0:\n        a.pop()\n    return tuple(a)\n\n\ndef evaluate(coefficients: Iterable[int | Fraction], n: int) -> Fraction:\n    out = Fraction(0)\n    for c in reversed(tuple(coefficients)):\n        out = out * n + c\n    return out\n\n\ndef mul(a: tuple[Fraction, ...], b: tuple[Fraction, ...]) -> tuple[Fraction, ...]:\n    out = [Fraction(0)] * (len(a) + len(b) - 1)\n    for i, x in enumerate(a):\n        for j, y in enumerate(b):\n            out[i + j] += x * y\n    return normalize(out)\n\n\ndef power(a: tuple[Fraction, ...], d: int) -> tuple[Fraction, ...]:\n    if d < 0:\n        raise ValueError(\'negative polynomial exponent\')\n    result = (Fraction(1),)\n    while d:\n        if d & 1:\n            result = mul(result, a)\n        a = mul(a, a)\n        d //= 2\n    return result\n\n\ndef subtract(a: tuple[Fraction, ...], b: tuple[Fraction, ...]) -> tuple[Fraction, ...]:\n    return normalize([(a[i] if i < len(a) else 0) - (b[i] if i < len(b) else 0)\n                      for i in range(max(len(a), len(b)))])\n\n\ndef floor_nth_root(n: int, d: int) -> int:\n    """Largest nonnegative r with r**d <= n, using integer arithmetic."""\n    if d < 2 or n < 0:\n        raise ValueError(\'requires d >= 2 and n >= 0\')\n    if n < 2:\n        return n\n    if d == 2:\n        return isqrt(n)\n    lo, hi = 0, 1 << ((n.bit_length() + d - 1) // d)\n    while lo + 1 < hi:\n        mid = (lo + hi) // 2\n        if mid ** d <= n:\n            lo = mid\n        else:\n            hi = mid\n    return lo if hi ** d > n else hi\n\n\ndef integer_power_root(v: int, d: int) -> int | None:\n    if d < 2:\n        raise ValueError(\'d >= 2 is required\')\n    if v < 0 and d % 2 == 0:\n        return None\n    sign = -1 if v < 0 else 1\n    r = floor_nth_root(abs(v), d)\n    return sign * r if r ** d == abs(v) else None\n\n\ndef hit_indices(coefficients: Iterable[int], d: int, k: int, end: int, start: int = 1):\n    if d < 2 or start < 1 or end < 0:\n        raise ValueError(\'requires d >= 2, start >= 1 and end >= 0\')\n    c = normalize(coefficients)\n    if any(x.denominator != 1 for x in c):\n        raise ValueError(\'S must be in Z[x]\')\n    ints = [int(x) for x in c]\n    for n in range(start, end + 1):\n        v = 0\n        for x in reversed(ints):\n            v = v * n + x\n        v += k\n        m = integer_power_root(v, d)\n        if m is not None:\n            yield (n, m)\n\n\ndef count(coefficients: Iterable[int], d: int, k: int, end: int) -> int:\n    return sum(1 for _ in hit_indices(coefficients, d, k, end))\n\n\ndef windows(coefficients: Iterable[int], d: int, k: int, end: int) -> list[dict]:\n    if end < 1:\n        return []\n    hits = set(n for n, _ in hit_indices(coefficients, d, k, end))\n    running = 0\n    maxima: dict[int, tuple[int, int]] = {}\n    for n in range(1, end + 1):\n        running += n in hits\n        j = n.bit_length() - 1\n        current = maxima.get(j)\n        if current is None or running * current[0] > current[1] * n:\n            maxima[j] = (n, running)\n    return [{\'lo\': 1 << j, \'hi\': min(end, (1 << (j + 1)) - 1),\n             \'maximum_at\': n, \'count_at_maximum\': a, \'max_ratio\': a / n}\n            for j, (n, a) in sorted(maxima.items())]\n\n\ndef leading_integer_root(a: int, d: int) -> int | None:\n    return integer_power_root(a, d)\n\n\n@dataclass(frozen=True)\nclass RigidCertificate:\n    d: int\n    coefficients: tuple[int, ...]  # F, low-to-high\n    root_numerators: tuple[int, ...]  # Q = sum root_numerators[i] x^i / denominator\n    denominator: int\n    remainder_numerators: tuple[int, ...]  # R=F-Q^d, divided by denominator**d\n    cutoff: int  # no integer-power hits n >= cutoff, unless exact_identity\n    exact_identity: bool\n\n    def to_json(self):\n        return asdict(self)\n\n\ndef rigid_certificate(coefficients: Iterable[int], d: int) -> RigidCertificate | None:\n    """Construct a rational truncation and an independently checkable finite-hit cutoff.\n\n    Returns None outside the rigid leading-degree branch. The cutoff can be very large.\n    """\n    if d < 2:\n        raise ValueError(\'d >= 2 is required\')\n    f = normalize(coefficients)\n    if any(c.denominator != 1 for c in f):\n        raise ValueError(\'F must be in Z[x]\')\n    if f == (0,):\n        return RigidCertificate(d, (0,), (0,), 1, (0,), 1, True)\n    M = len(f) - 1\n    if M % d:\n        return None\n    b = leading_integer_root(int(f[-1]), d)\n    if b is None:\n        return None\n    q = M // d\n    Q = [Fraction(0)] * (q + 1)\n    Q[q] = Fraction(b)\n    for j in range(1, q + 1):\n        coefficient = M - j\n        current = power(normalize(Q), d)\n        known = current[coefficient] if coefficient < len(current) else 0\n        Q[q - j] = (f[coefficient] - known) / (d * b ** (d - 1))\n    Q = normalize(Q)\n    R = subtract(f, power(Q, d))\n    D = lcm(*(c.denominator for c in Q))\n    qnum = tuple(int(c * D) for c in Q)\n    rnum = tuple(int(c * D ** d) for c in R)\n    if R == (0,):\n        if any(c.denominator != 1 for c in Q):\n            raise ArithmeticError(\'integral closure violated\')\n        return RigidCertificate(d, tuple(map(int, f)), qnum, D, rnum, 1, True)\n    r = len(R) - 1\n    if q == 0 or r >= (d - 1) * q:\n        raise ArithmeticError(\'truncation degree invariant failed\')\n    # At n >= B, |Q(n)| >= |b| n^q/2; |R(n)| <= C n^r.\n    L = sum(abs(c) for c in Q[:-1])\n    C = sum(abs(c) for c in R)\n    LR = sum(abs(c) for c in R[:-1])\n    B0 = max(1, (2 * L / abs(b)).__ceil__(),\n             (2 * LR / abs(R[-1])).__ceil__())\n    a = Fraction(abs(b), 2)\n    def sufficient(n: int) -> bool:\n        return (C * n ** r <= a ** d * n ** (d * q) / 2\n                and C * n ** r < a ** (d - 1) * n ** ((d - 1) * q) / (2 * D))\n    hi = B0\n    while not sufficient(hi):\n        hi *= 2\n    lo = B0\n    while lo < hi:\n        mid = (lo + hi) // 2\n        if sufficient(mid):\n            hi = mid\n        else:\n            lo = mid + 1\n    return RigidCertificate(d, tuple(map(int, f)), qnum, D, rnum, lo, False)\n\n\ndef verify_certificate(cert: RigidCertificate) -> bool:\n    """Rebuild Q,R and check all integer inequalities at the cutoff exactly."""\n    d, D, B = cert.d, cert.denominator, cert.cutoff\n    if d < 2 or D < 1 or B < 1:\n        return False\n    try:\n        F = normalize(cert.coefficients)\n        Q = normalize(Fraction(x, D) for x in cert.root_numerators)\n        R = subtract(F, power(Q, d))\n        if tuple(int(x * D ** d) for x in R) != tuple(cert.remainder_numerators):\n            return False\n        if R == (0,):\n            return cert.exact_identity and B == 1 and all(x.denominator == 1 for x in Q)\n        if cert.exact_identity or len(F) < 2 or len(Q) < 2:\n            return False\n        q, r = len(Q) - 1, len(R) - 1\n        b = Q[-1]\n        if b.denominator != 1 or len(F) - 1 != d * q or r >= (d - 1) * q:\n            return False\n        L, C = sum(abs(c) for c in Q[:-1]), sum(abs(c) for c in R)\n        LR = sum(abs(c) for c in R[:-1])\n        a = abs(b) / 2\n        return (B >= 1 and abs(b) * B >= 2 * L\n                and abs(R[-1]) * B >= 2 * LR\n                and C * B ** r <= a ** d * B ** (d * q) / 2\n                and C * B ** r < a ** (d - 1) * B ** ((d - 1) * q) / (2 * D))\n    except (ValueError, ZeroDivisionError, OverflowError):\n        return False\n', 'python/perfectpower/movement_language.py': '"""Finite weighted movement languages with exact populations and numerical inference.\n\nThis is a generic DAG/explicit-duration model, not a claim of athlete quantum\nphysics. Counts and ranks are exact; log-weight posteriors use binary64.\n"""\nfrom dataclasses import dataclass\nfrom fractions import Fraction\nfrom itertools import product\nfrom math import exp, log, factorial, isfinite, inf\nimport random\n\n@dataclass(frozen=True)\nclass Edge:\n    source: int\n    target: int\n    log_weight: float = 0.\n    reward: tuple = ()\n    label: object = None\n\n\ndef logsum(values):\n    values=list(values)\n    if not values:return -inf\n    maximum=max(values)\n    return maximum+log(sum(exp(v-maximum) for v in values)) if maximum!=-inf else -inf\n\n\nclass MovementLanguage:\n    """An acyclic, finite word channel; parallel edges are labeled alternatives."""\n    def __init__(self, nodes, edges, start, terminals, max_edges=200000):\n        if type(nodes) is not int or not 1<=nodes<=100000:raise ValueError(\'node budget\')\n        if type(max_edges) is not int or max_edges<0:raise ValueError(\'edge budget\')\n        self.nodes=nodes;self.edges=list(edges);self.start=start;self.terminals=frozenset(terminals)\n        if len(self.edges)>max_edges:raise ValueError(\'edge budget\')\n        if type(start) is not int or not 0<=start<nodes or any(type(v) is not int or not 0<=v<nodes for v in self.terminals):raise ValueError(\'invalid endpoint\')\n        self.dimension=len(self.edges[0].reward) if self.edges else 0\n        if self.dimension>128:raise ValueError(\'reward budget\')\n        self.out=[[] for _ in range(nodes)];self.inc=[[] for _ in range(nodes)]\n        for i,e in enumerate(self.edges):\n            if type(e.source) is not int or type(e.target) is not int or not 0<=e.source<e.target<nodes:raise ValueError(\'edges must advance topological order\')\n            if not isfinite(e.log_weight) or len(e.reward)!=self.dimension or any(not isfinite(x) for x in e.reward):raise ValueError(\'nonfinite or inconsistent reward\')\n            if e.source in self.terminals:raise ValueError(\'terminal must be absorbing\')\n            self.out[e.source].append(i);self.inc[e.target].append(i)\n        self.count=[0]*nodes;self.backward=[-inf]*nodes;self.best=[-inf]*nodes\n        for v in range(nodes-1,-1,-1):\n            if v in self.terminals:self.count[v]=1;self.backward[v]=self.best[v]=0.\n            else:\n                self.count[v]=sum(self.count[self.edges[i].target] for i in self.out[v])\n                self.backward[v]=logsum(self.edges[i].log_weight+self.backward[self.edges[i].target] for i in self.out[v])\n                self.best[v]=max((self.edges[i].log_weight+self.best[self.edges[i].target] for i in self.out[v]),default=-inf)\n        self.forward=[-inf]*nodes;self.forward[start]=0.\n        for v in range(nodes):\n            if v!=start:self.forward[v]=logsum(self.forward[self.edges[i].source]+self.edges[i].log_weight for i in self.inc[v])\n        self.log_partition=self.backward[start]\n\n    @property\n    def population(self):return self.count[self.start]\n\n    def conditional_count(self,prefix=()):\n        node=self.start\n        for i in prefix:\n            if type(i) is not int or i not in self.out[node]:return 0\n            node=self.edges[i].target\n        return self.count[node]\n\n    def unrank(self,index):\n        if type(index) is not int or not 0<=index<self.population:raise ValueError(\'rank outside language\')\n        node=self.start;path=[]\n        while node not in self.terminals:\n            for i in self.out[node]:\n                target=self.edges[i].target\n                if index<self.count[target]:path.append(i);node=target;break\n                index-=self.count[target]\n            else:raise RuntimeError(\'inconsistent counts\')\n        return path\n\n    def rank(self,path):\n        node=self.start;rank=0\n        for i in path:\n            if type(i) is not int or i not in self.out[node]:raise ValueError(\'invalid path\')\n            for j in self.out[node]:\n                if j==i:break\n                rank+=self.count[self.edges[j].target]\n            node=self.edges[i].target\n        if node not in self.terminals:raise ValueError(\'incomplete path\')\n        return rank\n\n    def sample(self,seed=None,weighted=True):\n        if not self.population:raise ValueError(\'empty language\')\n        rng=random.Random(seed)\n        if not weighted:return self.unrank(rng.randrange(self.population))\n        node=self.start;path=[]\n        while node not in self.terminals:\n            bank=[i for i in self.out[node] if self.count[self.edges[i].target]]\n            probabilities=[exp(self.edges[i].log_weight+self.backward[self.edges[i].target]-self.backward[node]) for i in bank]\n            u=rng.random()*sum(probabilities);i=bank[-1]\n            for j,p in zip(bank,probabilities):\n                u-=p\n                if u<0:i=j;break\n            path.append(i);node=self.edges[i].target\n        return path\n\n    def map_path(self):\n        if not self.population:raise ValueError(\'empty language\')\n        node=self.start;path=[]\n        while node not in self.terminals:\n            i=max(self.out[node],key=lambda i:self.edges[i].log_weight+self.best[self.edges[i].target])\n            path.append(i);node=self.edges[i].target\n        return path\n\n    def edge_posterior(self):\n        if not self.population:return [0.]*len(self.edges)\n        return [exp(self.forward[e.source]+e.log_weight+self.backward[e.target]-self.log_partition) if self.forward[e.source]!=-inf and self.backward[e.target]!=-inf else 0. for e in self.edges]\n\n    def moments(self):\n        """Additive reward means, marginal variances and all-path support bounds."""\n        if not self.population:raise ValueError(\'empty language\')\n        d=self.dimension;mean=[[0.]*d for _ in range(self.nodes)];variances=[[0.]*d for _ in range(self.nodes)]\n        lower=[[inf]*d for _ in range(self.nodes)];upper=[[-inf]*d for _ in range(self.nodes)]\n        for v in range(self.nodes-1,-1,-1):\n            if v in self.terminals:lower[v]=upper[v]=[0.]*d;continue\n            branches=[]\n            for i in self.out[v]:\n                e=self.edges[i];t=e.target\n                if not self.count[t]:continue\n                p=exp(e.log_weight+self.backward[t]-self.backward[v]);branches.append((e,p))\n                for j,r in enumerate(e.reward):\n                    mean[v][j]+=p*(r+mean[t][j])\n                    lower[v][j]=min(lower[v][j],r+lower[t][j]);upper[v][j]=max(upper[v][j],r+upper[t][j])\n            # Conditional variance avoids subtracting nearly equal raw moments.\n            for e,p in branches:\n                for j,r in enumerate(e.reward):variances[v][j]+=p*(variances[e.target][j]+(r+mean[e.target][j]-mean[v][j])**2)\n        m=mean[self.start];variance=variances[self.start]\n        expected_log=sum(p*e.log_weight for p,e in zip(self.edge_posterior(),self.edges))\n        return {\'mean\':m,\'variance\':variance,\'support_min\':lower[self.start],\'support_max\':upper[self.start],\n                \'entropy_nats\':max(0.,self.log_partition-expected_log),\'map_probability\':exp(self.best[self.start]-self.log_partition)}\n\n    def exact_weighted(self,weights,rewards=None):\n        """Rational reference calculation; never rationalize floating likelihoods."""\n        if len(weights)!=len(self.edges):raise ValueError(\'weight length\')\n        if any(not isinstance(w,(int,Fraction)) or isinstance(w,bool) or w<0 for w in weights):raise ValueError(\'nonnegative rational weights required\')\n        rewards=[e.reward for e in self.edges] if rewards is None else rewards\n        if len(rewards)!=len(self.edges) or any(len(r)!=self.dimension for r in rewards) or any(not isinstance(x,(int,Fraction)) or isinstance(x,bool) for r in rewards for x in r):raise ValueError(\'rational rewards required\')\n        z=[Fraction(0)]*self.nodes;a=[[Fraction(0)]*self.dimension for _ in range(self.nodes)];b=[[Fraction(0)]*self.dimension for _ in range(self.nodes)]\n        for v in range(self.nodes-1,-1,-1):\n            if v in self.terminals:z[v]=Fraction(1)\n            else:\n                for i in self.out[v]:\n                    e=self.edges[i];t=e.target;w=weights[i];z[v]+=w*z[t]\n                    for j,r in enumerate(rewards[i]):a[v][j]+=w*(r*z[t]+a[t][j]);b[v][j]+=w*(r*r*z[t]+2*r*a[t][j]+b[t][j])\n        total=z[self.start]\n        if not total:return {\'partition\':total,\'mean\':None,\'variance\':None}\n        m=[x/total for x in a[self.start]]\n        return {\'partition\':total,\'mean\':m,\'variance\':[s/total-x*x for s,x in zip(b[self.start],m)]}\n\n\ndef segment_channel(costs,min_duration,max_duration,rewards=None,temperature=1.,boundary_windows=None,max_edges=200000):\n    """Compile all contiguous ordered phase segmentations into a DAG.\n\n    Every frame belongs to one nonempty phase. Window k restricts the end of\n    phase k (inclusive integers). Costs/rewards are per-frame observations.\n    """\n    costs=[tuple(row) for row in costs];n=len(costs)\n    if not 1<=n<=4096:raise ValueError(\'observation window requires 1..4096 frames\')\n    k=len(costs[0]);minimum=list(min_duration);maximum=list(max_duration)\n    if not 1<=k<=16 or len(minimum)!=k or len(maximum)!=k or any(type(a) is not int or type(b) is not int or not 1<=a<=b for a,b in zip(minimum,maximum)):raise ValueError(\'duration constraints\')\n    if not isfinite(temperature) or temperature<=0:raise ValueError(\'temperature must be positive\')\n    if any(len(c)!=k or any(not isfinite(x) for x in c) for c in costs):raise ValueError(\'invalid phase costs\')\n    windows=[(0,n)]*k if boundary_windows is None else boundary_windows\n    if len(windows)!=k or any(len(w)!=2 or any(type(x) is not int for x in w) or not 0<=w[0]<=w[1]<=n for w in windows):raise ValueError(\'boundary windows\')\n    if rewards is None:rewards=[[tuple(1 if j==phase else 0 for j in range(k)) for phase in range(k)] for _ in range(n)]\n    else:rewards=[[tuple(v) for v in row] for row in rewards]\n    d=len(rewards[0][0])\n    if len(rewards)!=n or any(len(row)!=k or any(len(v)!=d or any(not isfinite(x) for x in v) for v in row) for row in rewards):raise ValueError(\'invalid rewards\')\n    prefix=[[0.]*(n+1) for _ in range(k)];rp=[[[0.]*(n+1) for _ in range(d)] for _ in range(k)]\n    for phase in range(k):\n        for t in range(n):\n            prefix[phase][t+1]=prefix[phase][t]+costs[t][phase]\n            for j in range(d):rp[phase][j][t+1]=rp[phase][j][t]+rewards[t][phase][j]\n    edges=[]\n    for phase in range(k):\n        for start in range(n):\n            for end in range(start+minimum[phase],min(n,start+maximum[phase])+1):\n                if not windows[phase][0]<=end<=windows[phase][1]:continue\n                remain=n-end\n                if not sum(minimum[phase+1:])<=remain<=sum(maximum[phase+1:]):continue\n                if phase==0 and start!=0:continue\n                r=tuple(rp[phase][j][end]-rp[phase][j][start] for j in range(d))\n                edges.append(Edge(phase*(n+1)+start,(phase+1)*(n+1)+end,-(prefix[phase][end]-prefix[phase][start])/temperature,r,(phase,start,end)))\n                if len(edges)>max_edges:raise ValueError(\'edge budget\')\n    channel=MovementLanguage((k+1)*(n+1),edges,0,[k*(n+1)+n],max_edges)\n    channel.frames=n;channel.phases=k\n    return channel\n\n\ndef phase_summary(channel):\n    if not channel.population:raise ValueError(\'empty phase language\')\n    n,k=channel.frames,channel.phases;boundary=[{} for _ in range(k)];occupancy=[[0.]*k for _ in range(n)]\n    for e,p in zip(channel.edges,channel.edge_posterior()):\n        phase,a,b=e.label;boundary[phase][b]=boundary[phase].get(b,0.)+p\n        for t in range(a,b):occupancy[t][phase]+=p\n    def quantile(bank,q):\n        total=sum(bank.values());c=0.\n        for value,p in sorted(bank.items()):\n            c+=p\n            if c>=q*total:return value\n        return max(bank)\n    return {\'count\':str(channel.population),\'log_partition\':channel.log_partition,\'boundary_quantiles\':[[quantile(b,q) for q in (.05,.5,.95)] for b in boundary],\n            \'boundary_probability\':boundary,\'occupancy\':occupancy,**channel.moments()}\n\n\ndef tensor_signature(points,degree=3):\n    """Chen signature of a piecewise-linear path, up to degree three.\n\n    Floating observations stay floating; order-dependent coordinates are not\n    physiological quantum phases. A straight segment uses tensor exponential.\n    """\n    points=[tuple(map(float,p)) for p in points]\n    if not points or degree not in (1,2,3):raise ValueError(\'points and degree 1..3 required\')\n    dim=len(points[0])\n    if not 1<=dim<=6 or any(len(p)!=dim or any(not isfinite(x) for x in p) for p in points):raise ValueError(\'point dimension or finiteness\')\n    words=[list(product(range(dim),repeat=k)) for k in range(degree+1)]\n    s=[{():1.}]+[{w:0. for w in words[k]} for k in range(1,degree+1)]\n    for a,b in zip(points,points[1:]):\n        delta=[y-x for x,y in zip(a,b)];e=[{():1.}]\n        for k in range(1,degree+1):\n            e.append({w:prod_float(delta[i] for i in w)/factorial(k) for w in words[k]})\n        s=[{w:sum(s[j][w[:j]]*e[k-j][w[j:]] for j in range(k+1)) for w in words[k]} for k in range(degree+1)]\n    return s\n\n\ndef prod_float(values):\n    p=1.\n    for v in values:p*=v\n    return p\n\n\ndef main():\n    import argparse,json\n    parser=argparse.ArgumentParser(description=\'Finite movement segmentation populations and weighted uncertainty\')\n    parser.add_argument(\'input\',help=\'JSON costs, min_duration, max_duration and optional rewards/temperature/boundary_windows\')\n    parser.add_argument(\'--rank\',type=int);parser.add_argument(\'--samples\',type=int,default=0);parser.add_argument(\'--seed\',type=int,default=0)\n    args=parser.parse_args()\n    if not 0<=args.samples<=1000:parser.error(\'samples must be in 0..1000\')\n    with open(args.input) as f:spec=json.load(f)\n    c=segment_channel(**spec)\n    result=phase_summary(c) if c.population else {\'count\':\'0\',\'status\':\'empty_language\'}\n    if args.rank is not None:result[\'ranked_segments\']=[c.edges[i].label for i in c.unrank(args.rank)]\n    result[\'samples\']=[[c.edges[i].label for i in c.sample(args.seed+j)] for j in range(args.samples)]\n    result[\'arithmetic_scope\']=\'Counts and ranks exact; likelihood-derived uncertainty uses floating arithmetic.\'\n    print(json.dumps(result))\n\nif __name__==\'__main__\':main()\n', 'python/perfectpower/movement_features.py': '"""Source-only movement features and uncertain explicit-duration phase summaries.\n\nNumpy is optional to the core language, required for this observation adapter.\nTimes are seconds and coordinates/speed yards and yards/second. No NFL outcomes\nare accepted. Phase templates are operational models, not labeled ground truth.\n"""\nimport math\nimport numpy as np\nfrom scipy.signal import savgol_filter\nfrom .movement_language import segment_channel,phase_summary,tensor_signature\n\n\ndef trajectory(time,xy,speed=None,smoothing=7):\n    time=np.asarray(time,float);xy=np.asarray(xy,float)\n    if time.ndim!=1 or xy.shape!=(len(time),2) or len(time)<12 or not np.isfinite(time).all() or not np.isfinite(xy).all():raise ValueError(\'finite trajectory with at least twelve frames required\')\n    dt=np.diff(time)\n    if (dt<=0).any() or not np.allclose(dt,.1,atol=.002,rtol=0):raise ValueError(\'contiguous 10 Hz frames required; split gaps explicitly\')\n    if smoothing not in (5,7,9) or smoothing>len(time):raise ValueError(\'smoothing window\')\n    p=savgol_filter(xy,smoothing,2,axis=0);v=np.gradient(p,time,axis=0);geometric=np.linalg.norm(v,axis=1)\n    if speed is None:s=geometric\n    else:\n        s=np.asarray(speed,float)\n        if s.shape!=time.shape or not np.isfinite(s).all() or (s<0).any():raise ValueError(\'finite nonnegative speed required\')\n        s=savgol_filter(s,smoothing,2);s=np.maximum(s,0)\n    a=np.gradient(s,time);theta=np.unwrap(np.arctan2(v[:,1],v[:,0]));w=np.gradient(theta,time)\n    # Heading is undefined at rest: interpolate from observed moving frames.\n    good=geometric>.5\n    if good.sum()<5:raise ValueError(\'insufficient observed motion\')\n    w=np.interp(time,time[good],w[good]);w=np.clip(w,-math.pi*3,math.pi*3)\n    return {\'time\':time,\'xy\':p,\'speed\':s,\'acceleration\':a,\'angular_rate\':w,\'geometric_speed\':geometric,\'heading_reliability\':float(good.mean())}\n\n\ndef attempt_features(time,xy,speed=None,onset=.5,temperature=1.,smoothing=7,window=61,degree=3):\n    z=trajectory(time,xy,speed,smoothing);s=z[\'speed\'];active=np.flatnonzero((s[:-2]>onset)&(s[1:-1]>onset)&(s[2:]>onset))\n    if not len(active):raise ValueError(\'no sustained onset\')\n    start=int(active[0]);end=min(len(s),start+window)\n    if end-start<16:raise ValueError(\'short active window\')\n    t=z[\'time\'][start:end];s=s[start:end];a=z[\'acceleration\'][start:end];w=np.abs(z[\'angular_rate\'][start:end]);p=z[\'xy\'][start:end];n=len(s)\n    # Fixed scale choices; no cohort statistics or outcomes enter extraction.\n    aa=np.clip(a/5,-3,3);ww=np.clip(w/math.pi,0,3)\n    centers=np.array([[.3,0],[-.7,.1],[0,.7],[.7,0]])\n    obs=np.c_[aa,ww];cost=((obs[:,None,:]-centers[None,:,:])**2).sum(axis=2)/2\n    dt=.1;reward=[]\n    for acceleration,turn in zip(a,w):\n        row=[]\n        for k in range(4):\n            v=[0.]*12;v[k]=dt;v[4+k]=acceleration*dt;v[8+k]=turn*dt;row.append(v)\n        reward.append(row)\n    c=segment_channel(cost,[2]*4,[max(2,n-6)]*4,reward,temperature=temperature)\n    summary=phase_summary(c);duration=np.array(summary[\'mean\'][:4]);accel=np.array(summary[\'mean\'][4:8])/duration;turn=np.array(summary[\'mean\'][8:12])/duration\n    points=np.c_[(t-t[0])/max(t[-1]-t[0],.1),s/10,aa,ww]\n    sig=tensor_signature(points,degree)\n    features={\'frames\':float(n),\'onset_seconds\':float(t[0]-z[\'time\'][0]),\'path_length\':float(np.linalg.norm(np.diff(p,axis=0),axis=1).sum()),\'peak_speed\':float(np.quantile(s,.95)),\n              \'speed_gain\':float(s[-1]-s[0]),\'mean_acceleration\':float(np.mean(a)),\'braking_impulse\':float(np.maximum(-a,0).sum()*dt),\'turn_work\':float((w*s).sum()*dt),\n              \'recovery_accel\':float(accel[3]),\'brake_accel\':float(accel[1]),\'redirect_turn\':float(turn[2]),\'phase_entropy_per_frame\':summary[\'entropy_nats\']/n,\n              \'map_probability\':summary[\'map_probability\'],\'heading_reliability\':z[\'heading_reliability\'],\'speed_disagreement\':float(np.mean(np.abs(s-z[\'geometric_speed\'][start:end])))}\n    for k in range(4):\n        features[f\'phase_duration_{k}\']=float(duration[k]);features[f\'phase_sd_{k}\']=math.sqrt(summary[\'variance\'][k]);features[f\'phase_boundary_width_{k}\']=(summary[\'boundary_quantiles\'][k][2]-summary[\'boundary_quantiles\'][k][0])*dt\n    if degree>=2:\n        for i in range(4):\n            for j in range(i+1,4):features[f\'order_area_{i}_{j}\']=(sig[2][(i,j)]-sig[2][(j,i)])/2\n    if degree>=3:\n        # Six triple-order coordinates; avoid a 64-feature small-cohort expansion.\n        import itertools\n        for word in itertools.permutations((1,2,3)):features[\'order_triple_\'+\'\'.join(map(str,word))]=sig[3][word]\n    return features,{\'phase\':summary,\'population\':str(c.population),\'edges\':len(c.edges),\'frames\':n,\'start\':start,\'end\':end,\'temperature\':temperature,\'onset\':onset,\'scales\':{\'speed\':10,\'acceleration\':5,\'turn\':math.pi},\'model_scope\':\'Operational phase templates, not calibrated physiological phase probabilities.\'}\n\n\ndef ensemble_features(time,xy,speed=None,onsets=(.5,1.,2.),temperature=1.,smoothing=7):\n    runs=[];audit=[]\n    for onset in onsets:\n        try:f,r=attempt_features(time,xy,speed,onset,temperature,smoothing);runs.append(f);audit.append({\'onset\':onset,\'population\':r[\'population\'],\'edges\':r[\'edges\'],\'frames\':r[\'frames\']})\n        except ValueError as error:audit.append({\'onset\':onset,\'rejected\':str(error)})\n    if not runs:raise ValueError(\'all source-only onset scenarios rejected\')\n    names=runs[0].keys();features={name:float(np.mean([r[name] for r in runs])) for name in names}\n    for name in (\'recovery_accel\',\'brake_accel\',\'redirect_turn\',\'order_area_1_2\',\'order_area_1_3\',\'order_area_2_3\'):\n        if name in features:features[\'onset_range_\'+name]=float(np.ptp([r[name] for r in runs]))\n    features[\'onset_scenarios\']=float(len(runs))\n    return features,{\'scenarios\':audit,\'weight_scope\':\'Equal sensitivity scenarios, not calibrated probability weights.\'}\n', 'python/perfectpower/movement_residual.py': '"""Player-separated residual ridge with an exact zero-correction option.\n\nOptional scientific dependencies: requirements-movement.txt. Never reads labels\nfrom evaluation players. Source-repeat weights are measurement heuristics, not\ncalibrated phase probabilities. Existing movement/PerfectJet engines unchanged.\n"""\nfrom dataclasses import dataclass\nimport numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import Ridge\n\nPHYS = [\'combine_height\',\'combine_weight\',\'hand_size\',\'arm_length\',\'wing_span\',\'ten_yd_split\',\'forty\',\'vertical\',\'broad_jump\',\'three_cone\',\'short_shuttle\',\'bench_reps\',\'ngs_athleticism_score\',\'ngs_college_production_score\',\'ngs_final_score\']\nSETS = {\'full15\': PHYS, \'core8\': [\'combine_height\',\'combine_weight\',\'ten_yd_split\',\'forty\',\'vertical\',\'broad_jump\',\'three_cone\',\'short_shuttle\'], \'motor4\': [\'ten_yd_split\',\'forty\',\'three_cone\',\'short_shuttle\']}\nALPHAS = (.1,1.,10.,100.,1000.,10000.,np.inf)\nCORRECTION_ALPHAS = (1.,10.,100.,1000.)\nSHRINKAGES = (.25,.5,1.)\nPREFIXES = {\'kinematic\': (\'path_length\',\'peak_speed\',\'speed_gain\',\'mean_acceleration\',\'braking_impulse\',\'turn_work\'), \'ordered\': (\'order_area_\',\'order_triple_\'), \'uncertain\': (\'recovery_accel\',\'brake_accel\',\'redirect_turn\',\'phase_duration_\',\'phase_sd_\',\'phase_boundary_width_\',\'phase_entropy_per_frame\',\'onset_range_\')}\nFAMILY_NAMES = (\'kinematic\',\'ordered\',\'uncertain\',\'combined\',\'reliability_combined\')\n\n\ndef validate(rows):\n    required = set(PHYS) | {\'nfl_id\',\'draft_year\',\'target\',\'combine_position\'}\n    if not required <= set(rows):\n        raise ValueError(\'missing player schema: \' + \',\'.join(sorted(required-set(rows))))\n    if rows.nfl_id.isna().any() or rows.draft_year.isna().any():\n        raise ValueError(\'missing player identity/class\')\n    if rows.groupby(\'nfl_id\').draft_year.nunique().max() != 1:\n        raise ValueError(\'player occurs in multiple classes\')\n    if not np.isfinite(rows.target).all() or not rows.target.between(0,1).all():\n        raise ValueError(\'target must be finite in [0,1]\')\n\n\ndef separated(train, test):\n    if set(train.nfl_id) & set(test.nfl_id):\n        raise ValueError(\'athlete leakage\')\n\n\ndef splits(rows, k=3, seed=714):\n    """Class-stratified identity splits; bootstrap duplicate rows stay together."""\n    groups = rows[[\'nfl_id\',\'draft_year\']].drop_duplicates()\n    if len(groups) < 4:\n        raise ValueError(\'at least four distinct training players required\')\n    k = min(k,len(groups)); buckets = [[] for _ in range(k)]\n    rng = np.random.default_rng(seed)\n    for _, g in groups.groupby(\'draft_year\',sort=True):\n        ids = g.nfl_id.to_numpy().copy(); rng.shuffle(ids)\n        # Rotate short class slices so every fold receives players when possible.\n        start = min(range(k),key=lambda j:len(buckets[j]))\n        for j, ident in enumerate(ids):buckets[(start+j)%k].append(ident)\n    for ids in buckets:\n        mask = rows.nfl_id.isin(ids).to_numpy()\n        if mask.any() and (~mask).any():yield np.flatnonzero(~mask),np.flatnonzero(mask)\n\n\nclass Numeric:\n    def fit(self, frame):\n        x = frame.to_numpy(dtype=float); x = np.where(np.isfinite(x),x,np.nan)\n        # All-missing training columns become constant zeros without warnings.\n        self.median = np.array([np.median(c[np.isfinite(c)]) if np.isfinite(c).any() else 0. for c in x.T])\n        x = np.where(np.isfinite(x),x,self.median)\n        self.mean=x.mean(axis=0); self.scale=x.std(axis=0); self.scale[self.scale<1e-12]=1\n        return self\n    def transform(self,frame):\n        x=frame.to_numpy(dtype=float)\n        return (np.where(np.isfinite(x),x,self.median)-self.mean)/self.scale\n\n\nclass ConventionalDesign:\n    def __init__(self, rows, name):\n        self.cols=SETS[name]+sorted(c for c in rows if c.startswith(\'available__\'))\n        self.numeric=Numeric().fit(rows[self.cols])\n        self.positions=sorted(rows.combine_position.fillna(\'MISSING\').astype(str).unique())\n    def transform(self,rows):\n        pos=rows.combine_position.fillna(\'MISSING\').astype(str).to_numpy()\n        return np.c_[self.numeric.transform(rows[self.cols]), np.column_stack([pos==p for p in self.positions])]\n\n\ndef ridge_predictions(x,y,xt,alpha):\n    if np.isinf(alpha):return np.full(len(xt),np.mean(y))\n    return Ridge(alpha=alpha).fit(x,y).predict(xt)\n\n\ndef families(rows):\n    cols=sorted(c for c in rows if c.startswith(\'movement__\'))\n    out={name:[c for c in cols if c.split(\'__\')[-1].startswith(prefix)] for name,prefix in PREFIXES.items()}\n    out[\'combined\']=sorted(set(sum(out.values(),[])));out[\'reliability_combined\']=out[\'combined\']\n    if any(not v for v in out.values()):raise ValueError(\'missing movement family\')\n    return out\n\n\nclass Baseline:\n    def __init__(self,rows,seed=714):\n        y=rows.target.to_numpy(); configs=[(s,a) for s in SETS for a in ALPHAS]\n        losses=np.zeros(len(configs)); folds=list(splits(rows,3,seed))\n        for i,j in folds:\n            for si,s in enumerate(SETS):\n                design=ConventionalDesign(rows.iloc[i],s)\n                x=design.transform(rows.iloc[i]);xt=design.transform(rows.iloc[j])\n                for ai,a in enumerate(ALPHAS):\n                    ci=si*len(ALPHAS)+ai\n                    losses[ci]+=np.square(y[j]-np.clip(ridge_predictions(x,y[i],xt,a),0,1)).sum()/len(rows)\n        ci=int(np.argmin(losses));self.name,self.alpha=configs[ci]\n        self.design=ConventionalDesign(rows,self.name)\n        self.mean=float(y.mean());self.model=None if np.isinf(self.alpha) else Ridge(alpha=self.alpha).fit(self.design.transform(rows),y)\n        self.audit={\'physical_set\':self.name,\'alpha\':\'mean\' if np.isinf(self.alpha) else self.alpha,\'inner_mse\':losses.tolist()}\n    def predict(self,rows):\n        p=np.full(len(rows),self.mean) if self.model is None else self.model.predict(self.design.transform(rows))\n        return np.clip(p,0,1)\n\n\ndef repeat_reliability(events, train_ids, columns):\n    """Diagonal repeat-noise estimate, pooled within player per drill.\n\n    Insufficient repeats or zero total variance yields zero reliability rather\n    than inventing a calibration. Outcomes never enter this calculation.\n    """\n    e=events[events.nfl_id.isin(train_ids)];weights=[];support={}\n    for col in columns:\n        _,drill,feature=col.split(\'__\',2)\n        d=e[e.drill_name.eq(drill)]\n        if feature not in d:weights.append(0.);continue\n        d=d[[\'nfl_id\',feature]].dropna();g=d.groupby(\'nfl_id\')[feature]\n        n=g.count();df=int((n-1).clip(lower=0).sum());total=float(d[feature].var(ddof=1)) if len(d)>1 else 0.\n        within=float(((d[feature]-g.transform(\'mean\'))**2).sum()/df) if df else 0.\n        r=float(np.clip(1-within/total,0,1)) if df>0 and total>1e-12 else 0.\n        weights.append(r);support[drill]={\'players\':int(len(n)),\'repeat_players\':int((n>1).sum()),\'within_degrees_freedom\':df}\n    return np.asarray(weights),support\n\n\nclass ResidualFit:\n    def __init__(self,rows,events,seed=714):\n        validate(rows); self.rows=rows;self.cols=families(rows);self.baseline=Baseline(rows,seed)\n        # OOF residual response: each nuisance baseline is itself tuned without\n        # the held player\'s response. Full-data in-sample residuals are not used.\n        oof=np.empty(len(rows))\n        for f,(i,j) in enumerate(splits(rows,3,seed+1)):\n            separated(rows.iloc[i],rows.iloc[j])\n            oof[j]=Baseline(rows.iloc[i],seed+20+f).predict(rows.iloc[j])\n        residual=rows.target.to_numpy()-oof\n        self.transforms={};self.models={};self.support={}\n        # Always residualize against the full conventional design for comparability.\n        design=ConventionalDesign(rows,\'full15\');self.full_design=design;b=design.transform(rows)\n        for family in FAMILY_NAMES:\n            cols=self.cols[family];norm=Numeric().fit(rows[cols]);z=norm.transform(rows[cols]);nuisance=Ridge(alpha=10).fit(b,z)\n            z=z-nuisance.predict(b).reshape(z.shape)\n            w=np.ones(len(cols))\n            if family==\'reliability_combined\':\n                w,support=repeat_reliability(events,set(rows.nfl_id),cols);self.support=support\n                w=np.sqrt(w)\n            z=z*w;self.transforms[family]=(norm,nuisance,w)\n            for a in CORRECTION_ALPHAS:self.models[family,a]=Ridge(alpha=a,fit_intercept=False).fit(z,residual)\n    def predictions(self,test):\n        separated(self.rows,test);base=self.baseline.predict(test);out={(\'zero\',0.,0.):base}\n        b=self.full_design.transform(test)\n        for family in FAMILY_NAMES:\n            norm,nuisance,w=self.transforms[family];z=norm.transform(test[self.cols[family]])\n            z=(z-nuisance.predict(b).reshape(z.shape))*w\n            for a in CORRECTION_ALPHAS:\n                correction=self.models[family,a].predict(z)\n                for lam in SHRINKAGES:out[family,a,lam]=np.clip(base+lam*correction,0,1)\n        return out\n\n\ndef choose(predictions,y, family=\'selected\'):\n    zero=(\'zero\',0.,0.); options=[zero]+[c for c in predictions if c!=zero and (family==\'selected\' or c[0]==family)]\n    losses={c:np.square(y-predictions[c]) for c in options}\n    best=min(options,key=lambda c:losses[c].mean())\n    improvement=losses[zero]-losses[best]\n    se=float(improvement.std(ddof=1)/np.sqrt(len(y))) if len(y)>1 else np.inf\n    picked=best if improvement.mean()>se else zero\n    return picked,{\'config\':list(picked),\'minimum_config\':list(best),\'paired_gain\':float(improvement.mean()),\'paired_standard_error\':se,\'zero_inner_mse\':float(losses[zero].mean()),\'minimum_inner_mse\':float(losses[best].mean())}\n\n\ndef fit_predict(train,test,events,seed=714):\n    separated(train,test);validate(train);validate(test)\n    train=train.reset_index(drop=True);y=train.target.to_numpy();inner={}\n    for f,(i,j) in enumerate(splits(train,3,seed)):\n        fit=ResidualFit(train.iloc[i],events,seed+100+f)\n        for config,p in fit.predictions(train.iloc[j]).items():\n            if config not in inner:inner[config]=np.empty(len(train))\n            inner[config][j]=p\n    fit=ResidualFit(train,events,seed+200);test_bank=fit.predictions(test)\n    pp={\'conventional\':test_bank[\'zero\',0.,0.]};audit={\'baseline\':fit.baseline.audit,\'selection\':{},\'repeat_support\':fit.support}\n    for family in FAMILY_NAMES+(\'selected\',):\n        config,selection=choose(inner,y,family);pp[family]=test_bank[config];audit[\'selection\'][family]=selection\n    return pp,audit\n', 'python/perfectpower/combine_compact.py': '"""Four-coordinate, bounded, source-only movement correction.\n\nRetrospective measurement study; predictions never depend on held-out labels.\n"""\nimport numpy as np\nfrom sklearn.linear_model import Ridge\nfrom .movement_residual import Baseline, ConventionalDesign, Numeric, splits, separated, validate\n\nCOLUMNS = (\n \'movement__PASS_RUSH_DRILL__speed_gain\',\n \'movement__PASS_RUSH_DRILL__recovery_accel\',\n \'movement__RUN_THE_HOOP_DRILL__redirect_turn\',\n \'movement__FOUR_BAG_AGILITY_DRILL__brake_accel\',\n)\n\nclass CompactFit:\n    def __init__(self, train, seed=714):\n        validate(train)\n        if not set(COLUMNS)<=set(train):raise ValueError(\'four declared movement coordinates required\')\n        self.ids=set(train.nfl_id);self.baseline=Baseline(train,seed)\n        response=np.empty(len(train))\n        for f,(i,j) in enumerate(splits(train,3,seed+1)):\n            response[j]=train.iloc[j].target.to_numpy()-Baseline(train.iloc[i],seed+20+f).predict(train.iloc[j])\n        self.design=ConventionalDesign(train,\'full15\');b=self.design.transform(train)\n        self.numeric=Numeric().fit(train[list(COLUMNS)]);z=self.numeric.transform(train[list(COLUMNS)])\n        self.nuisance=Ridge(alpha=10).fit(b,z)\n        z=np.clip(z-self.nuisance.predict(b).reshape(z.shape),-3,3)\n        self.correction=Ridge(alpha=10,fit_intercept=False).fit(z,response)\n        self.audit={\'baseline\':self.baseline.audit,\'coordinates\':list(COLUMNS),\'missing_fraction\':train[list(COLUMNS)].isna().mean().to_dict()}\n    def predict(self, test):\n        if self.ids & set(test.nfl_id):raise ValueError(\'athlete leakage\')\n        base=self.baseline.predict(test);z=self.numeric.transform(test[list(COLUMNS)])\n        z=np.clip(z-self.nuisance.predict(self.design.transform(test)).reshape(z.shape),-3,3)\n        change=np.clip(.5*self.correction.predict(z),-.02,.02)\n        return base,np.clip(base+change,0,1)\n\ndef fit_predict(train,test,seed=714):\n    separated(train,test);validate(train)\n    train=train.reset_index(drop=True);base_oof=np.empty(len(train));fixed_oof=np.empty(len(train))\n    for f,(i,j) in enumerate(splits(train,3,seed)):\n        base_oof[j],fixed_oof[j]=CompactFit(train.iloc[i],seed+100+f).predict(train.iloc[j])\n    gain=(train.target.to_numpy()-base_oof)**2-(train.target.to_numpy()-fixed_oof)**2\n    se=float(gain.std(ddof=1)/np.sqrt(len(gain)));apply=bool(gain.mean()>se)\n    fit=CompactFit(train,seed+200);base,fixed=fit.predict(test)\n    prior=float(train.target.mean());counts=train.groupby(\'combine_position\',dropna=False).target.agg([\'sum\',\'count\'])\n    role=[]\n    for position in test.combine_position:\n        c=counts.loc[position] if position in counts.index else None\n        role.append((float(c[\'sum\'])+20*prior)/(float(c[\'count\'])+20) if c is not None else prior)\n    pred={\'conventional\':base,\'global_mean\':np.full(len(test),prior),\'role_mean\':np.asarray(role),\'fixed\':fixed,\'selected\':fixed if apply else base.copy()}\n    audit={**fit.audit,\'apply_correction\':apply,\'inner_gain\':float(gain.mean()),\'inner_player_se\':se,\'maximum_absolute_fixed_change\':float(np.max(np.abs(fixed-base)))}\n    return pred,audit\n', 'scripts/run_movement_transfer.py': '#!/usr/bin/env python3\n"""Private-data adapter and honest player-held-out movement transfer experiment.\n\nWrites athlete-level data only to --private-out. Public aggregate receipt is\nseparate. Existing parallel experiments are read-only inputs, never overwritten.\n"""\nimport argparse,json,sys,time,hashlib\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parents[1];sys.path.insert(0,str(ROOT/\'python\'))\nimport numpy as np,pandas as pd\nfrom perfectpower.movement_features import ensemble_features\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import StandardScaler,OneHotEncoder\nfrom sklearn.linear_model import Ridge\nfrom sklearn.decomposition import PCA\nfrom sklearn.model_selection import KFold\n\nDRILLS=[\'PASS_RUSH_DRILL\',\'RUN_THE_HOOP_DRILL\',\'FOUR_BAG_AGILITY_DRILL\',\'FRONT_START_WAVE_DRILL_AND_LATERAL_REACTION\']\nPHYS=[\'combine_height\',\'combine_weight\',\'hand_size\',\'arm_length\',\'wing_span\',\'ten_yd_split\',\'forty\',\'vertical\',\'broad_jump\',\'three_cone\',\'short_shuttle\',\'bench_reps\',\'ngs_athleticism_score\',\'ngs_college_production_score\',\'ngs_final_score\']\nFAMILIES={\'kinematic\':[\'path_length\',\'peak_speed\',\'speed_gain\',\'mean_acceleration\',\'braking_impulse\',\'turn_work\'],\n          \'ordered\':[\'order_area_\',\'order_triple_\'],\n          \'uncertain\':[\'recovery_accel\',\'brake_accel\',\'redirect_turn\',\'phase_duration_\',\'phase_sd_\',\'phase_boundary_width_\',\'phase_entropy_per_frame\',\'onset_range_\']}\n\n\ndef source_metadata(rows):\n    """One audited Combine metadata row per player/class, without NFL fields.\n\n    Repeated rush rows may repeat source measurements or omit them. Distinct\n    observed values are conflicting evidence; row order must never resolve it.\n    """\n    keys=[\'nfl_id\',\'draft_year\'];required=set(keys+[\'combine_position\']+PHYS)\n    if not required<=set(rows):raise ValueError(\'source Combine metadata missing: \'+\',\'.join(sorted(required-set(rows))))\n    x=rows[keys+[\'combine_position\']+PHYS].copy()\n    if x[keys].isna().any().any():raise ValueError(\'missing source player/class identity\')\n    if x.groupby(\'nfl_id\').draft_year.nunique().gt(1).any():raise ValueError(\'source player occurs in multiple draft classes\')\n    position=x.combine_position\n    if position.isna().any() or not position.map(lambda v:isinstance(v,str) and bool(v.strip())).all():\n        raise ValueError(\'source combine_position required for every row\')\n    for col in PHYS:\n        try:x[col]=pd.to_numeric(x[col],errors=\'raise\')\n        except (TypeError,ValueError) as error:raise ValueError(\'invalid source conventional field \'+col) from error\n        if np.isinf(x[col].to_numpy(dtype=float)).any():raise ValueError(\'infinite source conventional field \'+col)\n    group=x.groupby(keys,sort=True)\n    for col in [\'combine_position\']+PHYS:\n        if group[col].nunique(dropna=True).gt(1).any():raise ValueError(\'inconsistent source Combine field \'+col)\n    return group[[\'combine_position\']+PHYS].first().reset_index()\n\n\ndef extract(data,cohort,dest):\n    started=time.perf_counter();dest.mkdir(parents=True,exist_ok=True);x=pd.read_pickle(cohort)\n    if not {\'nfl_id\',\'draft_year\',\'pressure_event\'}<=set(x):raise ValueError(\'cohort schema\')\n    metadata=source_metadata(x)\n    tracking=pd.read_csv(data/\'combine_tracking.csv\')\n    tracking=tracking.merge(metadata[[\'nfl_id\',\'draft_year\']],on=[\'nfl_id\',\'draft_year\'],how=\'inner\',validate=\'many_to_one\')\n    tracking=tracking[tracking.entity_type.eq(\'PLAYER\')&tracking.drill_name.isin(DRILLS)]\n    rows=[];reject={};attempts=0;populations=[]\n    for key,g in tracking.groupby([\'nfl_id\',\'draft_year\',\'event_id\'],sort=True):\n        attempts+=1;g=g.sort_values(\'time\');t=pd.to_datetime(g.time);seconds=(t-t.iloc[0]).dt.total_seconds().to_numpy()\n        try:f,a=ensemble_features(seconds,g[[\'x\',\'y\']].to_numpy(),g.s.to_numpy())\n        except ValueError as error:reject[str(error)]=reject.get(str(error),0)+1;continue\n        rows.append({\'nfl_id\':key[0],\'draft_year\':key[1],\'event_id\':key[2],\'drill_name\':g.drill_name.iloc[0],**f})\n        populations.extend(int(s[\'population\']) for s in a[\'scenarios\'] if \'population\' in s)\n        if len(rows)%50==0:print(json.dumps({\'attempts\':attempts,\'accepted\':len(rows),\'seconds\':time.perf_counter()-started}),flush=True)\n    if not rows:raise ValueError(\'no eligible attempts\')\n    events=pd.DataFrame(rows);events.to_pickle(dest/\'events.pkl\')\n    featurecols=[c for c in events if c not in (\'nfl_id\',\'draft_year\',\'event_id\',\'drill_name\')]\n    wide=events.groupby([\'nfl_id\',\'draft_year\',\'drill_name\'])[featurecols].median().unstack(\'drill_name\');wide.columns=[f\'movement__{drill}__{name}\' for name,drill in wide.columns]\n    unit=x.groupby([\'nfl_id\',\'draft_year\'],sort=True).pressure_event.agg([\'mean\',\'count\']).rename(columns={\'mean\':\'target\',\'count\':\'exposure\'}).reset_index()\n    unit=unit.merge(metadata,on=[\'nfl_id\',\'draft_year\'],validate=\'one_to_one\').merge(wide.reset_index(),on=[\'nfl_id\',\'draft_year\'],how=\'left\',validate=\'one_to_one\')\n    for drill in DRILLS:\n        cols=[c for c in unit if c.startswith(\'movement__\'+drill+\'__\')];unit[\'available__\'+drill]=unit[cols].notna().any(axis=1).astype(float) if cols else 0.\n    unit.to_pickle(dest/\'players.pkl\')\n    receipt={\'input_cohort_sha256\':hashlib.sha256(cohort.read_bytes()).hexdigest(),\'source_tracking_sha256\':hashlib.sha256((data/\'combine_tracking.csv\').read_bytes()).hexdigest(),\n        \'source_attempts\':attempts,\'accepted_attempts\':len(events),\'source_rejections\':reject,\'players\':len(unit),\'players_by_year\':{str(k):int(v) for k,v in unit.groupby(\'draft_year\').size().items()},\n        \'phase_populations_min\':str(min(populations)),\'phase_populations_max\':str(max(populations)),\'scenarios\':len(populations),\'extract_seconds\':time.perf_counter()-started,\n        \'target\':\'Mean observed pressure indicator in the supplied eligible rookie rush cohort; conditional, not unconditional career success.\',\n        \'position_source\':\'Audited source combine_position; NFL position is never a predictor\',\n        \'conventional_source_consistency_checked\':True,\n        \'onset_weights\':\'Equal source-only sensitivity scenarios .5,1,2 yd/s; not calibrated phase probabilities\',\'frame_scope\':\'Up to first 61 contiguous 10Hz active frames. Gaps rejected. Four operational ordered phase templates, not labeled physiology.\'}\n    (dest/\'extraction.json\').write_text(json.dumps(receipt,indent=2)+\'\\n\');return unit,receipt\n\n\ndef model_predict(train,test,cols,alpha,k):\n    if set(train.nfl_id)&set(test.nfl_id):raise ValueError(\'athlete leakage\')\n    for rows in (train,test):\n        if \'combine_position\' not in rows or rows.combine_position.isna().any():raise ValueError(\'source combine_position required\')\n    basecols=[c for c in PHYS if c in train]+[c for c in train if c.startswith(\'available__\')]\n    imp=SimpleImputer(strategy=\'median\',keep_empty_features=True);sc=StandardScaler();b=sc.fit_transform(imp.fit_transform(train[basecols]));bt=sc.transform(imp.transform(test[basecols]))\n    enc=OneHotEncoder(handle_unknown=\'ignore\',sparse_output=False);b=np.c_[b,enc.fit_transform(train[[\'combine_position\']])];bt=np.c_[bt,enc.transform(test[[\'combine_position\']])]\n    if cols and k:\n        mi=SimpleImputer(strategy=\'median\',keep_empty_features=True);ms=StandardScaler();m=ms.fit_transform(mi.fit_transform(train[cols]));mt=ms.transform(mi.transform(test[cols]))\n        # Residualize measured movement against conventional traits inside fold.\n        nuisance=Ridge(alpha=10).fit(b,m);m-=nuisance.predict(b).reshape(m.shape);mt-=nuisance.predict(bt).reshape(mt.shape)\n        pca=PCA(n_components=min(k,len(train)-1,len(cols)),svd_solver=\'full\');m=pca.fit_transform(m);mt=pca.transform(mt)\n        b=np.c_[b,m];bt=np.c_[bt,mt]\n    fitted=Ridge(alpha=alpha).fit(b,train.target)\n    return np.clip(fitted.predict(bt),0,1)\n\n\ndef bank():return [(a,k) for a in (1.,10.,100.) for k in (0,2,4)]\n\ndef select(train,cols,seed):\n    folds=list(KFold(n_splits=min(3,len(train)),shuffle=True,random_state=seed).split(train))\n    losses=[]\n    for a,k in bank():losses.append(float(np.mean([np.mean((train.iloc[j].target-model_predict(train.iloc[i],train.iloc[j],cols,a,k))**2) for i,j in folds])))\n    ix=int(np.argmin(losses));return bank()[ix],losses\n\n\ndef compare(truth,base,candidate,seed=717):\n    d=(truth-base)**2-(truth-candidate)**2;rng=np.random.default_rng(seed);b=rng.integers(0,len(d),size=(4000,len(d)));samples=d[b].mean(axis=1)\n    bm=float(np.mean((truth-base)**2));cm=float(np.mean((truth-candidate)**2))\n    return {\'baseline_mse\':bm,\'candidate_mse\':cm,\'relative_gain\':(bm-cm)/bm if bm else None,\'absolute_gain\':float(d.mean()),\'athlete_bootstrap_95\':np.quantile(samples,[.025,.975]).tolist(),\'interval_scope\':\'Conditional on retained fitted predictions; does not include refitting or multiplicity across adaptive prior experiments\',\'players\':len(d)}\n\n\ndef evaluate(unit,out):\n    if not set(PHYS)<=set(unit):raise ValueError(\'complete conventional baseline columns required\')\n    allcols=[c for c in unit if c.startswith(\'movement__\')];families={\'conventional\':[]}\n    for name,prefixes in FAMILIES.items():families[name]=[c for c in allcols if any(c.split(\'__\')[-1].startswith(p) for p in prefixes)]\n    families[\'combined\']=sorted(set(sum([families[k] for k in FAMILIES],[])))\n    dev=unit[unit.draft_year.isin([2023,2024])].reset_index(drop=True);test=unit[unit.draft_year.eq(2025)].reset_index(drop=True)\n    if len(dev)<12 or not len(test):raise ValueError(\'insufficient cohort for declared splits\')\n    predictions={name:np.zeros(len(dev)) for name in families};selection=[]\n    for fold,(i,j) in enumerate(KFold(n_splits=4,shuffle=True,random_state=714).split(dev)):\n        for name,cols in families.items():\n            config,loss=select(dev.iloc[i],cols,800+fold);predictions[name][j]=model_predict(dev.iloc[i],dev.iloc[j],cols,*config)\n            selection.append({\'outer_fold\':fold,\'family\':name,\'config\':config,\'inner_mse\':loss});print(json.dumps({\'fold\':fold,\'family\':name}),flush=True)\n    future={};forward=[];tr=dev[dev.draft_year.eq(2023)];va=dev[dev.draft_year.eq(2024)]\n    for name,cols in families.items():\n        losses=[float(np.mean((va.target-model_predict(tr,va,cols,*c))**2)) for c in bank()];ix=int(np.argmin(losses));future[name]=model_predict(dev,test,cols,*bank()[ix]);forward.append({\'family\':name,\'config\':bank()[ix],\'validation_mse\':losses})\n    results={\'schema\':\'pp-movement-transfer/1\',\'baseline_columns\':PHYS,\'families\':{k:len(v) for k,v in families.items()},\'nested_selection\':selection,\'forward_selection\':forward,\n        \'split_scope\':\'Entire athletes held out. Imputation, scales, residualization, PCA and alpha selected within training folds. Forward choices use 2023→2024 only.\',\n        \'2025_scope\':\'Exploratory reuse: this draft class has already been inspected in parallel experiments. No untouched external-test or competition-win claim.\',\n        \'baseline_fallback\':True,\'model_scope\':\'Equal-player conditional pressure-rate regression. Source Combine position and drill availability included. NFL position is excluded; source measurements are audited for constancy before aggregation.\'}\n    for label,rows,pp in [(\'nested_2023_2024\',dev,predictions),(\'exploratory_2025\',test,future)]:\n        results[label]={name:compare(rows.target.to_numpy(),pp[\'conventional\'],p) for name,p in pp.items() if name!=\'conventional\'}\n        q=rows[[\'nfl_id\',\'draft_year\',\'target\',\'exposure\']].copy()\n        for name,p in pp.items():q[name]=p\n        q.to_pickle(out/(label+\'.pkl\'))\n    return results\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'--data\',type=Path,required=True);p.add_argument(\'--cohort\',type=Path,required=True);p.add_argument(\'--private-out\',type=Path,required=True);p.add_argument(\'--aggregate\',type=Path,required=True);p.add_argument(\'--reuse\',action=\'store_true\');a=p.parse_args()\n    if a.reuse:unit=pd.read_pickle(a.private_out/\'players.pkl\');ex=json.loads((a.private_out/\'extraction.json\').read_text())\n    else:unit,ex=extract(a.data,a.cohort,a.private_out)\n    result=evaluate(unit,a.private_out);result[\'extraction\']=ex;a.aggregate.parent.mkdir(parents=True,exist_ok=True);a.aggregate.write_text(json.dumps(result,indent=2)+\'\\n\');print(json.dumps({k:result[k] for k in (\'nested_2023_2024\',\'exploratory_2025\')}))\nif __name__==\'__main__\':main()\n', 'scripts/run_movement_residual.py': '#!/usr/bin/env python3\n"""Reproduce nested residual corrections from authorized private checkpoints.\n\nNever publishes player rows. Input is players.pkl/events.pkl from the existing\nmovement transfer extractor, plus source combine_position on players. Requires\nunique player rows. Outputs only aggregates to --aggregate; all predictions go\nto an explicit --private-out directory outside the source checkout.\n"""\nimport argparse,hashlib,json,platform,sys,time\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parents[1];sys.path.insert(0,str(ROOT/\'python\'))\nimport numpy as np,pandas as pd, sklearn\nfrom perfectpower.movement_residual import fit_predict,splits,FAMILY_NAMES,validate\n\n\ndef summary(rows,pred,seed=717,draws=4000):\n    y=rows.target.to_numpy();base=(y-pred[\'conventional\'])**2;rng=np.random.default_rng(seed)\n    # Average repeated outer squared losses before calling this helper when used\n    # with development repeats: bootstrap players, never fold appearances.\n    return loss_summary(rows,{k:(y-v)**2 for k,v in pred.items()},seed,draws)\n\n\ndef loss_summary(rows,loss,seed=717,draws=4000):\n    rng=np.random.default_rng(seed);base=np.asarray(loss[\'conventional\']);groups=[np.flatnonzero(rows.draft_year.to_numpy()==c) for c in sorted(rows.draft_year.unique())]\n    indices=np.column_stack([rng.choice(g,(draws,len(g)),replace=True) for g in groups])\n    result={\'players\':len(rows),\'baseline_mse\':float(base.mean()),\'comparisons\':{}}\n    for name,l in loss.items():\n        if name==\'conventional\':continue\n        d=base-l;boot=d[indices].mean(axis=1)\n        result[\'comparisons\'][name]={\'mse\':float(np.mean(l)),\'absolute_gain\':float(d.mean()),\'relative_gain\':float(d.mean()/base.mean()),\'paired_player_bootstrap_95\':np.quantile(boot,[.025,.975]).tolist(),\'player_win_fraction\':float(np.mean(d>0)), \'by_class\':{str(c):{\'players\':len(g),\'baseline_mse\':float(base[g].mean()),\'candidate_mse\':float(np.mean(l[g])),\'relative_gain\':float(d[g].mean()/base[g].mean())} for c,g in zip(sorted(rows.draft_year.unique()),groups)}}\n    result[\'interval_scope\']=\'Fixed fitted predictions; class-stratified paired players. Does not include refitting, adaptive research selection or new-cohort uncertainty.\'\n    return result\n\n\ndef evaluate(unit,events,out,refits=100):\n    validate(unit)\n    if unit.nfl_id.duplicated().any():raise ValueError(\'expected one row per player\')\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    dev=unit[unit.draft_year.isin([2023,2024])].reset_index(drop=True);future=unit[unit.draft_year.eq(2025)].reset_index(drop=True)\n    result={\'schema\':\'pp-movement-residual/1\',\'protocol_sha256\':hashlib.sha256((ROOT/\'docs/MOVEMENT_RESIDUAL_PROTOCOL.json\').read_bytes()).hexdigest(),\'prior_2025_inspection\':True,\'scope\':\'All classes previously inspected; conditional rookie participation target; development study, no untouched validation.\',\'repeats\':[],\'temporal\':{}}\n    losses=[]\n    for seed in (714,914,1114):\n        pred={k:np.zeros(len(dev)) for k in (\'conventional\',)+FAMILY_NAMES+(\'selected\',)};audits=[]\n        for fold,(i,j) in enumerate(splits(dev,4,seed)):\n            p,a=fit_predict(dev.iloc[i],dev.iloc[j],events,seed+fold)\n            for k,v in p.items():pred[k][j]=v\n            a[\'fold\']=fold;audits.append(a)\n            print(json.dumps({\'stage\':\'nested\',\'seed\':seed,\'fold\':fold}),flush=True)\n        loss={k:np.square(dev.target.to_numpy()-v) for k,v in pred.items()};losses.append(loss)\n        result[\'repeats\'].append({\'seed\':seed,\'evaluation\':loss_summary(dev,loss),\'outer_audits\':audits})\n        private=dev[[\'nfl_id\',\'draft_year\',\'target\']].copy()\n        for k,v in pred.items():private[k]=v\n        private.to_pickle(out/f\'nested_{seed}.pkl\')\n    result[\'development\']=loss_summary(dev,{k:np.mean([l[k] for l in losses],axis=0) for k in losses[0]})\n    for label,tr,te in [(\'2023_to_2024\',dev[dev.draft_year.eq(2023)],dev[dev.draft_year.eq(2024)]),(\'exploratory_2025\',dev,future)]:\n        tr=tr.reset_index(drop=True);te=te.reset_index(drop=True)\n        p,a=fit_predict(tr,te,events,1714);result[\'temporal\'][label]={\'evaluation\':summary(te,p),\'audit\':a}\n        private=te[[\'nfl_id\',\'draft_year\',\'target\']].copy()\n        for k,v in p.items():private[k]=v\n        private.to_pickle(out/f\'{label}.pkl\')\n        # Entire estimator, preprocessing, baseline and correction selection refit.\n        # Duplicate resampled identities are assigned together in all folds.\n        gains=[];zero=0;rng=np.random.default_rng(2714);private_refits=[]\n        for b in range(refits):\n            ix=np.concatenate([rng.choice(g.index,len(g),replace=True) for _,g in tr.groupby(\'draft_year\')])\n            train=tr.iloc[ix].reset_index(drop=True)\n            pp,aa=fit_predict(train,te,events,1714)\n            test_ix=np.concatenate([rng.choice(g.index,len(g),replace=True) for _,g in te.groupby(\'draft_year\')])\n            y=te.target.to_numpy();lb=np.square(y-pp[\'conventional\']);lc=np.square(y-pp[\'selected\'])\n            gains.append(float(np.mean((lb-lc)[test_ix])));zero+=aa[\'selection\'][\'selected\'][\'config\'][0]==\'zero\'\n            private_refits.append((pp[\'conventional\'],pp[\'selected\']))\n            if (b+1)%10==0:print(json.dumps({\'stage\':\'full_refit\',\'comparison\':label,\'done\':b+1,\'total\':refits}),flush=True)\n        np.savez_compressed(out/(label+\'_refits.npz\'),predictions=np.array(private_refits))\n        result[\'temporal\'][label][\'full_refit_bootstrap\']={\'draws\':refits,\'absolute_gain_95\':np.quantile(gains,[.025,.975]).tolist() if gains else None,\'mean_absolute_gain\':float(np.mean(gains)) if gains else None,\'positive_gain_fraction\':float(np.mean(np.array(gains)>0)) if gains else None,\'zero_selection_count\':zero,\'scope\':\'Class-stratified training-player resample and complete nested refit, then paired test-player resample. Conditional on existing classes; not new-cohort or researcher-selection uncertainty.\'}\n    result[\'source_support\']={\'players\':len(unit),\'by_class\':{str(k):int(v) for k,v in unit.groupby(\'draft_year\').size().items()},\'attempts\':len(events),\'missing_fraction_by_family\':{name:float(unit[cols].isna().to_numpy().mean()) for name,cols in __import__(\'perfectpower.movement_residual\',fromlist=[\'families\']).families(unit).items()}}\n    return result\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'--players\',type=Path,required=True);p.add_argument(\'--events\',type=Path,required=True);p.add_argument(\'--private-out\',type=Path,required=True);p.add_argument(\'--aggregate\',type=Path,required=True);p.add_argument(\'--refits\',type=int,default=100);a=p.parse_args()\n    if a.refits<0:p.error(\'--refits must be nonnegative\')\n    if ROOT==a.private_out.resolve() or ROOT in a.private_out.resolve().parents:p.error(\'--private-out must be outside repository\')\n    start=time.perf_counter();r=evaluate(pd.read_pickle(a.players),pd.read_pickle(a.events),a.private_out,a.refits)\n    r[\'input_sha256\']={k:hashlib.sha256(v.read_bytes()).hexdigest() for k,v in [(\'players\',a.players),(\'events\',a.events)]}\n    r[\'source_sha256\']={str(v.relative_to(ROOT)):hashlib.sha256(v.read_bytes()).hexdigest() for v in [ROOT/\'python/perfectpower/movement_residual.py\',Path(__file__).resolve()]}\n    r[\'versions\']={\'python\':platform.python_version(),\'numpy\':np.__version__,\'pandas\':pd.__version__,\'sklearn\':sklearn.__version__};r[\'elapsed_seconds\']=time.perf_counter()-start\n    a.aggregate.parent.mkdir(parents=True,exist_ok=True);a.aggregate.write_text(json.dumps(r,indent=2,allow_nan=False)+\'\\n\');print(json.dumps(r[\'development\']),flush=True)\nif __name__==\'__main__\':main()\n', 'docs/COMBINE_COMPACT_PROTOCOL.json': '{\n  "schema": "pp-combine-compact-protocol/1",\n  "date": "2026-10-28",\n  "status": "New bounded retrospective development experiment; specifications written before this experiment\'s outcomes. All cohorts previously inspected; not independent validation.",\n  "question": "Can four fixed Combine-only movement coordinates add stable rookie pressure-rate information to a separately tuned conventional baseline without large extrapolation?",\n  "cohort": "Reuse exactly the preserved 74 rookies,361 accepted Combine attempts,14665 eligible rookie rushes. Whole-player rows;2023/2024 development45 players;2025 previously inspected exploratory29. No outcome-based roster exclusions.",\n  "target": "Observed pressure-event fraction in supplied eligible charted rookie rush cohort. Conditional on selection, playing opportunity and context. Not the two-season pressures per defensive snap target in SCOUTING_CONFIRMATION_PROTOCOL.json.",\n  "features": ["movement__PASS_RUSH_DRILL__speed_gain", "movement__PASS_RUSH_DRILL__recovery_accel", "movement__RUN_THE_HOOP_DRILL__redirect_turn", "movement__FOUR_BAG_AGILITY_DRILL__brake_accel"],\n  "feature_choice": "Launch speed change, operational phase recovery, hoop redirection and four-bag braking span four interpretable quantities. Fixed before running this experiment; existing outcome exploration is explicitly acknowledged. Phase terms remain model-defined rather than physiologically annotated.",\n  "conventional": "Reuse movement_residual.Baseline: full15/core8/motor4 sets, penalties0.1/1/10/100/1000/10000/mean, three fold whole-player class-stratified tuning; shared source Combine position and four drill-availability indicators. Includes NGS scores in full15, not purely physical. No NFL position predictor.",\n  "correction": "Cross-fit conventional baseline within training to obtain response residuals. Four coordinates median-imputed, scaled and residualized against full conventional design using Ridge10 in training only; clip nuisance-residualized coordinates to[-3,3]. Fit Ridge10 without intercept to cross-fitted responses. Fixed half correction, clipped to +/-0.02 absolute pressure fraction, then final prediction clipped[0,1]. No additional hyperparameter grid.",\n  "selection": "Three inner whole-player class-stratified folds compare only exact-zero versus fixed bounded correction. Apply correction only if paired inner mean squared-loss gain exceeds its player standard error; otherwise return exact same conventional predictions. This heuristic is not an unseen-player dominance guarantee.",\n  "outer": "Three class-stratified four-fold development repeats seeds714/914/1114. Average losses within each player before bootstrap. Temporal2023_to_2024 and exploratory2025 refit all preprocessing/tuning using only earlier training players; no future labels enter predictions.",\n  "controls": "Training-only global mean; training-only source-position mean shrunk toward training global mean with strength20; corrected conventional baseline; fixed bounded correction; inner-selected correction. Retain all results and positions, no post-hoc winner selection.",\n  "uncertainty": "4000 class-stratified paired player bootstrap draws seed717 conditional on retained predictions, no causal or new-cohort guarantee. Temporal sensitivity:20 complete training-player bootstrap refits with duplicate identities grouped, paired test-player resampling. Exactly20 for this release; more may be declared as sensitivity, not optional-stopping significance. Fixed-prediction diagnostic intervals use Bonferroni alpha.05/8 across2 temporal comparisons x4 candidate controls (global mean,role mean,fixed,selected),4000 resamples. Twenty-refit percentile intervals are coarse descriptive sensitivity only and do not establish simultaneous coverage. Development repeats not independent samples.",\n  "negative_control": "16 class-and-source-position stratified player permutations of the four player-median movement coordinates; refit entire temporal2023_to_2024 pipeline only. Preserve Combine/context fields and labels, report fraction reassigned. Descriptive null sensitivity, not a valid broad independence test or researcher-selection-adjusted p-value.",\n  "decision": "No model promotion from this retrospective study. Report exact fallback frequency, worst correction, every gain/loss and refit uncertainty. A useful candidate requires positive gains across all development repeats and temporal2024 with uncertainty excluding zero, followed by genuinely uninspected later players; these are evidence gates, not permission to tune after looking.",\n  "unresolved": "Rush-level pre-snap context table absent from compact continuation. No context-standardization or independent trial noise claim can be made from player totals. Preserve opportunity/context confounding and original broader-family failures. Future sealed two-season2025 confirmation remains untouched."\n}\n', 'scripts/prepare_movement_residual.py': '#!/usr/bin/env python3\n"""Prepare residual-study private inputs without modifying any donor experiment."""\nimport argparse,json\nfrom pathlib import Path\nimport pandas as pd\nfrom run_movement_transfer import extract\n\ndef prepare(data,cohort,out):\n    unit,receipt=extract(data,cohort,out)\n    source=pd.read_pickle(cohort)\n    if \'combine_position\' not in source:raise ValueError(\'source Combine position required\')\n    if source.groupby(\'nfl_id\').combine_position.nunique().max()>1:raise ValueError(\'inconsistent source position\')\n    unit[\'combine_position\']=unit.nfl_id.map(source.groupby(\'nfl_id\').combine_position.first())\n    unit.to_pickle(out/\'players.pkl\')\n    return receipt\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'--data\',type=Path,required=True);p.add_argument(\'--cohort\',type=Path,required=True);p.add_argument(\'--private-out\',type=Path,required=True);a=p.parse_args()\n    root=Path(__file__).resolve().parents[1]\n    if root==a.private_out.resolve() or root in a.private_out.resolve().parents:p.error(\'private output must be outside repository\')\n    print(json.dumps(prepare(a.data,a.cohort,a.private_out)))\nif __name__==\'__main__\':main()\n', 'scripts/run_combine_compact.py': '#!/usr/bin/env python3\n"""Run fixed compact Combine experiment on preserved private player inputs."""\nimport argparse,hashlib,json,platform,sys,time\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parents[1];sys.path.insert(0,str(ROOT/\'python\'))\nimport numpy as np,pandas as pd,sklearn\nfrom perfectpower.combine_compact import COLUMNS,fit_predict\nfrom perfectpower.movement_residual import splits,validate\nfrom run_movement_residual import loss_summary\n\ndef sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()\ndef summary(rows,pred):return loss_summary(rows,{k:(rows.target.to_numpy()-v)**2 for k,v in pred.items()})\n\ndef run(unit,events,out,refits=20,permutations=16):\n    validate(unit)\n    if unit.nfl_id.duplicated().any():raise ValueError(\'one row per player required\')\n    if unit.groupby(\'draft_year\').size().to_dict()!={2023:23,2024:22,2025:29} or len(unit)!=74 or int(unit.exposure.sum())!=14665 or len(events)!=361:raise ValueError(\'fixed cohort must match74 players/361 attempts/14665 rushes with23/22/29 classes\')\n    if not set(zip(events.nfl_id,events.draft_year))<=set(zip(unit.nfl_id,unit.draft_year)):raise ValueError(\'unmatched source attempt identity\')\n    if not set(COLUMNS)<=set(unit):raise ValueError(\'missing compact source coordinates\')\n    if not {\'nfl_id\',\'draft_year\',\'event_id\',\'drill_name\'}<=set(events):raise ValueError(\'attempt provenance required\')\n    if events.duplicated([\'nfl_id\',\'draft_year\',\'event_id\']).any():raise ValueError(\'duplicate attempt\')\n    # The narrow experiment consumes saved source features, not fresh labels.\n    for col in COLUMNS:\n        _,drill,feature=col.split(\'__\',2)\n        values=events[events.drill_name.eq(drill)].groupby([\'nfl_id\',\'draft_year\'])[feature].median()\n        actual=np.array([values.get((r.nfl_id,r.draft_year),np.nan) for r in unit.itertuples()])\n        if not np.allclose(actual,unit[col],equal_nan=True,rtol=1e-10,atol=1e-12):raise ValueError(\'player/attempt mismatch \'+col)\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    dev=unit[unit.draft_year.isin([2023,2024])].reset_index(drop=True);future=unit[unit.draft_year.eq(2025)].reset_index(drop=True)\n    result={\'schema\':\'pp-combine-compact/1\',\'protocol_sha256\':sha(ROOT/\'docs/COMBINE_COMPACT_PROTOCOL.json\'),\'independent_confirmation\':False,\'models_promoted\':False,\'sealed_two_season_outcomes_opened\':False,\'cohort\':{\'players\':len(unit),\'attempts\':len(events),\'rushes\':int(unit.exposure.sum()),\'by_class\':{str(k):int(v) for k,v in unit.groupby(\'draft_year\').size().items()},\'source_position_counts\':{str(k):int(v) for k,v in unit.groupby(\'combine_position\').size().items()}},\'repeats\':[],\'temporal\':{},\'feature_source_medians_verified\':True}\n    all_loss=[]\n    for seed in (714,914,1114):\n        pred={k:np.empty(len(dev)) for k in (\'conventional\',\'global_mean\',\'role_mean\',\'fixed\',\'selected\')};audits=[]\n        for f,(i,j) in enumerate(splits(dev,4,seed)):\n            pp,a=fit_predict(dev.iloc[i],dev.iloc[j],seed+f)\n            for k,v in pp.items():pred[k][j]=v\n            audits.append(a)\n        loss={k:(dev.target.to_numpy()-v)**2 for k,v in pred.items()};all_loss.append(loss)\n        result[\'repeats\'].append({\'seed\':seed,\'evaluation\':loss_summary(dev,loss),\'audits\':audits})\n        private=dev[[\'nfl_id\',\'draft_year\',\'target\']].copy()\n        for k,v in pred.items():private[k]=v\n        private.to_pickle(out/f\'nested_{seed}.pkl\')\n        print(json.dumps({\'stage\':\'nested\',\'seed\':seed,\'selected_gain\':result[\'repeats\'][-1][\'evaluation\'][\'comparisons\'][\'selected\'][\'relative_gain\']}),flush=True)\n    result[\'development\']=loss_summary(dev,{k:np.mean([x[k] for x in all_loss],axis=0) for k in all_loss[0]})\n    for label,tr,te in [(\'2023_to_2024\',dev[dev.draft_year.eq(2023)],dev[dev.draft_year.eq(2024)]),(\'exploratory_2025\',dev,future)]:\n        tr=tr.reset_index(drop=True);te=te.reset_index(drop=True);pp,a=fit_predict(tr,te,1714)\n        result[\'temporal\'][label]={\'evaluation\':summary(te,pp),\'audit\':a}\n        uncertainty_rng=np.random.default_rng(717);resample=uncertainty_rng.integers(len(te),size=(4000,len(te)))\n        bl=(te.target.to_numpy()-pp[\'conventional\'])**2\n        for name in pp:\n            if name!=\'conventional\':\n                delta=bl-(te.target.to_numpy()-pp[name])**2\n                result[\'temporal\'][label][\'evaluation\'][\'comparisons\'][name][\'conditional_bonferroni_95\']=np.quantile(delta[resample].mean(axis=1),[.05/16,1-.05/16]).tolist()\n        private=te[[\'nfl_id\',\'draft_year\',\'target\']].copy()\n        for k,v in pp.items():private[k]=v\n        private.to_pickle(out/f\'{label}.pkl\')\n        rng=np.random.default_rng(2714);gains={k:[] for k in pp if k!=\'conventional\'};zeros=0;max_change=0.;arrays=[]\n        for b in range(refits):\n            ix=np.concatenate([rng.choice(g.index,len(g),replace=True) for _,g in tr.groupby(\'draft_year\')]);sample=tr.iloc[ix].reset_index(drop=True)\n            bp,ba=fit_predict(sample,te,1714);jx=rng.integers(len(te),size=len(te));base=(te.target.to_numpy()-bp[\'conventional\'])**2\n            for k in gains:gains[k].append(float(np.mean((base-(te.target.to_numpy()-bp[k])**2)[jx])))\n            zeros+=not ba[\'apply_correction\'];max_change=max(max_change,ba[\'maximum_absolute_fixed_change\']);arrays.append(np.stack([bp[k] for k in pp]))\n        np.savez_compressed(out/f\'{label}_refits.npz\',predictions=np.asarray(arrays),model_names=np.asarray(list(pp)))\n        result[\'temporal\'][label][\'full_refits\']={\'draws\':refits,\'zero_selection_count\':zeros,\'max_fixed_change\':max_change,\'comparisons\':{k:{\'absolute_gain_95\':np.quantile(v,[.025,.975]).tolist() if v else None,\'positive_fraction\':float(np.mean(np.asarray(v)>0)) if v else None} for k,v in gains.items()},\'scope\':\'Complete training-player resample/refit then paired test-player resample; 20-draw percentile tails are coarse descriptive sensitivity, not calibrated simultaneous coverage or independent confirmation.\'}\n        print(json.dumps({\'stage\':\'temporal\',\'label\':label,\'selected_gain\':result[\'temporal\'][label][\'evaluation\'][\'comparisons\'][\'selected\'][\'relative_gain\']}),flush=True)\n    # Conditional mislinking control preserves source-position/class strata.\n    tr=dev[dev.draft_year.eq(2023)].reset_index(drop=True);te=dev[dev.draft_year.eq(2024)].reset_index(drop=True)\n    rng=np.random.default_rng(3714);null=[];fractions=[]\n    for b in range(permutations):\n        shuffled=dev.copy();mapping=np.arange(len(dev))\n        for _,g in dev.groupby([\'draft_year\',\'combine_position\'],dropna=False):mapping[g.index]=rng.permutation(g.index)\n        shuffled.loc[:,list(COLUMNS)]=dev.iloc[mapping][list(COLUMNS)].to_numpy()\n        sp,_=fit_predict(shuffled[shuffled.draft_year.eq(2023)].reset_index(drop=True),shuffled[shuffled.draft_year.eq(2024)].reset_index(drop=True),1714)\n        null.append(summary(te,sp)[\'comparisons\'][\'selected\'][\'absolute_gain\']);fractions.append(float(np.mean(mapping!=np.arange(len(dev)))))\n    result[\'mislinking_control\']={\'draws\':permutations,\'absolute_gains\':null,\'mean_reassigned_fraction\':float(np.mean(fractions)) if fractions else None,\'scope\':\'Class/source-position constrained mislinking sensitivity; not a formal p-value or new independent evidence.\'}\n    return result\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    for key in (\'players\',\'events\',\'private-out\',\'aggregate\'):p.add_argument(\'--\'+key,type=Path,required=True)\n    p.add_argument(\'--refits\',type=int,default=20);p.add_argument(\'--permutations\',type=int,default=16);a=p.parse_args()\n    if a.refits<0 or a.permutations<0:p.error(\'nonnegative iteration counts required\')\n    if a.private_out.resolve()==ROOT or ROOT in a.private_out.resolve().parents:p.error(\'private outputs must stay outside checkout\')\n    start=time.perf_counter();r=run(pd.read_pickle(a.players),pd.read_pickle(a.events),a.private_out,a.refits,a.permutations)\n    r[\'input_sha256\']={k:sha(v) for k,v in [(\'players\',a.players),(\'events\',a.events)]};r[\'source_sha256\']={str(v.relative_to(ROOT)):sha(v) for v in [ROOT/\'python/perfectpower/combine_compact.py\',Path(__file__),ROOT/\'python/perfectpower/movement_residual.py\']};r[\'versions\']={\'python\':platform.python_version(),\'numpy\':np.__version__,\'pandas\':pd.__version__,\'sklearn\':sklearn.__version__};r[\'elapsed_seconds\']=time.perf_counter()-start\n    a.aggregate.parent.mkdir(parents=True,exist_ok=True);a.aggregate.write_text(json.dumps(r,indent=2,allow_nan=False)+\'\\n\')\nif __name__==\'__main__\':main()\n', 'scripts/audit_combine_repeatability.py': '#!/usr/bin/env python3\n"""Outcome-free split-attempt audit of the four declared source coordinates."""\nimport argparse,hashlib,json,sys\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parents[1];sys.path.insert(0,str(ROOT/\'python\'))\nimport numpy as np,pandas as pd\nfrom scipy.stats import spearmanr\nfrom perfectpower.combine_compact import COLUMNS\n\ndef audit(events):\n    # Only source attempts from development classes; never consume NFL labels.\n    e=events[events.draft_year.isin([2023,2024])].copy()\n    if e.duplicated([\'nfl_id\',\'draft_year\',\'event_id\']).any():raise ValueError(\'duplicate attempt\')\n    r={\'schema\':\'pp-combine-repeatability/1\',\'outcome_free\':True,\'classes\':[2023,2024],\'scope\':\'Deterministic alternating event-ID split within athlete/drill. Repetitions may differ in setup/instruction. Agreement is descriptive, not biological reliability or calibrated phase probability.\',\'features\':{}}\n    for col in COLUMNS:\n        _,drill,f=col.split(\'__\',2);d=e[e.drill_name.eq(drill)].dropna(subset=[f]).sort_values([\'nfl_id\',\'event_id\']);pairs=[]\n        for _,g in d.groupby(\'nfl_id\',sort=True):\n            if len(g)>=2:pairs.append((float(g.iloc[::2][f].median()),float(g.iloc[1::2][f].median())))\n        x=np.asarray(pairs).reshape(-1,2);n=len(x);corr=float(spearmanr(x[:,0],x[:,1]).statistic) if n>=4 and (np.std(x,axis=0)>0).all() else None\n        if corr is not None and not np.isfinite(corr):corr=None\n        boot=[];rng=np.random.default_rng(4728)\n        if n>=8:\n            for _ in range(1000):\n                v=x[rng.integers(n,size=n)]\n                if (np.std(v,axis=0)>0).all():\n                    z=float(spearmanr(v[:,0],v[:,1]).statistic)\n                    if np.isfinite(z):boot.append(z)\n        r[\'features\'][col]={\'attempts\':len(d),\'players\':int(d.nfl_id.nunique()),\'repeat_players\':n,\'split_spearman\':corr,\'descriptive_player_bootstrap_95\':np.quantile(boot,[.025,.975]).tolist() if boot else None,\'bootstrap_valid_draws\':len(boot),\'median_absolute_split_difference\':float(np.median(np.abs(x[:,0]-x[:,1]))) if n else None,\'units\':\'Inherited operational feature units; compare within feature only.\'}\n    return r\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'--events\',type=Path,required=True);p.add_argument(\'--out\',type=Path,required=True);a=p.parse_args();r=audit(pd.read_pickle(a.events));r[\'input_sha256\']=hashlib.sha256(a.events.read_bytes()).hexdigest();r[\'source_sha256\']=hashlib.sha256(Path(__file__).read_bytes()).hexdigest();a.out.parent.mkdir(parents=True,exist_ok=True);a.out.write_text(json.dumps(r,indent=2,allow_nan=False)+\'\\n\')\n'}
SOURCE_RECEIPT={'schema': 'pp-kaggle-notebook-code/1', 'module_sha256': {'upstream/analysis.py': '03abc5d6a5b1a74bcb9f038fa60cd62574acd9e459e7faf6ab009ea9dfcfe519', 'upstream/repo_predict.py': 'db27f6ea463cf829e72e7b7b7aa891adc0539de6240599bc89dcdc758a3bf04b', 'upstream/mechanism.py': '5df271425fe6beb8bdd879f49b3c2074ea45adfcf886a110ce36f12022df411e', 'upstream/observability.py': '96c30e72491f920ed4a9cde803a4a0334a5fb1992fc5290bc71339d92e014445', 'upstream/turn_transfer.py': 'b8c7fab1913d76d725c6b0bedaf0d5079d92601049caafe65f42d2bb82ec0f80', 'upstream/front_transfer.py': 'c47374229644a05e4e879e467214ba1e9e89cd7b51e77c013adf6ce089079e11', 'python/perfectpower/__init__.py': '079063aae98135b155bd1b52f279b75c96cae8fb44d8606bd5d4162dbe464b25', 'python/perfectpower/core.py': '02f86a80d53e63e2d752f8c8cae6487806e75836ce27520af9f621146aa6a830', 'python/perfectpower/movement_language.py': '0589538b07f88d0533f8ea1f42a2a8119a9e1b9c919a0cbbda3edc2237d9e334', 'python/perfectpower/movement_features.py': 'd7255fb929032e85696dd85994df2547456d0363d6a0e60abc1e16b69b9365c4', 'python/perfectpower/movement_residual.py': 'f27725bdc0dd0ea96458af15df66e0bb219345a744aafe36defcf49de3795ff4', 'python/perfectpower/combine_compact.py': '6adee26b58470350b107227b69e2e9c501b238656e98c883ae93fadee641d90a', 'scripts/run_movement_transfer.py': '10be0a361b5872f379f1d5aadf7a11da59869fe04aacb954ff3338ed179cf870', 'scripts/run_movement_residual.py': 'd2bb540dbfbdc7a9326a51b0b163126fe3cb91c9f4ea59725c2c0aa82c76530b', 'docs/COMBINE_COMPACT_PROTOCOL.json': 'd0512097e20fe9959b72897da6eccb1149f3442a8742f8e358390f747e4c521c', 'scripts/prepare_movement_residual.py': '198545514f183989f9a9f7434ae90c299a89ba4a7ca21ce0a20986f4b725afba', 'scripts/run_combine_compact.py': '336e8dd94cc91f412032868751b07083103d76e1d9174077df9c87cc19bc36aa', 'scripts/audit_combine_repeatability.py': '8460d1c543669ea51e1dbce56600c9160e0f7d1a74ed63142686c268bd772910'}, 'upstream_notebook_sha256': '191d25b089529e953d0624122037a8504af1d4d865d8dcc6268670b4a03e1af4', 'no_competition_rows_bundled': True}

for relative,source in MODULES.items():
    target=REPO/relative;target.parent.mkdir(parents=True,exist_ok=True)
    assert hashlib.sha256(source.encode()).hexdigest()==SOURCE_RECEIPT['module_sha256'][relative]
    target.write_text(source)
sys.path[:0]=[str(REPO/'python'),str(REPO/'scripts'),str(REPO/'upstream')]
print('Verified public code modules:',len(MODULES))


## Reconstruct the same sample, then use source Combine position

The historical cohort adapter is preserved byte for byte and only its preparation function is called. It selects the old participation-conditioned cohort using NFL records. The prediction code uses **source Combine position** and source measurements; it does not use eventual NFL position, NFL movement, pressure outcomes or future exposures as movement inputs. The extraction audit rejects inconsistent player-level source values.

For a trusted saved continuation, set both `BDB_PLAYERS` and `BDB_EVENTS` to the exact private extracted files already authorized by you. Pickle files execute Python on loading, so this optional path is only for your own trusted saved files. A supplied continuation bypasses raw reconstruction; its receipt records that fact and it is not a substitute for a full official-data reproducibility run.


In [ ]:
OFFICIAL_HASHES={'combine_results.csv': 'e4773ef68167d508f3916d446cd411dfd77a87e5765d3babcb986201fb143b86', 'combine_tracking.csv': '541c69f6629f5f803b08033c5c386735582120973562134b2c1e7ec6553fb0d3', 'game_tracking_2023.csv': '385e620b6b94e8cd91f0d5776395b76b10d52a34401c764c21b4e98ab617855c', 'game_tracking_2024.csv': '06b7f34304414306dab53d5cb4c5d28485c5f62376442bcb913f9f01184ece6d', 'game_tracking_2025.csv': '419b6bc0388a9210f61911ec2f1351b1e8d56355c0ac8576f24a3bbcd95e54ca', 'games.csv': 'ca33543f6939af4ee393c568c60920c476ba91ddbfbdce5a06578b9492fa11fd', 'player_career_successes.csv': '0d4b8975debda4e2481218f30a16429dad5b806ecdd2fc4367291ac1460ed0a9', 'player_play.csv': '68926a268c0a49e58616001fb35ea179018d9cbbce1c5c31e7a4095273c35b2a', 'players.csv': '2cd7e0309e7aa40533028dc537c2c877dbedbe0fdb14f337fc18af31a5861c12'}

provided_players=os.environ.get('BDB_PLAYERS')
provided_events=os.environ.get('BDB_EVENTS')
if bool(provided_players)!=bool(provided_events):raise ValueError('Set BDB_PLAYERS and BDB_EVENTS together')
if provided_players:
    PLAYERS=Path(provided_players).resolve();EVENTS=Path(provided_events).resolve()
    if not PLAYERS.is_file() or not EVENTS.is_file():raise FileNotFoundError('Trusted continuation files missing')
    input_mode='trusted extracted continuation; official raw reconstruction skipped'
else:
    DATA=Path(os.environ.get('BDB_DATA_ROOT','/kaggle/input/nfl-big-data-bowl-2027')).resolve()
    if not (DATA/'players.csv').is_file():raise FileNotFoundError('Attach official competition data or set BDB_DATA_ROOT')
    for name,expected in OFFICIAL_HASHES.items():
        h=hashlib.sha256()
        with (DATA/name).open('rb') as stream:
            for chunk in iter(lambda:stream.read(8*1024*1024),b''):h.update(chunk)
        if h.hexdigest()!=expected:raise ValueError('Official dataset version differs: '+name)
    import front_transfer as historical_cohort
    import prepare_movement_residual as prepare_current
    COHORT=WORK/'private_front';EXTRACTED=WORK/'private_current'
    with (WORK/'private_extraction.log').open('w') as log,contextlib.redirect_stdout(log):
        historical_cohort.prepare(DATA,COHORT)
        prepare_current.prepare(DATA,COHORT/'private_rows.pkl',EXTRACTED)
    PLAYERS=EXTRACTED/'players.pkl';EVENTS=EXTRACTED/'events.pkl'
    input_mode='nine source hashes verified; official raw cohort reconstructed'
print(input_mode)


## Fixed compact comparison

The experiment specification fixes the compact movement block before this run. Whole athletes remain together through outer folds, inner tuning and training resamples. Conventional preprocessing and correction fits use training players only. The policy may return the baseline exactly. A zero option is a model-selection safeguard, not a promise of better future prediction.

The primary outcome is equal-player squared error on the existing conditional pressure-rate target. Earlier classes are development;2025 is exploratory. No new cohort becomes independent merely because this notebook is new. The runner saves private player predictions outside the public-code directory and exposes only aggregate receipts below. Set `BDB_REFITS` to the desired predeclared full-refit count before running; the default is20 and its interval is a coarse diagnostic.


In [ ]:
import run_combine_compact as compact_runner
REFITS=int(os.environ.get('BDB_REFITS','20'))
PERMUTATIONS=int(os.environ.get('BDB_PERMUTATIONS','16'))
if REFITS<0 or PERMUTATIONS<0:raise ValueError('Iteration counts must be nonnegative')
RESULT=WORK/'aggregate_compact.json'
PRIVATE=WORK/'private_compact_results'
argv=['run_combine_compact.py','--players',str(PLAYERS),'--events',str(EVENTS),'--private-out',str(PRIVATE),'--aggregate',str(RESULT),'--refits',str(REFITS),'--permutations',str(PERMUTATIONS)]
old_argv=sys.argv
try:
    sys.argv=argv
    with (WORK/'private_models.log').open('w') as log,contextlib.redirect_stdout(log):
        compact_runner.main()
finally:sys.argv=old_argv
result=json.loads(RESULT.read_text())
import audit_combine_repeatability as repeatability_audit
repeatability=repeatability_audit.audit(pd.read_pickle(EVENTS))
(WORK/'aggregate_repeatability.json').write_text(json.dumps(repeatability,indent=2,allow_nan=False)+'\n')
print('Aggregate receipt:',RESULT.name)
print(json.dumps({k:v for k,v in result.items() if k not in ('source_sha256','input_sha256','repeats')},indent=2)[:15000])


## Results: keep gain, loss and fallback visible

The three figures below summarize only recorded aggregate comparisons. Positive relative gain means lower error. The first figure keeps the earlier negative study visible. The next two report this run's comparisons and source-attempt repeatability; missing expected receipt fields cause a clear error rather than a fabricated chart. Intervals are conditional diagnostics and do not cover the full adaptive research history or future class shift. The exact aggregate receipt is the authority. Class stability appears in the table. Alternating event-ID split agreement uses only2023/2024 Combine attempts and no NFL outcomes; it is descriptive agreement across potentially different instructions, not biological reliability or calibrated physiology.


In [ ]:
HISTORICAL_DEVELOPMENT={'players': 45, 'baseline_mse': 0.0009532055704712809, 'comparisons': {'kinematic': {'mse': 0.0009540337801238186, 'absolute_gain': -8.282096525379349e-07, 'relative_gain': -0.0008688678268303175, 'paired_player_bootstrap_95': [-7.003984191125325e-06, 4.861841263349938e-06], 'player_win_fraction': 0.2, 'by_class': {'2023': {'players': 23, 'baseline_mse': 0.0006314988944018242, 'candidate_mse': 0.0006346338697356693, 'relative_gain': -0.004964340178005639}, '2024': {'players': 22, 'baseline_mse': 0.0012895352772711677, 'candidate_mse': 0.0012879518682568842, 'relative_gain': 0.0012278911963028408}}}, 'ordered': {'mse': 0.0009532055704712809, 'absolute_gain': 0.0, 'relative_gain': 0.0, 'paired_player_bootstrap_95': [0.0, 0.0], 'player_win_fraction': 0.0, 'by_class': {'2023': {'players': 23, 'baseline_mse': 0.0006314988944018242, 'candidate_mse': 0.0006314988944018242, 'relative_gain': 0.0}, '2024': {'players': 22, 'baseline_mse': 0.0012895352772711677, 'candidate_mse': 0.0012895352772711677, 'relative_gain': 0.0}}}, 'uncertain': {'mse': 0.0009798124111907833, 'absolute_gain': -2.660684071950253e-05, 'relative_gain': -0.027913014300102817, 'paired_player_bootstrap_95': [-5.890203369540132e-05, -3.0589150063504467e-07], 'player_win_fraction': 0.06666666666666667, 'by_class': {'2023': {'players': 23, 'baseline_mse': 0.0006314988944018242, 'candidate_mse': 0.000663496563568566, 'relative_gain': -0.050669398553818565}, '2024': {'players': 22, 'baseline_mse': 0.0012895352772711677, 'candidate_mse': 0.0013105062518867384, 'relative_gain': -0.01626242801201064}}}, 'combined': {'mse': 0.0009584676345588614, 'absolute_gain': -5.262064087580554e-06, 'relative_gain': -0.005520387470017512, 'paired_player_bootstrap_95': [-1.2104078625220561e-05, 4.589848208297889e-07], 'player_win_fraction': 0.06666666666666667, 'by_class': {'2023': {'players': 23, 'baseline_mse': 0.0006314988944018242, 'candidate_mse': 0.000637715721171174, 'relative_gain': -0.00984455685427366}, '2024': {'players': 22, 'baseline_mse': 0.0012895352772711677, 'candidate_mse': 0.0012937991803732623, 'relative_gain': -0.003306542424428754}}}, 'reliability_combined': {'mse': 0.0009725492908260465, 'absolute_gain': -1.9343720354765605e-05, 'relative_gain': -0.02029333540843844, 'paired_player_bootstrap_95': [-3.760030009433882e-05, -4.832465368595631e-06], 'player_win_fraction': 0.06666666666666667, 'by_class': {'2023': {'players': 23, 'baseline_mse': 0.0006314988944018242, 'candidate_mse': 0.0006443343661816224, 'relative_gain': -0.020325406574078535}, '2024': {'players': 22, 'baseline_mse': 0.0012895352772711677, 'candidate_mse': 0.001315683075681581, 'relative_gain': -0.02027691593342501}}}, 'selected': {'mse': 0.0009806406208433213, 'absolute_gain': -2.743505037204046e-05, 'relative_gain': -0.02878188212693313, 'paired_player_bootstrap_95': [-5.948314324989682e-05, -3.434231804924119e-07], 'player_win_fraction': 0.26666666666666666, 'by_class': {'2023': {'players': 23, 'baseline_mse': 0.0006314988944018242, 'candidate_mse': 0.0006666315389024112, 'relative_gain': -0.0556337387318242}, '2024': {'players': 22, 'baseline_mse': 0.0012895352772711677, 'candidate_mse': 0.001308922842872455, 'relative_gain': -0.015034536815707801}}}}, 'interval_scope': 'Fixed fitted predictions; class-stratified paired players. Does not include refitting, adaptive research selection or new-cohort uncertainty.'}

old_rows=[(name,value['relative_gain']*100) for name,value in HISTORICAL_DEVELOPMENT['comparisons'].items()]
fig,ax=plt.subplots(figsize=(8,3.6))
ax.barh([x[0] for x in old_rows],[x[1] for x in old_rows],color=['#327860' if x[1]>0 else '#b55d45' for x in old_rows])
ax.axvline(0,color='#444',linewidth=.8);ax.set_xlabel('Earlier residual study: relative MSE gain (%)')
ax.set_title('Historical negative evidence remains part of the record')
plt.tight_layout();plt.show()

# Runner schema: development.comparisons maps model name to relative_gain.
development=result['development']
comparison_rows=[(name,value['relative_gain']*100) for name,value in development['comparisons'].items()]
fig,ax=plt.subplots(figsize=(8,3.6))
ax.barh([x[0] for x in comparison_rows],[x[1] for x in comparison_rows],color=['#327860' if x[1]>0 else '#b55d45' for x in comparison_rows])
ax.axvline(0,color='#444',linewidth=.8);ax.set_xlabel('Compact study: relative MSE gain (%)')
ax.set_title('Fixed comparisons, including losses')
plt.tight_layout();plt.show()

class_rows=[]
for name,value in development['comparisons'].items():
    for year,cell in value.get('by_class',{}).items():class_rows.append({'model':name,'draft_class':year,'relative_gain_percent':100*cell['relative_gain']})
if not class_rows:raise ValueError('Aggregate receipt lacks required class stability diagnostics')
class_table=pd.DataFrame(class_rows)
display(class_table)

measurement_rows=[]
for name,cell in repeatability['features'].items():
    measurement_rows.append({'feature':name.replace('movement__','').replace('__',' / '),'repeat_players':cell['repeat_players'],'split_spearman':cell['split_spearman']})
measurement_table=pd.DataFrame(measurement_rows)
fig,ax=plt.subplots(figsize=(8,3.6))
for row in measurement_rows:
    if row['split_spearman'] is not None:
        ax.scatter(row['repeat_players'],row['split_spearman'],s=60)
        ax.annotate(row['feature'],(row['repeat_players'],row['split_spearman']),xytext=(4,4),textcoords='offset points',fontsize=8)
ax.axhline(0,color='#444',linewidth=.8);ax.set_xlabel('Players with repeated source attempts')
ax.set_ylabel('Alternating-split Spearman correlation');ax.set_ylim(-1.1,1.1)
ax.set_title('Measurement support before scouting claims; absent points mean insufficient support')
plt.tight_layout();plt.show()
display(measurement_table)


## Football interpretation and submission boundary

In the recovered74-player cohort, the supplied source Combine position is `DL` for every player. The trained role-mean control therefore equals the global-mean control here; these are intentionally retained as transparent equivalent controls, not independent stratified benchmarks.

A movement metric can be a useful measurement without yet being a dependable scouting predictor. A favorable point estimate here is a hypothesis for later players, not a validated talent ranking. Source noise, sparse repeated attempts, selection into NFL opportunities, position and usage differences all remain plausible explanations.

For coaches, the operational takeaway must match the actual result: use reproducible movement profiles to describe and investigate a player's drill behavior; retain conventional forecasts unless incremental value survives the declared comparisons. No physiological phase calibration or causal development recommendation is claimed.

This notebook is a reproducible continuation artifact. It has not been published or submitted to Kaggle by this code. The competition Writeup must remain within2000 words and fewer than10 tables/figures, explicitly link Combine sensor tracking with NFL performance, and attach a public Kaggle notebook. Verify current official requirements before final submission: https://www.kaggle.com/competitions/nfl-big-data-bowl-2027/overview/timeline .
